# Test Noise Passive Feedback

Measure a broad feedback-off baseline once, then adjust only the feedback measurement.
The two spectra keep independent windows, fits and filter corrections.

1. Load configuration and hardware helpers; verify identity and shutdown (cells 1–11).
2. Load the acquisition helpers (cell 12).
3. Predict the filter response, check FFT scaling, measure the disconnected electronics
   path using OUT1 loopback, and explicitly select model or measured corrections (cells 13–18).
4. Restore passive wiring, take the baseline once, then run manual feedback measurements
   (cells 19–21).

Code cells are numbered 1–21; Jupyter execution counts may differ. The old DC-investigation section is removed. Live raw-input offset,
clipping, register and shutdown guards still run during measurements.
All measurement examples are commented out. Uncomment only the action you intend
and follow its wiring instructions. No saved JSON or automatic sweep is required.

**Emergency stop:** `full_shutdown()`. A crashed kernel cannot execute it;
keep a physical means of interrupting the return path.


## 1 — Configure the measurement

```text
                       mixer → CIC → optional FIR → DMA → FFT → PSD correction
IN1, 125 MS/s --------+
                       phase/gain block → OUT1 → installed electrode return path
```

The commissioned return wiring is OUT1 -> installed high-pass return path -> trap electrode.
Only the upper path is calibrated in cells 13–18. The CIC/FIR are not in the OUT1
feedback loop. The LO selects the measured RF band. CIC decimation trades
bandwidth for sample rate. FIR compensates central-band sag but rejects outer
frequencies: an enabled FIR is not flat across its whole Nyquist band. Software
can correct bounded amplitude loss, but cannot undo aliases, clipping or lost
SNR. FFT bins, Hann ENBW, displayed span and resonance linewidth are distinct.

Definition/model cells do not acquire data. Hardware calls check identity and
headroom and attempt shutdown on errors/Ctrl-C. A crashed kernel cannot run
shutdown. Physically disconnect the trap return and input before injected-tone
electronics tests. Keep a physical means to interrupt feedback.


In [ ]:
# This setup changes frequently, so the notebook records only facts that alter
# its calculations. All unknown analogue gain and phase are calibrated from the
# measured closed-loop resonator response rather than entered as component data.
TOPOLOGY = {
    'campaign_label': 'current-high-pass-feedback-path',
    'resonance_frequency_hz': 40.9e6,
    'high_pass_filter_in_feedback_path': True,
    'input_mode': 'native-high-impedance',
}

RP_LIMITS = {
    'sample_rate_hz': 125.0e6, 'adc_bits': 14,
    'input_impedance_ohm': 1.0e6, 'input_capacitance_f': 10e-12,
    'input_lv_full_scale_vpk': 1.0, 'input_lv_absolute_max_vpk': 6.0,
    'input_hv_full_scale_vpk': 20.0, 'input_hv_absolute_max_vpk': 30.0,
    'input_bandwidth_hz': 50.0e6, 'output_impedance_ohm': 50.0,
    'output_full_scale_vpk_into_50ohm': 1.0,
    'output_bandwidth_hz': 50.0e6, 'output_slew_rate_v_per_s': 200.0e6,
}
FS_CLK = RP_LIMITS['sample_rate_hz']
ADC_RANGE = 'LV'
ADC_OPERATING_GUARD_VPK = 0.80
ADC_RAW_VOLTS_PER_COUNT = RP_LIMITS['input_lv_full_scale_vpk'] / 8192.0
# DMA contains the 32-bit mixer/CIC output, not 14-bit ADC counts.  The RTL
# maps a full-scale ADC sine to complex baseband magnitude 2^29, independent
# of the power-of-two decimation.  Keeping these two scales separate prevents
# a harmless sub-millivolt CIC record being reported as tens of volts.
CIC_COUNTS_PER_ADC_VPK = float(1 << 29)
CIC_VOLTS_PER_COUNT = RP_LIMITS['input_lv_full_scale_vpk'] / CIC_COUNTS_PER_ADC_VPK
OUT1_OPERATING_GUARD_VPK = 0.80
IN1_DC_OFFSET_WARNING_V = 0.050
IN1_DC_OFFSET_MAX_V = 0.100
CENTRE_HZ = TOPOLOGY['resonance_frequency_hz']
INITIAL_Q_ESTIMATE = 150.0
DISCOVERY_SEARCH_HALF_WIDTH_HZ = 0.50e6
MIN_ACQUISITION_DECIMATION_LOG2 = 6
MAX_ACQUISITION_DECIMATION_LOG2 = 12
FIR_MIN_DECIMATION_LOG2 = 8
MAX_RESPONSE_CORRECTION_DB = 12.0
BINS_PER_LINEWIDTH = 20
LINEWIDTHS_EACH_SIDE = 2.5
MAX_CAPTURE_SAMPLES = 1 << 20
# Defaults for optional passive helper calls; manual calls specify their own settings.
PASSIVE_AVERAGES = 20
SETTLE = 0.10
N_AVG = 64
DRIVE_MAX = 0.05
BUILD_ID_EXPECTED = '0xFE76AE09'
TRANSPORT_FACTORY = None       # tests inject MemoryTransport
CAPTURE_BACKEND = None          # tests inject complex passive records


## 2 — Load numerical dependencies

**Purpose.** Import NumPy, SciPy and Matplotlib for PSD calculations, fitting and plots.

**Hardware.** None. **Expected result.** A short dependency/version message. **Failure meaning.**
A missing numerical package. **Safe to rerun.** Yes.


In [ ]:
import os, mmap, struct, time, json, copy, hashlib, math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

print('Standalone numerical dependencies loaded.')


## 3 — Load analysis and safety helpers

**Purpose.** defines every pure-analysis and record-integrity function used below. It is an inline, reviewed copy owned by this notebook.

**Hardware.** none. **Wiring.** irrelevant. **Expected result.** none. **Pass.** the cell completes. **Failure.** a missing standard numerical dependency. **Safe to rerun.** yes.


In [ ]:
"""Safe, manual operating-point recommendations for electrode feedback.

This module contains no Red Pitaya register access.  It turns measurements from
the coil/electrode characterisation notebook into reviewable setup records.  A
notebook-specific adapter performs the eventual hardware write, after the
record and the current calibration have both been checked again.

The deliberately small public interface is:

    make_calibration(...)       validate and freeze measured calibration data
    feedback_q_setup(...)       recommend a Q-enhancement operating point
    feedback_shift_setup(...)   recommend a frequency-shift operating point
    validate_setup(...)         detect stale or modified setup records

Only interpolation between accepted measurements is allowed.  This is an
important safety property: a linear fit may describe the data nicely while
still being a dangerous basis for extrapolation towards self-oscillation.
"""
from __future__ import print_function

import copy
import hashlib
import json
import math
import time


SCHEMA_VERSION = 2


class CalibrationError(ValueError):
    """The measurements do not support a safe recommendation."""


class SetupError(ValueError):
    """A requested or saved setup is incomplete, stale, or unsafe."""


def _finite(name, value):
    value = float(value)
    if not math.isfinite(value):
        raise CalibrationError('%s must be finite' % name)
    return value


def _canonical(value):
    """Stable JSON representation used for calibration/setup fingerprints."""
    return json.dumps(value, sort_keys=True, separators=(',', ':'),
                      allow_nan=False).encode('utf-8')


def _digest(value):
    return hashlib.sha256(_canonical(value)).hexdigest()


def _without_digest(value):
    out = copy.deepcopy(value)
    out.pop('digest', None)
    return out


def _seal(value):
    value = copy.deepcopy(value)
    value['digest'] = _digest(_without_digest(value))
    return value


def _check_seal(value, label, error_type=SetupError):
    if not isinstance(value, dict) or not value.get('digest'):
        raise error_type('%s has no integrity fingerprint' % label)
    if value['digest'] != _digest(_without_digest(value)):
        raise error_type('%s changed after it was created' % label)


def _one_request(values):
    chosen = [(name, value) for name, value in values if value is not None]
    if len(chosen) > 1:
        raise SetupError('give exactly one request, not %s' %
                         ', '.join(name for name, _ in chosen))
    return chosen[0] if chosen else (None, None)


def _sorted_unique(points, xname, required):
    cleaned = []
    for number, point in enumerate(points):
        if not isinstance(point, dict):
            raise CalibrationError('%s point %d is not a dictionary' % (xname, number))
        row = {name: _finite('%s[%d].%s' % (xname, number, name), point[name])
               for name in required}
        row.update({name: value for name, value in point.items() if name not in row})
        if not bool(point.get('accepted', True)):
            continue
        cleaned.append(row)
    cleaned.sort(key=lambda row: row[xname])
    if len(cleaned) < 2:
        raise CalibrationError('%s needs at least two accepted measured points' % xname)
    xs = [row[xname] for row in cleaned]
    if any(b <= a for a, b in zip(xs[:-1], xs[1:])):
        raise CalibrationError('%s points must be unique' % xname)
    return cleaned


def make_calibration(baseline, q_points, shift_points, phases, evidence,
                     safety, build_id, created_utc=None):
    """Validate and freeze the evidence required by both setup helpers.

    ``q_points`` are accepted measured regenerative settings with gain, K and Q.
    ``shift_points`` use signed_gain so the two shift directions remain explicit.
    Measurements that failed their notebook quality gates must either be omitted
    or carry ``accepted=False``; they can never become recommendation anchors.
    """
    base = {name: _finite('baseline.%s' % name, baseline[name])
            for name in ('f0_hz', 'q0', 'linewidth_hz')}
    if base['f0_hz'] <= 0 or base['q0'] <= 0 or base['linewidth_hz'] <= 0:
        raise CalibrationError('baseline frequency, Q, and linewidth must be positive')

    phase_names = ('regenerative_deg', 'degenerative_deg',
                   'positive_shift_deg', 'negative_shift_deg')
    phase_data = {name: _finite('phases.%s' % name, phases[name]) % 360.0
                  for name in phase_names}
    phase_data['uncertainty_deg'] = _finite(
        'phases.uncertainty_deg', phases['uncertainty_deg'])
    if phase_data['uncertainty_deg'] <= 0:
        raise CalibrationError('phase uncertainty must be measured and positive')

    q_rows = _sorted_unique(q_points, 'gain',
                            ('gain', 'k', 'q', 'linewidth_hz',
                             'residual_shift_hz', 'uncertainty_q'))
    shift_rows = _sorted_unique(shift_points, 'signed_gain',
                                ('signed_gain', 'signed_k', 'shift_hz', 'q',
                                 'residual_q_change', 'uncertainty_hz'))
    if any(row['gain'] < 0 or row['k'] < 0 for row in q_rows):
        raise CalibrationError('Q-enhancement gain and K must be non-negative')

    max_k = _finite('safety.max_k', safety['max_k'])
    if not 0 < max_k <= 0.3:
        raise CalibrationError('bench-resonator max_k must be in (0, 0.3]')
    if any(row['k'] > max_k + 1e-12 for row in q_rows):
        raise CalibrationError('an accepted Q point exceeds the bench-resonator K limit')
    if any(abs(row.get('signed_k', 0.0)) > max_k + 1e-12 for row in shift_rows):
        raise CalibrationError('an accepted shift point exceeds the bench-resonator K limit')

    required_evidence = ('noise_floor_valid', 'baseline_valid', 'linearity_valid',
                         'phase_valid', 'gain_linearity_valid', 'drift_valid',
                         'effective_feedback_valid')
    missing = [name for name in required_evidence if not bool(evidence.get(name, False))]
    if missing:
        raise CalibrationError('required calibration evidence failed: %s' %
                               ', '.join(missing))
    required_hardware = ('topology_digest', 'high_pass_filter_in_feedback_path')
    missing = [name for name in required_hardware if name not in safety]
    if missing:
        raise CalibrationError(
            'Calibration lacks concrete hardware identity fields: %s. Rebuild the '
            'campaign record from the current notebook configuration.' % ', '.join(missing))
    if not str(safety['topology_digest']).strip():
        raise CalibrationError(
            'The calibration has no setup fingerprint. Rebuild it from the current '
            'campaign before requesting or applying feedback.')
    if not bool(safety['high_pass_filter_in_feedback_path']):
        raise CalibrationError(
            'This streamlined workflow requires the already-installed high-pass '
            'filter in the feedback return path. If it is removed, stop and create '
            'a new campaign with an explicit DC-return-path safety assessment.')


    calibration = {
        'schema_version': SCHEMA_VERSION,
        'created_utc': created_utc or time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'build_id': str(build_id),
        'baseline': base,
        'phases': phase_data,
        'q_points': q_rows,
        'shift_points': shift_rows,
        'evidence': copy.deepcopy(evidence),
        'safety': copy.deepcopy(safety),
        'electrode_voltage_directly_measured': False,
        'feedback_path_transfer_directly_measured': False,
        'effective_loop_status': 'closed_loop_inference_only',
    }
    return _seal(calibration)


def validate_calibration(calibration):
    if not isinstance(calibration, dict) or not calibration.get('digest'):
        raise CalibrationError(
            'NO COMPLETED CALIBRATION: CALIBRATION is empty or lacks its integrity '
            'fingerprint. The configuration cell resets CALIBRATION to None. '
            'After restoring measurements, rerun the Cell 15 build_calibration '
            'call successfully before Cell 16. A failed build does not create '
            'a calibration. Do not manually add a digest or mark failed evidence '
            'as accepted; rebuilding performs the required checks. No hardware '
            'was accessed by this recommendation request.')
    _check_seal(calibration, 'calibration', CalibrationError)
    if calibration.get('schema_version') != SCHEMA_VERSION:
        raise CalibrationError('unsupported calibration schema')
    return calibration


def _bracket(rows, field, target):
    """Return two measured rows surrounding target; never extrapolate."""
    target = float(target)
    lo, hi = rows[0][field], rows[-1][field]
    if target < lo - 1e-12 or target > hi + 1e-12:
        raise SetupError('requested %s %.6g is outside measured range %.6g..%.6g' %
                         (field, target, lo, hi))
    for left, right in zip(rows[:-1], rows[1:]):
        if left[field] <= target <= right[field]:
            return left, right
    return rows[-2], rows[-1]


def _interpolate(rows, field, target, names):
    rows = sorted(rows, key=lambda row: row[field])
    left, right = _bracket(rows, field, target)
    span = right[field] - left[field]
    weight = 0.0 if span == 0 else (float(target) - left[field]) / span
    result = {name: float(left[name]) + weight * (float(right[name]) - float(left[name]))
              for name in names}
    result[field] = float(target)
    return result, [left[field], right[field]], weight


def _q_target(calibration, target_q, q_ratio, target_k, gain):
    rows, base = calibration['q_points'], calibration['baseline']
    name, value = _one_request((('target_q', target_q), ('q_ratio', q_ratio),
                                ('target_k', target_k), ('gain', gain)))
    if name is None:
        # The first non-zero accepted measurement is deliberately preferred.
        # A notebook may display every rung, but an omitted target must never
        # silently select the operating point closest to instability.
        conservative = next((row for row in rows if row['gain'] > 0), rows[0])
        return 'recommended_conservative', conservative['gain'], 'gain'
    value = float(value)
    if name == 'target_q':
        return name, value, 'q'
    if name == 'q_ratio':
        if value <= 0:
            raise SetupError('q_ratio must be positive')
        return name, value * base['q0'], 'q'
    return name, value, 'k' if name == 'target_k' else 'gain'


def _new_setup(calibration, mode, request, resolved, bracket, weight):
    setup = {
        'schema_version': SCHEMA_VERSION,
        'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'mode': mode,
        'request': request,
        'calibration_digest': calibration['digest'],
        'build_id': calibration['build_id'],
        'baseline': copy.deepcopy(calibration['baseline']),
        'resolved': resolved,
        'interpolation_bracket': list(bracket),
        'interpolation_weight': float(weight),
        'safety': {'max_k': float(calibration['safety']['max_k']),
                   'interpolation_only': True},
        'validation': None,
        'electrode_voltage_directly_measured': False,
        'feedback_path_transfer_directly_measured': False,
    }
    return _seal(setup)


def feedback_q_setup(calibration, target_q=None, q_ratio=None,
                     target_k=None, gain=None):
    """Return a reviewable Q-enhancement setup; never touch hardware."""
    validate_calibration(calibration)
    request_name, target, field = _q_target(
        calibration, target_q, q_ratio, target_k, gain)
    values = ('gain', 'k', 'q', 'linewidth_hz', 'residual_shift_hz', 'uncertainty_q')
    resolved, bracket, weight = _interpolate(
        calibration['q_points'], field, target, values)
    if resolved['k'] > calibration['safety']['max_k'] + 1e-12:
        raise SetupError('recommended K exceeds the bench-resonator limit')
    resolved.update({
        'phase_deg': calibration['phases']['regenerative_deg'],
        'phase_uncertainty_deg': calibration['phases']['uncertainty_deg'],
        'stability_margin': 1.0 - resolved['k'],
        'predicted_f0_hz': calibration['baseline']['f0_hz'] +
                           resolved['residual_shift_hz'],
    })
    return _new_setup(calibration, 'q_enhancement',
                      {request_name: target}, resolved, bracket, weight)


def feedback_shift_setup(calibration, shift_hz=None, shift_linewidths=None,
                         signed_gain=None, target_k=None, direction=None):
    """Return one or two reviewable frequency-shift setups; no hardware access.

    With no request, a dictionary containing conservative positive and negative
    measured endpoints is returned.  ``target_k`` needs direction +1 or -1.
    """
    validate_calibration(calibration)
    name, value = _one_request((('shift_hz', shift_hz),
                                ('shift_linewidths', shift_linewidths),
                                ('signed_gain', signed_gain),
                                ('target_k', target_k)))
    rows = calibration['shift_points']
    if name is None:
        # Present the smallest measured non-zero setting in each direction.
        # Larger endpoints remain available through an explicit request.
        negative = max((row for row in rows if row['signed_gain'] < 0),
                       key=lambda row: row['signed_gain'])
        positive = min((row for row in rows if row['signed_gain'] > 0),
                       key=lambda row: row['signed_gain'])
        return {'negative': feedback_shift_setup(calibration,
                                                 signed_gain=negative['signed_gain']),
                'positive': feedback_shift_setup(calibration,
                                                 signed_gain=positive['signed_gain'])}
    value = float(value)
    if name == 'shift_linewidths':
        target, field = value * calibration['baseline']['linewidth_hz'], 'shift_hz'
    elif name == 'shift_hz':
        target, field = value, 'shift_hz'
    elif name == 'signed_gain':
        target, field = value, 'signed_gain'
    else:
        if direction not in (-1, 1):
            raise SetupError('target_k requires direction=-1 or direction=+1')
        target = float(direction) * value
        # Shift points carry measured vector K when available.  Keep its sign so
        # the two physical frequency directions cannot be confused.
        field = 'signed_k'
        if not all(field in row for row in rows):
            raise SetupError('shift calibration has no signed_k measurements')

    values = ('signed_gain', 'signed_k', 'shift_hz', 'q',
              'residual_q_change', 'uncertainty_hz')
    resolved, bracket, weight = _interpolate(rows, field, target, values)
    resolved['k'] = abs(resolved['signed_k'])
    if resolved['k'] > calibration['safety']['max_k'] + 1e-12:
        raise SetupError('recommended K exceeds the bench-resonator limit')
    positive = resolved['shift_hz'] >= 0
    resolved.update({
        'phase_deg': calibration['phases'][
            'positive_shift_deg' if positive else 'negative_shift_deg'],
        'phase_uncertainty_deg': calibration['phases']['uncertainty_deg'],
        'stability_margin': 1.0 - resolved['k'],
        'predicted_f0_hz': calibration['baseline']['f0_hz'] + resolved['shift_hz'],
        'predicted_linewidth_hz': (calibration['baseline']['linewidth_hz'] *
                                   calibration['baseline']['q0'] /
                                   max(resolved['q'], 1e-30)),
    })
    return _new_setup(calibration, 'frequency_shift', {name: value},
                      resolved, bracket, weight)


def validate_setup(setup, calibration, current_build_id=None, max_age_s=None,
                   now_s=None):
    """Revalidate a setup immediately before a notebook applies it."""
    validate_calibration(calibration)
    _check_seal(setup, 'setup')
    if setup.get('calibration_digest') != calibration['digest']:
        raise SetupError('setup belongs to a different or stale calibration')
    if current_build_id is not None and str(current_build_id) != str(setup['build_id']):
        raise SetupError('current FPGA build does not match the calibration')
    if setup['resolved'].get('k', 0.0) > calibration['safety']['max_k'] + 1e-12:
        raise SetupError('setup exceeds the current bench-resonator K limit')
    if max_age_s is not None:
        # The notebook supplies age from its monotonic laboratory timestamp;
        # parsing wall-clock strings here would be timezone-dependent on old OSes.
        age = float(now_s if now_s is not None else time.time()) - float(
            calibration['evidence']['created_epoch_s'])
        if age < 0 or age > float(max_age_s):
            raise SetupError('calibration is stale (age %.0f s)' % age)
    return True


def quantized_feedback_vector(coefficient_counts, phase_deg, frequency_hz,
                              latency_samples=3, scale=16384):
    """Return the exact signed CI/CQ vector written for a requested phase."""
    import numpy as np
    count = int(coefficient_counts)
    if count < 1 or count >= 32768:
        raise CalibrationError('coefficient_counts must be in [1, 32767]')
    corrected_phase = (float(phase_deg) -
                       360.0*float(latency_samples)*float(frequency_hz)/125.0e6)
    ci = int(round(count*np.cos(np.radians(corrected_phase))))
    cq = int(round(count*np.sin(np.radians(corrected_phase))))
    actual_count = float(np.sqrt(float(ci*ci + cq*cq)))
    actual_corrected_phase = float(np.degrees(np.arctan2(cq, ci)))
    actual_user_phase = (actual_corrected_phase +
                         360.0*float(latency_samples)*float(frequency_hz)/125.0e6) % 360.0
    phase_error = (actual_user_phase-float(phase_deg)+180.0) % 360.0-180.0
    return {'ci_signed': ci, 'cq_signed': cq,
            'ci_word': ci & 0xFFFF, 'cq_word': cq & 0xFFFF,
            'actual_coefficient_counts': actual_count,
            'actual_gain': actual_count/float(scale),
            'actual_phase_deg': actual_user_phase,
            'phase_error_deg': phase_error,
            'magnitude_error_fraction': actual_count/float(count)-1.0}


def simulate_feedback_fixed_point(input_codes, ci_signed, cq_signed=None,
                                  delayed_codes=None, scale=16384):
    """Reproduce the RTL multiply, arithmetic shift and 14-bit clipping.

    For a DC diagnostic, delayed_codes may be omitted: I and Q then contain
    the same constant offset. Python floor division matches an arithmetic
    right shift for negative integers, including the small-negative -> -1
    asymmetry that matters at one-count gain.
    """
    import numpy as np
    direct = np.asarray(input_codes, dtype=np.int64)
    delayed = direct if delayed_codes is None else np.asarray(delayed_codes, dtype=np.int64)
    if direct.shape != delayed.shape:
        raise CalibrationError('direct and delayed feedback samples must have matching shapes')
    ci = int(ci_signed); cq = int(0 if cq_signed is None else cq_signed)
    accumulator = ci*direct + cq*delayed
    scaled = np.floor_divide(accumulator, int(scale))
    return np.clip(scaled, -8192, 8191).astype(np.int64)


# ---------------------------------------------------------------------------
# Passive measurement and preflight helpers
# ---------------------------------------------------------------------------

def validate_topology(config):
    """Validate only calculation-critical setup facts."""
    required = ('campaign_label', 'resonance_frequency_hz',
                'high_pass_filter_in_feedback_path', 'input_mode')
    missing = [name for name in required if name not in config or config[name] in (None, '')]
    if missing:
        raise CalibrationError(
            'Campaign configuration is missing: %s. Enter only these calculation-critical '
            'values; individual amplifier, attenuator and cable details are not required.' %
            ', '.join(missing))
    frequency_hz = _finite('resonance_frequency_hz', config['resonance_frequency_hz'])
    if not 0.1e6 <= frequency_hz <= 50.0e6:
        raise CalibrationError(
            'Configured resonance %.6g Hz is outside the qualified '
            '0.1--50 MHz Red Pitaya measurement band. Feedback remains off.' % frequency_hz)
    if not bool(config['high_pass_filter_in_feedback_path']):
        raise CalibrationError(
            'The high-pass filter is not declared in the feedback path. This notebook '
            'will still permit feedback-off measurements, but active feedback requires '
            'a separate DC-return-path safety review.')
    return _seal({'schema_version': SCHEMA_VERSION,
                  'kind': 'test_noise_passive_feedback_topology',
                  'config': copy.deepcopy(config)})

def passive_welch_spectrum(records, sample_rate_hz, window='hann'):
    """Return a correctly normalized one-sided PSD from passive time records.

    ``records`` is a two-dimensional array (one row per independent capture).
    The returned metadata exposes bin width, equivalent-noise bandwidth and
    averaging uncertainty instead of hiding these quantities in a plot.
    """
    import numpy as np
    data = np.asarray(records, dtype=float)
    if data.ndim != 2 or data.shape[0] < 2 or data.shape[1] < 16:
        raise CalibrationError('need at least two passive records of 16 samples')
    if not np.all(np.isfinite(data)) or float(sample_rate_hz) <= 0:
        raise CalibrationError('records and sample rate must be finite')
    n = data.shape[1]
    if window == 'hann':
        weights = np.hanning(n)
    elif window == 'rectangular':
        weights = np.ones(n)
    else:
        raise CalibrationError('window must be hann or rectangular')
    detrended = data - np.mean(data, axis=1, keepdims=True)
    fft = np.fft.rfft(detrended * weights[None, :], axis=1)
    scale = float(sample_rate_hz) * float(np.sum(weights ** 2))
    individual = np.abs(fft) ** 2 / scale
    if n % 2 == 0:
        individual[:, 1:-1] *= 2.0
    else:
        individual[:, 1:] *= 2.0
    mean_psd = np.mean(individual, axis=0)
    stderr = np.std(individual, axis=0, ddof=1) / math.sqrt(data.shape[0])
    bin_hz = float(sample_rate_hz) / n
    enbw_hz = float(sample_rate_hz) * float(np.sum(weights ** 2)) / float(np.sum(weights) ** 2)
    return {'frequency_hz': np.fft.rfftfreq(n, 1.0 / float(sample_rate_hz)),
            'psd_v2_per_hz': mean_psd, 'stderr_v2_per_hz': stderr,
            'individual_psd_v2_per_hz': individual,
            'bin_hz': bin_hz, 'enbw_hz': enbw_hz,
            'averages': int(data.shape[0]),
            'relative_standard_uncertainty': 1.0 / math.sqrt(data.shape[0])}


def passive_iq_spectrum(records, sample_rate_hz, centre_hz, window='hann'):
    """Return a centered two-sided PSD for complex down-converted records."""
    import numpy as np
    data = np.asarray(records, dtype=complex)
    if data.ndim != 2 or data.shape[0] < 2 or data.shape[1] < 16:
        raise CalibrationError('need at least two complex passive records')
    if not np.all(np.isfinite(data)) or float(sample_rate_hz) <= 0:
        raise CalibrationError('records and sample rate must be finite')
    n = data.shape[1]
    weights = np.hanning(n) if window == 'hann' else np.ones(n)
    data = data - np.mean(data, axis=1, keepdims=True)
    transformed = np.fft.fftshift(np.fft.fft(data * weights[None, :], axis=1), axes=1)
    individual = np.abs(transformed) ** 2 / (float(sample_rate_hz) * np.sum(weights ** 2))
    return {'frequency_hz': float(centre_hz) + np.fft.fftshift(np.fft.fftfreq(n, 1.0 / float(sample_rate_hz))),
            'psd_v2_per_hz': np.mean(individual, axis=0),
            'stderr_v2_per_hz': np.std(individual, axis=0, ddof=1) / math.sqrt(data.shape[0]),
            'individual_psd_v2_per_hz': individual,
            'bin_hz': float(sample_rate_hz) / n,
            'enbw_hz': float(sample_rate_hz) * np.sum(weights ** 2) / np.sum(weights) ** 2,
            'averages': int(data.shape[0]),
            'relative_standard_uncertainty': 1.0 / math.sqrt(data.shape[0])}


def fit_passive_resonator(frequency_hz, psd):
    """Fit the bench resonator's broad passive-noise peak over a sloping floor."""
    import numpy as np
    from scipy.optimize import least_squares
    f = np.asarray(frequency_hz, dtype=float)
    y = np.asarray(psd, dtype=float)
    if f.ndim != 1 or y.shape != f.shape or len(f) < 20 or np.any(y <= 0):
        raise CalibrationError('fit requires matching positive 1D spectrum arrays')
    order = np.argsort(f); f, y = f[order], y[order]
    span = float(f[-1] - f[0]); centre = 0.5 * (f[0] + f[-1])
    spacing = float(np.median(np.diff(f)))
    x = (f - centre) / max(span, np.finfo(float).tiny)
    floor = float(np.percentile(y, 20)); peak_i = int(np.argmax(y))
    amplitude = max(float(y[peak_i] - floor), np.finfo(float).tiny)
    linewidth0 = max(span / 20.0, spacing * 4.0)

    def model(parameters):
        b0, b1, peak, f0, linewidth = parameters
        return b0 + b1 * x + peak / (1.0 + 4.0 * ((f - f0) / linewidth) ** 2)

    initial = [floor, 0.0, amplitude, float(f[peak_i]), linewidth0]
    lower = [0.0, -10 * max(floor, amplitude), 0.0, f[0], spacing]
    upper = [10 * max(y), 10 * max(floor, amplitude), 20 * max(y), f[-1], span]
    scale = np.maximum(np.sqrt(y), np.sqrt(np.median(y)) * 0.2)
    result = least_squares(lambda values: (model(values) - y) / scale,
                           initial, bounds=(lower, upper), max_nfev=20000)
    if not result.success:
        raise CalibrationError('passive resonator fit did not converge')
    fitted = model(result.x); residual = y - fitted
    dof = max(1, len(y) - len(result.x)); rss = float(np.sum(residual ** 2))
    try:
        covariance = np.linalg.inv(result.jac.T.dot(result.jac)) * (2 * result.cost / dof)
        uncertainty = np.sqrt(np.maximum(np.diag(covariance), 0.0))
    except np.linalg.LinAlgError:
        covariance = np.full((len(result.x), len(result.x)), np.nan)
        uncertainty = np.full(len(result.x), np.nan)
    f0, linewidth = float(result.x[3]), float(result.x[4])
    return {'accepted': bool(np.isfinite(uncertainty[3]) and linewidth > 0),
            'resonator_state': 'passive_noise', 'f0_hz': f0,
            'linewidth_hz': linewidth, 'q': f0 / linewidth,
            'noise_floor': float(result.x[0]), 'slope': float(result.x[1]),
            'resonator_amplitude': float(result.x[2]), 'model': fitted,
            'residual': residual, 'rss': rss, 'reduced_residual': rss / dof,
            'covariance': covariance,
            'f0_uncertainty_hz': float(uncertainty[3]),
            'linewidth_uncertainty_hz': float(uncertainty[4])}

def assess_passive_records(records, adc_limit_vpk, clipping_fraction=0.98,
                           stationarity_cv_limit=0.20):
    """Check ADC headroom, clipping, crest factor and record stationarity."""
    import numpy as np
    data = np.asarray(records, dtype=float)
    if data.ndim != 2 or data.size == 0:
        raise CalibrationError('passive records must be a non-empty 2D array')
    limit = abs(_finite('adc_limit_vpk', adc_limit_vpk))
    peak = float(np.max(np.abs(data)))
    rms_by_record = np.sqrt(np.mean((data - np.mean(data, axis=1, keepdims=True)) ** 2,
                                    axis=1))
    mean_rms = float(np.mean(rms_by_record))
    cv = float(np.std(rms_by_record, ddof=1) / mean_rms) if len(rms_by_record) > 1 and mean_rms else float('inf')
    clipped = bool(peak >= clipping_fraction * limit)
    stationary = bool(cv <= stationarity_cv_limit)
    return {'accepted': bool(not clipped and stationary), 'peak_v': peak,
            'headroom_v': limit - peak,
            'crest_factor': peak / mean_rms if mean_rms else float('inf'),
            'rms_v': mean_rms, 'rms_cv': cv, 'clipped': clipped,
            'stationary': stationary}


def detect_self_oscillation(frequency_hz, psd, baseline_psd,
                            carrier_ratio_limit=20.0,
                            broadband_ratio_limit=3.0):
    """Detect a narrow carrier or broadband power increase over baseline."""
    import numpy as np
    values = np.asarray(psd, dtype=float)
    baseline = np.asarray(baseline_psd, dtype=float)
    if values.shape != baseline.shape or values.size < 5:
        raise CalibrationError('current and baseline PSD arrays must match')
    floor = np.maximum(baseline, np.finfo(float).tiny)
    ratio = values / floor
    carrier_ratio = float(np.max(ratio))
    broadband_ratio = float(np.sum(values) / max(np.sum(baseline), np.finfo(float).tiny))
    return {'unsafe': bool(carrier_ratio > carrier_ratio_limit or
                           broadband_ratio > broadband_ratio_limit),
            'carrier_ratio': carrier_ratio,
            'broadband_ratio': broadband_ratio,
            'carrier_frequency_hz': float(np.asarray(frequency_hz)[int(np.argmax(ratio))])}


def paired_k(q_plus, q_minus):
    """Compute K from closely spaced opposite-phase Q measurements."""
    q_plus = _finite('q_plus', q_plus)
    q_minus = _finite('q_minus', q_minus)
    if q_plus <= 0 or q_minus <= 0:
        raise CalibrationError('paired Q measurements must be positive')
    return (q_plus - q_minus) / (q_plus + q_minus)


def classify_dc_offset_sources(offset_records, resolution_floor_v=1.0e-3,
                               sigma_multiplier=5.0):
    """Compare guided IN1 captures without pretending that correlation proves grounding.

    A difference is called resolved only when it exceeds both a practical 1 mV
    floor and `sigma_multiplier` times the combined standard uncertainty.  The
    result distinguishes the local ADC/front-end reference from additions that
    appear when the high-impedance input, powered electronics, and resonator are
    connected.  It cannot by itself prove that an addition is a ground current.
    """
    if not isinstance(offset_records, dict):
        raise CalibrationError('offset_records must be a dictionary of guided DC captures')
    local_label = 'in1_local_50ohm_reference'
    complete_label = 'complete_resonator_feedback_off'
    missing = [name for name in (local_label, complete_label)
               if name not in offset_records]
    if missing:
        raise CalibrationError(
            'DC-source classification requires %s. Run the local 50 ohm zero-reference '
            'capture and the complete feedback-off resonator capture before interpreting '
            'the offset.' % ', '.join(missing))

    def value_uncertainty(label):
        row = offset_records[label]
        value = _finite(label + '.dc_offset_v', row['dc_offset_v'])
        uncertainty = abs(_finite(
            label + '.dc_offset_uncertainty_v',
            row.get('dc_offset_uncertainty_v', 0.0)))
        return value, uncertainty

    def comparison(name, positive_label, reference_label):
        positive, up = value_uncertainty(positive_label)
        reference, ur = value_uncertainty(reference_label)
        combined = math.sqrt(up*up + ur*ur)
        threshold = max(float(resolution_floor_v),
                        float(sigma_multiplier)*combined)
        difference = positive-reference
        return {'name': name, 'condition': positive_label,
                'reference': reference_label, 'difference_v': difference,
                'combined_standard_uncertainty_v': combined,
                'resolved_threshold_v': threshold,
                'resolved': bool(abs(difference) > threshold)}

    local, local_u = value_uncertainty(local_label)
    local_threshold = max(float(resolution_floor_v),
                          float(sigma_multiplier)*local_u)
    comparisons = {}
    if 'in1_open_high_impedance' in offset_records:
        comparisons['open_input_increment'] = comparison(
            'open input minus local 50 ohm reference',
            'in1_open_high_impedance', local_label)
    if 'powered_chain_safe_reference' in offset_records:
        comparisons['powered_chain_increment'] = comparison(
            'powered referenced chain minus local 50 ohm reference',
            'powered_chain_safe_reference', local_label)
        comparisons['resonator_increment'] = comparison(
            'complete resonator minus powered referenced chain',
            complete_label, 'powered_chain_safe_reference')
    comparisons['external_total_increment'] = comparison(
        'complete resonator path minus local 50 ohm reference',
        complete_label, local_label)

    conclusions = []
    if abs(local) > local_threshold:
        conclusions.append(
            'A resolved offset remains with a local 50 ohm zero reference. The '
            'external amplifier, resonator and trap grounding cannot be the sole '
            'cause of that component; raw Red Pitaya ADC/front-end offset is present.')
    else:
        conclusions.append(
            'The local 50 ohm reference is consistent with zero at this test '
            'resolution; a large complete-system offset is therefore added externally.')
    if comparisons.get('open_input_increment', {}).get('resolved'):
        conclusions.append(
            'Opening native high-impedance IN1 changes the mean. Pickup, input bias '
            'or capacitive charging is measurable; an open input is not a zero-voltage reference.')
    if comparisons.get('powered_chain_increment', {}).get('resolved'):
        conclusions.append(
            'The powered detection chain adds a resolved offset even with its input '
            'safely referenced. Check its output offset, supply return and load.')
    if comparisons.get('resonator_increment', {}).get('resolved'):
        conclusions.append(
            'Connecting the resonator/trap changes the powered-chain offset. This is '
            'compatible with a ground-reference or coupling contribution, but does not prove it.')
    external = comparisons['external_total_increment']
    if external['resolved']:
        conclusions.append(
            'The complete external path changes the local-reference result. A missing '
            'or duplicated ground can be among the causes; measure the unbonded chassis '
            'potential before making any bond.')
    else:
        conclusions.append(
            'The complete path does not add a resolved DC shift beyond the local '
            'reference. Grounding may still affect noise, but it does not explain the measured DC offset.')
    return {'local_reference_offset_v': local,
            'local_reference_uncertainty_v': local_u,
            'local_offset_resolved': bool(abs(local) > local_threshold),
            'comparisons': comparisons, 'conclusions': conclusions,
            'grounding_proven': False,
            'automatic_offset_correction_authorised': False}



def plan_passive_acquisition(f0_hz, q_estimate, expected_shift_hz=0.0,
                             clock_hz=125e6, min_decimation_log2=6,
                             max_decimation_log2=12,
                             bins_per_linewidth=20,
                             linewidths_each_side=2.5,
                             shift_guard_fraction=0.25,
                             centre_on_expected_shift=False,
                             usable_band_fraction=0.80,
                             min_samples=2048, max_samples=1 << 20):
    """Plan a supported complex-baseband capture without hiding truncation.

    The FPGA acquisition register is qualified only for DEC_LOG2 6..12.  The
    complete FFT interval is not automatically a trustworthy fitting interval:
    CIC gain falls with distance from the LO and the FIR is bypassed below
    DEC_LOG2=8.  Callers must separately apply an explicit absolute-frequency
    search mask and a measured or modelled acquisition-response correction.
    """
    f0 = _finite('f0_hz', f0_hz)
    q = _finite('q_estimate', q_estimate)
    shift = abs(_finite('expected_shift_hz', expected_shift_hz))
    minimum = int(min_decimation_log2)
    maximum = int(max_decimation_log2)
    if minimum < 6 or maximum > 12 or minimum > maximum:
        raise CalibrationError(
            'Unsupported acquisition decimation request %d..%d. The qualified '
            'FPGA register range is DEC_LOG2=6..12 (R=64..4096). Do not use '
            'DEC_LOG2=4 or 5: those settings were not commissioned and expose '
            'a very broad uncorrected CIC response.' % (minimum, maximum))
    if f0 <= 0 or q <= 0 or bins_per_linewidth < 5 or linewidths_each_side < 1.5:
        raise CalibrationError(
            'Invalid acquisition request: f0=%.9g Hz, Q=%.9g, bins/linewidth=%r, '
            'linewidths/side=%r. Frequency and Q must be positive, at least five '
            'bins/linewidth and 1.5 linewidths/side are required.' %
            (f0, q, bins_per_linewidth, linewidths_each_side))
    linewidth = f0 / q
    shift_allowance = shift_guard_fraction * shift if centre_on_expected_shift else shift
    required_usable_span = 2.0 * (shift_allowance + linewidths_each_side * linewidth)
    chosen_decimation = None
    for decimation in range(maximum, minimum - 1, -1):
        sample_rate = float(clock_hz) / (1 << decimation)
        if usable_band_fraction * sample_rate >= required_usable_span:
            chosen_decimation = decimation
            break
    if chosen_decimation is None:
        largest_span = usable_band_fraction * float(clock_hz) / (1 << minimum)
        raise CalibrationError(
            'The requested usable span is %.6g Hz, but the largest qualified '
            'capture provides %.6g Hz at DEC_LOG2=%d. Reduce the shift allowance '
            'or linewidth coverage, or use overlapping LO tiles; the notebook '
            'will not silently use unsupported decimation or truncate a resonance.' %
            (required_usable_span, largest_span, minimum))
    sample_rate = float(clock_hz) / (1 << chosen_decimation)
    target_bin = linewidth / float(bins_per_linewidth)
    needed = max(int(min_samples), int(math.ceil(sample_rate / target_bin)))
    samples = 1 << int(math.ceil(math.log(max(needed, 2), 2)))
    if samples > int(max_samples):
        raise CalibrationError(
            'Q %.6g at %.9g Hz requires %d samples for %.1f bins/linewidth, '
            'above the safe limit %d. Increase the capture limit only after DMA '
            'memory bounds and acquisition time have been reviewed.' %
            (q, f0, samples, bins_per_linewidth, max_samples))
    bin_hz = sample_rate / samples
    return {'centre_hz': f0 + (float(expected_shift_hz)
                               if centre_on_expected_shift else 0.0),
            'reference_f0_hz': f0, 'q_estimate': q,
            'linewidth_hz': linewidth,
            'expected_shift_hz': float(expected_shift_hz),
            'centre_on_expected_shift': bool(centre_on_expected_shift),
            'decimation_log2': chosen_decimation,
            'sample_rate_hz': sample_rate, 'samples_per_record': samples,
            'bin_hz': bin_hz, 'bins_per_linewidth': linewidth / bin_hz,
            'full_span_hz': sample_rate,
            'usable_span_hz': usable_band_fraction * sample_rate,
            'record_duration_s': samples / sample_rate,
            'usable_band_fraction': float(usable_band_fraction),
            'fir_expected_state': ('active' if chosen_decimation >= 8 else 'bypassed'),
            'qualified_decimation_range': [6, 12]}


## 4 — Review settings and initial acquisition

**What this cell does.** It prints the fixed bench topology and calculates the first acquisition.
The raw capture is intentionally wider than the accepted 1 MHz search interval so filter edges
remain visible, but resonance finding is restricted to 40.4–41.4 MHz.

**Why it is needed.** The earlier run used unsupported decimation R=16 and interpreted the broad
CIC passband as a resonator. This cell now enforces the FPGA-qualified range R=64…4096 and reports
whether the FIR is expected to be active or bypassed.

**Hardware and wiring.** This cell performs calculations only. It does not read or write the Red
Pitaya. The campaign label must be changed whenever the installed detection or feedback path changes;
individual component gains and losses are deliberately determined from measurements instead of entered here.

**Expected result and pass criteria.** The discovery plan should select DEC_LOG2=6 (R=64), about
1.953 MHz raw span, and identify a 1 MHz accepted search interval. Any unsupported decimation or
insufficient bandwidth produces an explanatory error rather than a truncated spectrum.

**Failure meaning and rerun safety.** A failure is a planning/configuration problem, not evidence
about the resonator. Correct the declared frequency/Q/search settings and rerun; no hardware was
touched.


In [ ]:
def configuration_report():
    nominal_linewidth_hz = CENTRE_HZ / INITIAL_Q_ESTIMATE
    print('Campaign configuration')
    print('  analogue component values are deliberately not assumed')
    print('  effective gain and phase will be measured from the resonator response')
    print('  IN1 mode: %s' % TOPOLOGY['input_mode'])
    print('  high-pass filter in feedback path: %s' %
          TOPOLOGY['high_pass_filter_in_feedback_path'])
    print('  OUT2 is held off and is not a campaign prerequisite')
    print('  nominal centre %.3f MHz, Q %.1f, linewidth %.1f kHz' %
          (CENTRE_HZ / 1e6, INITIAL_Q_ESTIMATE, nominal_linewidth_hz / 1e3))
    print('  ADC operating guard %.3f Vpk' %
          ADC_OPERATING_GUARD_VPK)
    print('  Manual measurements use independent baseline/feedback settings in Step 6A.')
    print('  Step 7 verifies the acquisition filters; it does not alter OUT1 feedback.')
    return {
        'topology_digest': _digest(TOPOLOGY),
        'passive_measurement_mode': 'ready_for_live_safety_tests',
    }

CONFIGURATION_REPORT = configuration_report()
TOPOLOGY_RECORD = _seal({'schema_version': SCHEMA_VERSION,
                         'kind': 'bench_loop_topology',
                         'config': copy.deepcopy(TOPOLOGY)})
INITIAL_ACQUISITION_PLAN = plan_passive_acquisition(
    CENTRE_HZ, INITIAL_Q_ESTIMATE, 0.0,
    min_decimation_log2=MIN_ACQUISITION_DECIMATION_LOG2,
    max_decimation_log2=MAX_ACQUISITION_DECIMATION_LOG2,
    bins_per_linewidth=BINS_PER_LINEWIDTH,
    linewidths_each_side=LINEWIDTHS_EACH_SIDE,
    max_samples=MAX_CAPTURE_SAMPLES)
print('Initial passive capture example: %.1f kHz raw span, %.1f kHz planner span, %.1f Hz bins, DEC_LOG2=%d' %
      (INITIAL_ACQUISITION_PLAN['full_span_hz'] / 1e3,
       INITIAL_ACQUISITION_PLAN['usable_span_hz'] / 1e3,
       INITIAL_ACQUISITION_PLAN['bin_hz'],
       INITIAL_ACQUISITION_PLAN['decimation_log2']))
print('Suggested display band: %.3f to %.3f MHz; FIR expected %s' %
      ((CENTRE_HZ-DISCOVERY_SEARCH_HALF_WIDTH_HZ)/1e6,
       (CENTRE_HZ+DISCOVERY_SEARCH_HALF_WIDTH_HZ)/1e6,
       INITIAL_ACQUISITION_PLAN['fir_expected_state']))


## 5 — Define register access

**Purpose.** defines the real and mock register transports and plotting compatibility helpers.

**Hardware.** no hardware is opened until a class is instantiated. **Wiring.** unchanged. **Pass.** definitions complete without output. **Failure.** Python dependency/version problem. **Safe to rerun.** yes.


In [ ]:
class MemoryTransport(object):
    """Byte-addressable mock register block used by notebook tests."""
    def __init__(self, base, span):
        self.base = base
        self.m = bytearray(span)
        self.fd = None
        self.fail_reads = frozenset()
        self.fail_writes = frozenset()
        self.write_log = []

    def close(self):
        pass


class DevMemTransport(object):
    """Real Red Pitaya register mapping. Construction is the hardware boundary."""
    def __init__(self, base, span):
        self.base = base
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m = mmap.mmap(self.fd, span, offset=base)

    def close(self):
        self.m.close()
        os.close(self.fd)


class Bus(object):
    """One register block with an injectable real or memory-backed transport."""
    def __init__(self, base, span=0x1000, transport_factory=None):
        factory = transport_factory or TRANSPORT_FACTORY or DevMemTransport
        self.transport = factory(base, span)
        self.base = base
        self.fd = self.transport.fd
        self.m = self.transport.m

    def w(self, off, val):
        if off in getattr(self.transport, 'fail_writes', frozenset()):
            raise IOError('injected register-write failure at 0x%X' % off)
        self.m[off:off + 4] = struct.pack('<I', int(val) & 0xFFFFFFFF)
        if hasattr(self.transport, 'write_log'):
            self.transport.write_log.append((off, int(val) & 0xFFFFFFFF))

    def r(self, off):
        if off in getattr(self.transport, 'fail_reads', frozenset()):
            raise IOError('injected register-read failure at 0x%X' % off)
        return struct.unpack('<I', bytes(self.m[off:off + 4]))[0]

    def rs(self, off):
        value = self.r(off)
        return value - (1 << 32) if value & 0x80000000 else value

    def close(self):
        self.transport.close()


def shutdown_devices(generator, phase_shift, acquisition):
    """Attempt every shutdown action and verify zero feedback coefficients."""
    errors = []
    for label, action in (('generator', generator.off),
                          ('feedback', phase_shift.off),
                          ('acquisition', acquisition.stop)):
        try:
            action()
        except BaseException as exc:
            errors.append('%s: %s' % (label, exc))
    try:
        readback = phase_shift.readback()
        if int(readback['ci']) != 0 or int(readback['cq']) != 0:
            errors.append('feedback coefficient readback is not zero: %r' %
                          (readback,))
    except BaseException as exc:
        errors.append('feedback readback: %s' % exc)
    if errors:
        raise RuntimeError('safe shutdown could not be verified; physically '
                           'disconnect OUT1: ' + '; '.join(errors))
    return True


# =============================================================================
#  PLOTTING COMPATIBILITY  -  this board runs matplotlib 1.x
# =============================================================================
#  The 'C0'..'C9' colour shorthand arrived in matplotlib 2.0. On 1.x it is not
#  rejected when you pass it - it is accepted, stored, and then raises
#
#      ValueError: to_rgba: Invalid rgba arg "C3"
#
#  from inside the draw, at which point the figure is already built. What you see
#  is a traceback followed by a bare <matplotlib.figure.Figure at 0x...> instead
#  of a plot, and every OTHER cell still runs, so it looks like a plotting
#  nuisance rather than a version problem. Every colour in this notebook is
#  therefore written out in full.
CLR_BLUE   = '#1f77b4'
CLR_ORANGE = '#ff7f0e'
CLR_GREEN  = '#2ca02c'
CLR_RED    = '#d62728'
CLR_PURPLE = '#9467bd'
CLR_GREY   = '0.5'

# The paper's figures use the modern perceptual colour maps; matplotlib 1.x has
# none of them, so viridis is rebuilt here from its own anchors - identical
# output on any version.
VIRIDIS_ANCHORS = [
    (0.2670, 0.0049, 0.3294), (0.2823, 0.0950, 0.4173), (0.2788, 0.1755, 0.4834),
    (0.2590, 0.2515, 0.5247), (0.2297, 0.3224, 0.5457), (0.1994, 0.3876, 0.5546),
    (0.1727, 0.4488, 0.5579), (0.1490, 0.5081, 0.5573), (0.1276, 0.5669, 0.5506),
    (0.1206, 0.6258, 0.5335), (0.1579, 0.6838, 0.5017), (0.2461, 0.7389, 0.4520),
    (0.3692, 0.7889, 0.3829), (0.5160, 0.8312, 0.2943), (0.6785, 0.8637, 0.1895),
    (0.8456, 0.8873, 0.0997), (0.9932, 0.9062, 0.1439)]
CMAP_SEQ = matplotlib.colors.LinearSegmentedColormap.from_list(
    'viridis_local', VIRIDIS_ANCHORS, N=256)
CMAP_DIV = matplotlib.colors.LinearSegmentedColormap.from_list(
    'coolwarm_local',
    [(0.230, 0.299, 0.754), (0.553, 0.690, 0.996), (0.866, 0.866, 0.866),
     (0.968, 0.718, 0.612), (0.706, 0.016, 0.150)], N=256)
CMAP_SEQ.set_bad('black')
CMAP_DIV.set_bad('black')

# One place for the look of every figure. Setting it here means no cell needs to
# repeat grid, font and layout calls.
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110, 'figure.autolayout': True,
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'legend.fontsize': 8, 'legend.framealpha': 0.9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.grid': True, 'grid.color': '0.88', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'lines.linewidth': 1.2, 'lines.markersize': 4,
})

try:                       # inline figures when running under Jupyter...
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:          # ...and a file-only backend when running as a script
    matplotlib.use('Agg')

print('numpy %s, matplotlib %s' % (np.__version__, matplotlib.__version__))
if int(matplotlib.__version__.split('.')[0]) < 2:
    print('  matplotlib 1.x detected - using explicit hex colours (see above)')


## 6 — Define the acquisition interface

**Purpose.** defines the existing acquisition register map locally. **Hardware.** definitions only. **Wiring.** unchanged. **Pass.** no output. **Failure.** syntax/version problem. **Safe to rerun.** yes.


In [ ]:
# =============================================================================
#  ACQUISITION SYSTEM - base 0x40600000
# =============================================================================
#  VOLTMETER: tune the local oscillator to the drive, read PEEK_I/PEEK_Q.
#  RECORDER : stream to memory for spectra.
#  ORDER MATTERS: disable -> write settings -> reset -> enable. The decimator
#  holds a running total that assumes a fixed R. Retuning the oscillator is one
#  register write and is safe at any time.
# =============================================================================
class Acq(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = None

    # ---- setup ----
    def start(self, dec_log2, src=0):
        """Bring the chain up at a given decimation. Safe to call repeatedly."""
        self.w(0x00, 0)                 # disable while settings change
        self.w(0x08, dec_log2)          # decimation R = 2**dec_log2
        self.w(0x0C, src)               # 0 = IN1, 1 = IN2
        self.w(0x00, 1 | 2)             # enable + soft reset (self-clearing)
        self.w(0x00, 1)                 # running
        self.dec = dec_log2

    def stop(self):
        self.w(0x00, 0)

    # ---- tuning ----
    def tune_word(self, finc):
        """Set the local oscillator by its raw 32-bit frequency word."""
        self.w(0x04, int(finc) & 0xFFFFFFFF)

    def tune(self, f):
        """Set the local oscillator to the nearest word to f hertz."""
        self.tune_word(int(round(f / FS_CLK * (1 << 32))))

    # ---- voltmeter ----
    def measure(self, navg=None):
        """Average navg in-phase/quadrature pairs; return one complex number.

        At DC both are steady, so this is coherent averaging: a real signal adds
        linearly while noise averages down as the square root of navg."""
        navg = N_AVG if navg is None else navg
        zi = 0.0
        zq = 0.0
        for _ in range(navg):
            zi += self.rs(0x18)
            zq += self.rs(0x1C)
        return complex(zi / navg, zq / navg)

    # ---- housekeeping ----
    def f_out(self):
        """Output sample rate at the current decimation [Hz]."""
        return FS_CLK / (1 << self.dec)

    def window_hz(self):
        """Half-width of the CLEAN part of the receiver window [Hz].

        The decimator rejects by better than 75 dB only over the middle fifth of
        its output band, so the usable half-width is 0.1 x the output rate."""
        return 0.10 * self.f_out()

    def build_id(self):
        return self.r(0x48)

    def status2(self):
        return self.r(0x38)




## 7 — Define the signal generator

**Purpose.** Defines the existing generator interface so both outputs can be forced to zero during setup and emergency shutdown. **Hardware.** Definitions only; this cell does not produce a signal. **Pass.** No output. **Safe to rerun.** Yes.


In [ ]:
# =============================================================================
#  SIGNAL GENERATOR - base 0x40200000, stock Red Pitaya module
# =============================================================================
#  Replays a 16384-entry table; frequency is one register write.
#      f = step x 125e6 / 2**30      (about 0.116 Hz per step)
#  Registers: 0x04 amplitude (0x2000 = x1.0), 0x08 table size, 0x0C start,
#  0x10 step, table at 0x10000. Control 0x00: 0x11 free-run, 0xC0 held at zero.
# =============================================================================
class Generator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
        self._step = None

    def load_sine(self):
        """Write one cycle of a sine into the table. Takes a second or two:
        16384 single-word writes, and single-word is not negotiable here."""
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)     # table size, fixed point, inclusive
        self.w(0x0C, 0)                     # start replaying from entry 0

    def freq_word(self, f):
        """Nearest 30-bit frequency word to f hertz."""
        return int(round(f / FS_CLK * (1 << 30)))

    def set_word(self, step):
        """Set frequency by raw word; return the frequency actually produced."""
        self._step = int(step)
        self.w(0x10, self._step)
        return self._step * FS_CLK / float(1 << 30)

    def freq(self, f):
        """Set frequency in hertz; return the frequency actually produced."""
        return self.set_word(self.freq_word(f))

    def lo_word(self):
        """The 32-bit local-oscillator word that lands this signal EXACTLY at
        0 Hz. The generator counts in units of 2**-30 of the clock and the local
        oscillator in units of 2**-32, so the conversion is a factor of four and
        carries no rounding error at all."""
        return 4 * self._step

    def amp(self, frac):
        """Output amplitude as a fraction of full scale. CLAMPED, deliberately.

        The register is Q1.13 with 0x2000 = x1.0, and it happily accepts up to
        0x3FFF = x1.9999. The write below is masked with & 0x3FFF, so before this
        guard existed the mask turned mistakes into silent, WRONG drive levels:

            frac = -0.5  ->  0x3000  ->  x1.5   (a sign slip drives 50% HARDER)
            frac =  2.0  ->  0x0000  ->  x0.0   (silence, and no complaint)
            frac =  1.5  ->  0x3000  ->  x1.5   (clipped square wave, not a sine)

        Every one of those is silent on the bench. Anything outside
        [0, DRIVE_MAX] is now refused instead, so the only way to drive the coil
        harder is to raise DRIVE_MAX in the configuration cell on purpose."""
        frac = float(frac)
        if not np.isfinite(frac):
            raise ValueError('drive amplitude is not a finite number: %r' % frac)
        if frac < 0.0:
            raise ValueError(
                'negative drive amplitude (%.4f). A sign has no meaning here, and '
                'the register would silently wrap it to x%.4f.'
                % (frac, (int(round(frac * 8192)) & 0x3FFF) / 8192.0))
        if frac > DRIVE_MAX:
            raise ValueError(
                'drive %.4f exceeds DRIVE_MAX = %.4f. Nothing in this notebook '
                'may drive harder than the ceiling; raise it in the configuration '
                'cell if that is really what you want.' % (frac, DRIVE_MAX))
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)

    def _control_a_b_zero(self, a_control):
        """Write channel-A control while forcing generator channel B to zero.

        OUT2 is the phase-shifter monitor plus ASG channel B. A plain channel-A
        write clears B's control bits, so stale B state from another program could
        corrupt an external measurement. Bits 23:22 hold B at zero/reset."""
        self.w(0x24, 0)                    # B amplitude and DC offset both zero
        self.w(0x00, 0x00C00000 | int(a_control))

    def assert_b_zero(self):
        """Require channel B amplitude/offset zero and its engine held reset."""
        packed, control = self.r(0x24), self.r(0x00)
        if packed != 0 or ((control >> 22) & 3) != 3:
            raise RuntimeError('generator B is not zero: AMP/DC=0x%08X CTRL=0x%08X'
                               % (packed, control))
        return {'b_amp_dc': packed, 'control': control}

    def run(self):
        self._control_a_b_zero(0x40)       # reset the A read pointer
        self._control_a_b_zero(0x11)       # A free-runs; B remains forced zero

    def off(self):
        self.amp(0.0)
        self._control_a_b_zero(0xC0)       # both outputs held at zero/reset




## 8 — Define the feedback phase shifter

**Purpose.** defines coefficient, delay, and readback operations for OUT1 feedback. OUT2 is not configured. **Hardware.** definitions only. **Pass.** no output. **Safe to rerun.** yes.


In [ ]:
# =============================================================================
#  PHASE SHIFTER - base 0x40300000   (Step 8 only)
# =============================================================================
#  Output = A cos(phi) x I + A sin(phi) x Q, Q being the input delayed a quarter
#  period. Two corrections, both from the module header:
#   1. The quarter-period delay is under one clock at 40 MHz, so add whole signal
#      periods and pick the k whose fraction sits nearest 0 or 1.
#   2. Fixed THREE-sample latency (L = 3 since FIX 6; one sample is 115 deg at
#      40 MHz, so a stale value rotates the whole axis). Subtracted from the
#      request so the delivered phase is the requested one.
#  Both describe the DIGITAL path only. Step 8 finds its phases by measurement.
#  "Feedback off" is coefficients (0, 0), NOT the bypass bit.
# =============================================================================
class PhaseShift(Bus):

    LATENCY = 3      # datapath latency in samples (L = 3 since FIX 6; see above)

    def __init__(self):
        Bus.__init__(self, 0x40300000)
        self.f_ref = None

    def config_delay(self, f):
        """Programme the quarter-period delay line for frequency f."""
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000.0:
                continue
            frac = D - np.floor(D)
            score = min(frac, 1.0 - frac)       # interpolator error peaks at 0.5
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        if best is None:
            raise ValueError('no usable quarter-period delay at %.3f MHz' % (f / 1e6))
        self.w(0x08, best[1])                       # whole clocks
        self.w(0x0C, int(round(best[2] * 16384)))   # fraction, Q0.14
        self.w(0x10, 0)                             # bypass OFF
        self.f_ref = f
        return best[1], best[2]

    def set(self, A, phi_deg):
        """Apply gain/phase and prove the exact CI/CQ words were accepted."""
        if self.f_ref is None:
            raise RuntimeError('call config_delay() first')
        if not (0.0 <= A < 1.99):
            # A x 16384 must stay inside int16. At A = 2.0 the coefficient is
            # 32768, which wraps to -32768 and silently INVERTS the feedback.
            raise ValueError('gain must be in [0, 1.99)')
        phi = phi_deg - 360.0 * self.LATENCY * self.f_ref / FS_CLK
        ci = int(round(A * np.cos(np.radians(phi)) * 16384)) & 0xFFFF
        cq = int(round(A * np.sin(np.radians(phi)) * 16384)) & 0xFFFF
        self.w(0x00, ci); self.w(0x04, cq)
        if self.readback()['ci'] != ci or self.readback()['cq'] != cq:
            raise RuntimeError('phase/gain coefficient readback failed')
        return ci, cq

    def readback(self):
        """All state that can alter feedback or the OUT2 observation."""
        return {'ci': self.r(0x00) & 0xFFFF, 'cq': self.r(0x04) & 0xFFFF,
                'delay': self.r(0x08), 'delta': self.r(0x0C),
                'bypass': self.r(0x10) & 1, 'monitor': self.r(0x18) & 0x3F}

    def assert_held(self, expected):
        got = self.readback()
        changed = {k: (expected[k], got[k]) for k in expected if got[k] != expected[k]}
        if changed:
            raise RuntimeError('feedback/OUT2 settings changed during replay: %r' % changed)
        return got

    def off(self):
        self.w(0x00, 0)
        self.w(0x04, 0)

    def monitor(self, source='input', shift=0):
        """Route a signal to OUT2 and verify the register accepted it.

        source='input' observes the coil/ADC signal used by feedback; 'output'
        observes the phase-shifted feedback contribution. shift is a saturating
        left shift (0..7). Keep it at zero until the external level is known."""
        sources = {'off': 0, 'input': 1, 'output': 2}
        if source not in sources:
            raise ValueError("monitor source must be 'off', 'input' or 'output'")
        shift = int(shift)
        if not (0 <= shift <= 7):
            raise ValueError('monitor shift must be in [0, 7]')
        value = sources[source] | (shift << 3)
        self.w(0x18, value)
        got = self.r(0x18)
        if (got & 0x3F) != value:
            raise RuntimeError('OUT2 monitor readback failed: wrote 0x%02X, read 0x%08X; check the bitstream' % (value, got))
        return got

    def monitor_clipped(self):
        """True if OUT2 monitor scaling has clipped since configuration."""
        return bool(self.r(0x18) & (1 << 8))

    def monitor_off(self):
        self.monitor('off', 0)



## 9 — Define raw ADC capture

**Purpose.** defines ADC headroom and clipping captures. **Hardware.** definitions only. **Wiring.** IN1 remains directly connected in native high-impedance mode; no external terminator is fitted. **Pass.** no output. **Safe to rerun.** yes.


In [ ]:
# =============================================================================
#  INTERNAL SCOPE - base 0x40100000
# =============================================================================
#  16384 raw samples at up to 125 MSa/s. The only overload guard on the real
#  path: the emulator's sticky clip flag is gone once a real coil is measured.
#  Sees the same signal as the rest of the chain, so with the emulator enabled it
#  shows the emulator.
# =============================================================================
SC_BASE = 0x40100000
SC_SPAN = 0x40000
SC_BUF_A = 0x10000
SC_BUF_B = 0x20000
SC_NMAX = 1 << 14


class Scope(Bus):

    def __init__(self):
        Bus.__init__(self, SC_BASE, span=SC_SPAN)

    def _buf(self, off, n):
        """Read n samples, one 32-bit word at a time (this is a register block,
        not DDR). The buffer holds 14-bit two's-complement values."""
        ba = bytearray(4 * n)
        m = self.m
        for i in range(n):
            o = off + 4 * i
            ba[4 * i:4 * i + 4] = m[o:o + 4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=SC_NMAX, chan='A'):
        """Capture and return (samples, sample_rate). dec = 1 is every sample."""
        self.w(0x14, dec)
        self.w(0x10, n - 1)
        self.w(0x00, 2)        # reset the capture engine
        self.w(0x00, 1)        # arm
        self.w(0x04, 1)        # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        off = SC_BUF_A if chan == 'A' else SC_BUF_B
        return self._buf(off, n), FS_CLK / float(dec)

    def peak_frac(self, n=4096):
        """TOTAL peak excursion as a fraction of full scale. 1.0 means clipping.

        Deliberately includes everything - the converter's DC offset, broadband
        noise, out-of-band pickup - because all of it counts towards clipping.
        That makes it the right guard and the WRONG way to measure the size of a
        tone. Use tone_frac() for that."""
        x, _ = self.acquire(dec=1, n=n)
        return float(np.max(np.abs(x))) / 8191.0

    def tone_frac(self, f, n=4096):
        """Amplitude of the component at f alone, as a fraction of full scale.

        WHY THIS EXISTS. peak_frac() answers "how close am I to clipping". It was
        also being used to answer "how much 40 MHz is arriving", and those are
        not the same question. max|x| is inflated by the converter's DC offset
        and by every other thing in the record, so on this board it read about
        four times the true tone amplitude - which then over-predicted the
        feedback loop gain by the same factor.

        A least-squares projection onto cos and sin at the known frequency fixes
        it: the mean is removed first, so DC cannot contribute, and anything not
        at f is orthogonal to both basis vectors and averages away. No window and
        no scalloping loss, because f is known exactly rather than searched."""
        x, fs = self.acquire(dec=1, n=n)
        x = x - np.mean(x)                       # the converter's DC offset
        t = np.arange(len(x)) / fs
        basis = np.column_stack([np.cos(2.0 * np.pi * f * t),
                                 np.sin(2.0 * np.pi * f * t)])
        coef, _, _, _ = np.linalg.lstsq(basis, x, rcond=-1)
        return float(np.hypot(coef[0], coef[1])) / 8191.0



## 10 — Define bounded DMA capture

**Purpose.** defines bounded complex-IQ capture. Real DMA memory is discovered lazily; tests inject a capture backend. **Hardware.** no access until capture is called. **Pass.** definitions complete. **Safe to rerun.** yes before acquisition.


In [ ]:
# =============================================================================
#  MEMORY CAPTURE
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]


def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None


MEM_START = None
MEM_SIZE = 0x00800000
MEM_STOP = None
POISON = 0xDEADBEEF


def configure_capture_memory():
    """Discover safe DMA bounds only when real hardware capture is requested."""
    global MEM_START, MEM_STOP
    blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
    if not blob or len(blob) < 8:
        raise RuntimeError('cannot read the device-tree memory node')
    declared_end = _be32(blob, 0) + _be32(blob, 1)
    if declared_end >= 512 * 1024 ** 2:
        raise RuntimeError('no safe memory above the declared end')
    MEM_START = (declared_end + 0xFFF) & ~0xFFF
    MEM_STOP = MEM_START + (MEM_SIZE & ~7) - 8
    print('capture region 0x%08X .. 0x%08X inclusive (%d MB)' %
          (MEM_START, MEM_STOP + 7, MEM_SIZE // (1024 * 1024)))

def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, ((nbytes + 4095) // 4096) * 4096, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close()
        os.close(fd)


def _read_region(nbytes):
    """Bulk read of DDR. try/finally because everything here can raise, and a
    leaked mapping inside a scan loop surfaces much later as an unrelated-
    looking out-of-memory error."""
    fd = mm = None
    try:
        fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
        return mm[0:nbytes]
    finally:
        if mm is not None:
            mm.close()
        if fd is not None:
            os.close(fd)


def capture(lo_word, dec_log2, n_samples, settle=None, fir_mode='auto'):
    """Record n_samples of the complex stream. Returns a complex numpy array.

    lo_word is the RAW 32-bit local-oscillator word, so a caller that wants an
    exact zero-frequency landing can pass gen.lo_word() and get it. Raises on
    any transport fault rather than returning plausible garbage."""
    settle = SETTLE if settle is None else float(settle)
    if (int(dec_log2) != dec_log2 or not 6 <= dec_log2 <= 12 or
            int(n_samples) != n_samples or n_samples <= 0 or
            fir_mode not in ('auto', 'bypass') or not np.isfinite(settle) or settle < 0):
        raise ValueError('Capture requires integer decimation 6..12, positive samples, nonnegative settling and FIR auto/bypass.')
    dec_log2, n_samples = int(dec_log2), int(n_samples)
    fir_control = 64 if fir_mode == 'bypass' else 0
    expected_bypass = fir_mode == 'bypass' or dec_log2 < 8
    settle = max(settle, (8 if expected_bypass else 317)*(1 << dec_log2)/FS_CLK)
    if CAPTURE_BACKEND is not None:
        if fir_mode == 'auto':
            return CAPTURE_BACKEND(lo_word, dec_log2, n_samples, settle)
        return CAPTURE_BACKEND(lo_word, dec_log2, n_samples, settle, fir_mode=fir_mode)
    if MEM_START is None:
        configure_capture_memory()
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples exceeds the capture region' % n_samples)

    try:
        _poison(n_samples * 8)

        acq.w(0x00, 0)                       # disable while settings change
        acq.w(0x08, dec_log2)
        acq.w(0x0C, 0)
        acq.dec = dec_log2
        acq.tune_word(lo_word)
        acq.w(0x28, MEM_START)
        acq.w(0x2C, MEM_STOP)

        lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
        if lo_rb != MEM_START or hi_rb != MEM_STOP:
            acq.w(0x00, 0)
            observed_build = acq.build_id()
            raise RuntimeError(
                'DMA CAPTURE WAS NOT ARMED: the notebook wrote the qualified capture '
                'region 0x%08X..0x%08X, but the FPGA read back 0x%08X..0x%08X. '
                'The writer remains stopped, so no unbounded memory write occurred. '
                'Observed build ID is 0x%08X; expected %s. This usually means that a '
                'stock Red Pitaya application or another notebook replaced/controls '
                'the custom bitstream, or the hardware definitions were run before '
                'the qualified bitstream finished loading. Close other Red Pitaya apps, '
                'rerun Step 3, confirm the build ID, and retry from the passive baseline. '
                'Do not bypass this check.' %
                (MEM_START, MEM_STOP, lo_rb, hi_rb, observed_build, BUILD_ID_EXPECTED))

        if acq.r(0x08) != dec_log2 or acq.r(0x04) != (int(lo_word) & 0xFFFFFFFF):
            raise RuntimeError('LO/decimation readback failed; writer remains stopped. Close competing applications.')
        acq.w(0x00, fir_control | 1 | 2)                   # run at the new tuning...
        acq.w(0x00, fir_control | 1)
        time.sleep(settle)                   # ...let the pipeline flush...
        acq.w(0x00, fir_control | 1 | 16 | 128)            # ...then arm with a fresh writer
        acq.w(0x00, fir_control | 1 | 16)

        t_record = n_samples * (1 << dec_log2) / FS_CLK
        deadline = time.time() + t_record + 0.5
        while int(acq.r(0x34)) < n_samples and time.time() < deadline:
            time.sleep(0.001)
        acq.w(0x00, fir_control | 1)                       # disarm
        time.sleep(0.002)                    # let the last burst land

        st2 = acq.r(0x38)
        if (st2 & 9) or ((st2 & 2) and not expected_bypass):
            raise RuntimeError('DMA/FIR status 0x%X indicates dropped samples, write failure or active FIR overrun; discard the record.' % st2)
        if bool(st2 & 4) != expected_bypass:
            raise RuntimeError('FIR bypass status differs from requested mode; discard the record and recheck FPGA identity.')
        got = min(int(acq.r(0x34)), n_samples)
        if got < n_samples:
            raise RuntimeError('short record: asked %d, got %d' % (n_samples, got))

        buf = _read_region(got * 8)
        u32 = np.frombuffer(buf, dtype='<u4')
        limit = max(8, 0.001 * u32.size)
        if int(np.count_nonzero(u32 == POISON)) > limit:
            # A word still inside the writer's burst buffer is not a lost word. If
            # it is genuinely lost it will still be poison in 20 ms.
            time.sleep(0.02)
            u32 = np.frombuffer(_read_region(got * 8), dtype='<u4')
            if int(np.count_nonzero(u32 == POISON)) > limit:
                raise RuntimeError('unwritten words in the record - transport fault')

        iq = u32.astype(np.int64)
        i = np.where(iq[0::2] & 0x80000000, iq[0::2] - (1 << 32), iq[0::2])
        q = np.where(iq[1::2] & 0x80000000, iq[1::2] - (1 << 32), iq[1::2])
        return i.astype(float) + 1j * q.astype(float)
    finally:
        acq.w(0x00, 0)  # Always stop acquisition/writer; do not change OUT1 here.


## 11 — Verify hardware identity and shutdown

**Purpose.** Opens the existing register blocks, performs a full shutdown, and verifies the FPGA identity. OUT2 is kept off. `full_shutdown()` also stops acquisition and verifies the safe state.

**Wiring.** Keep the current IN1 and high-pass feedback paths unchanged. **Expected result.** Zero generator and feedback coefficients, OUT2 off, acquisition stopped. **Safe to rerun.** Yes.


In [ ]:
acq = Acq()
gen = Generator()
ps = PhaseShift()
sc = Scope()
# The campaign begins with the installed high-pass filter and establishes safe
# effective gain from low coefficient counts using resonator measurements.

def feedback_off():
    """Clear OUT1 feedback and both generators; OUT2 is normally already off."""
    errors = []
    for label, action in (('generator', gen.off), ('feedback', ps.off), ('acquisition', acq.stop)):
        try:
            action()
        except BaseException as exc:
            errors.append('%s: %s' % (label, exc))
    try:
        readback = ps.readback()
        if readback['ci'] or readback['cq']:
            errors.append('feedback coefficient readback is nonzero')
        gen.assert_b_zero()
    except BaseException as exc:
        errors.append('readback: %s' % exc)
    if errors:
        try:
            ps.monitor_off()
        finally:
            raise RuntimeError('feedback shutdown failed; disconnect OUT1: ' + '; '.join(errors))
    print('SAFE FEEDBACK STATE: both generators and OUT1 feedback are zero; acquisition is stopped.')
    return True

def out2_monitor_off():
    """Disable only the IN1-to-OUT2 monitor and verify its register readback."""
    ps.monitor_off()
    state = ps.readback()
    if state['monitor']:
        full_shutdown()
        raise RuntimeError('OUT2 monitor disable readback failed; full shutdown attempted')
    print('OUT2 monitor disabled and verified. OUT1 feedback state was not changed.')
    return True

def full_shutdown():
    """Attempt every safe action, including disabling the OUT2 connector."""
    errors = []
    for label, action in (('generator', gen.off), ('feedback', ps.off),
                          ('OUT2 monitor', ps.monitor_off), ('acquisition', acq.stop)):
        try:
            action()
        except BaseException as exc:
            errors.append('%s: %s' % (label, exc))
    try:
        state = ps.readback()
        gen.assert_b_zero()
        if state['ci'] or state['cq'] or state['monitor']:
            errors.append('nonzero final register state: %r' % state)
    except BaseException as exc:
        errors.append('final readback: %s' % exc)
    if errors:
        raise RuntimeError('FULL SHUTDOWN COULD NOT BE VERIFIED: ' + '; '.join(errors))
    print('FULL SAFE STATE: generators, OUT1, OUT2 and acquisition are off.')
    return True

def disable_everything():
    """Memorable emergency alias: disable generators, feedback, OUT2 and acquisition."""
    return full_shutdown()

full_shutdown()
CURRENT_BUILD_ID = '0x%08X' % acq.build_id()
if CURRENT_BUILD_ID.lower() != str(BUILD_ID_EXPECTED).lower():
    full_shutdown()
    raise RuntimeError('wrong FPGA identity: found %s, expected %s' %
                       (CURRENT_BUILD_ID, BUILD_ID_EXPECTED))
print('PASS: qualified FPGA identity', CURRENT_BUILD_ID)


## 12 — Load acquisition/filter helpers

**What this cell does.** It defines supported complex-baseband capture, ADC scaling, PSD
normalisation, CIC-response prediction, optional measured transfer correction, and record-health
checks. Merely running the cell does not acquire data.

**Why it is needed.** Mixing 40.9 MHz to baseband makes narrow measurements practical, but the CIC
decimator attenuates frequencies away from the LO. Below R=256 the 309-tap FIR cannot keep up and
is automatically bypassed. The notebook must therefore preserve both raw and corrected spectra
and must never treat the full FFT interval as flat.

**Hardware and wiring.** Definitions only. `read_acquisition_fir()` reads coefficient registers; `acquire_passive_records()` captures data. When called, IN1 must
remain connected directly in its native high-impedance mode; no input pad or terminator is fitted. OUT1 coefficients and both
generators are forced to zero for passive captures.

**Expected output.** Function definitions only. Later cells plot raw PSD, estimated/measured
acquisition gain, corrected PSD and rejected regions. Capture health reports peak voltage,
headroom, crest factor, clipping and record-to-record stationarity.

**Pass/failure/rerun.** Parseval, finite-data and headroom checks must pass. Unsupported
decimation, excessive response correction, zero/invalid records or nonstationarity are rejected
with the measured values and corrective action. Safe to rerun after feedback is off.


In [ ]:
def _lo_word(frequency_hz):
    return int(round(float(frequency_hz) / FS_CLK * (1 << 32))) & 0xFFFFFFFF


# These definitions are inlined in Step 6; the notebook imports no local file.
# The qualified 309-tap symmetric FIR stores 155 signed Q1.17 coefficients.
QUALIFIED_FIR_HALF_COUNTS = [0]*57 + [
    -1, 0, 1, 0, 0, 0, 0, 0, 1, 0, -1, 0, 1, 1, -1, -1, 0, 1, 1, 0,
    -1, -1, 2, 2, -1, -3, 0, 3, 1, -1, -1, -1, 1, 4, 1, -6, -3, 6,
    4, -4, -4, 1, 2, 3, 2, -5, -8, 6, 14, -4, -16, 0, 13, 3, -4, -4,
    -10, -1, 24, 11, -34, -23, 35, 30, -24, -26, 4, 2, 19, 43, -35,
    -102, 34, 158, -12, -181, -19, 135, 30, 14, 32, -295, -249, 716, 731,
    -1257, -1619, 1860, 3117, -2418, -5566, 2729, 9743, -2247, -18165,
    -1953, 44731, 71142]


def acquisition_fir_profile(half_counts=None):
    """Describe coefficients without writing hardware or trusting build ID alone."""
    values = np.asarray(QUALIFIED_FIR_HALF_COUNTS if half_counts is None else half_counts, dtype=float)
    if (values.shape != (155,) or not np.all(np.isfinite(values)) or
            np.any(values != np.rint(values)) or np.any(values < -131072) or np.any(values > 131071)):
        raise ValueError('FIR profile needs exactly 155 signed 18-bit integer coefficients.')
    counts = values.astype(np.int64)
    taps = np.concatenate((counts, counts[-2::-1])) / float(1 << 17)
    if abs(np.sum(taps)) < .01:
        raise ValueError('FIR DC gain is near zero. Check coefficient readback; no inverse correction is safe.')
    digest = hashlib.sha256(counts.astype('<i4').tobytes()).hexdigest()
    return {'half_counts': counts.tolist(), 'taps': taps, 'digest': digest,
            'dc_gain': float(np.sum(taps)), 'l1_gain_bound': float(np.sum(np.abs(taps)))}


def read_acquisition_fir():
    """Read the live I/shared FIR table; pointer writes do not change coefficients."""
    if ('0x%08X' % acq.build_id()).lower() != str(BUILD_ID_EXPECTED).lower():
        raise RuntimeError('Cannot read FIR: qualified FPGA identity does not match. Close competing board apps.')
    old_address = acq.r(0x40)
    values = []
    try:
        for address in range(155):
            acq.w(0x40, address)
            if acq.r(0x40) != address:
                raise RuntimeError('FIR address readback failed at %d; do not trust this calibration.' % address)
            acq.r(0x44)  # discard first read: coefficient RAM has registered read latency
            word = int(acq.r(0x44)) & 0x3FFFF
            values.append(word - 0x40000 if word & 0x20000 else word)
    finally:
        acq.w(0x40, old_address)
    return acquisition_fir_profile(values)


def cic_power_response(frequency_hz, plan, stages=4):
    """Exact normalized four-stage, differential-delay-one CIC power response."""
    if stages != 4:
        raise ValueError('This FPGA has four CIC stages. Changing order requires an RTL rebuild, not a parameter here.')
    rate = float(plan['sample_rate_hz'])
    ratio = 1 << int(plan['decimation_log2'])
    relative = (np.asarray(frequency_hz, dtype=float)-float(plan['centre_hz']))/rate
    # sin(pi*f*R/Fs)/(R*sin(pi*f/Fs)) = sinc(f/Fout)/sinc(f/Fs).
    return np.abs(np.sinc(relative)/np.sinc(relative/ratio))**8


def filter_response_components(frequency_hz, plan):
    """Predict CIC, FIR and combined gains, using stored or live-read coefficients."""
    from scipy.signal import freqz
    dec = int(plan['decimation_log2'])
    mode = plan.get('fir_mode', 'auto')
    if not 6 <= dec <= 12 or mode not in ('auto', 'bypass'):
        raise ValueError("Use decimation_log2=6..12 and fir_mode='auto' or 'bypass'.")
    frequency = np.asarray(frequency_hz, dtype=float)
    offset = (frequency-float(plan['centre_hz']))/float(plan['sample_rate_hz'])
    profile = acquisition_fir_profile(plan.get('fir_half_counts'))
    _, response = freqz(profile['taps'], worN=2*np.pi*offset)
    cic = cic_power_response(frequency, plan)
    bypass = mode == 'bypass' or dec < FIR_MIN_DECIMATION_LOG2
    fir = np.ones_like(cic) if bypass else np.abs(response)**2
    return {'cic': cic, 'fir': fir, 'combined': cic*fir,
            'fir_bypassed': bypass, 'fir_digest': profile['digest']}


def acquisition_response_power(frequency_hz, plan, calibration=None):
    """Return total power gain; never assume an enabled FIR is flat everywhere."""
    frequency = np.asarray(frequency_hz, dtype=float)
    model = filter_response_components(frequency, plan)
    if calibration is not None:
        checks = (calibration.get('accepted') is True,
                  calibration.get('decimation_log2') == int(plan['decimation_log2']),
                  calibration.get('fir_bypassed') == model['fir_bypassed'],
                  calibration.get('fir_digest') == model['fir_digest'],
                  calibration.get('clock_hz') == float(plan['sample_rate_hz'])*(1 << int(plan['decimation_log2'])))
        if not all(checks):
            raise ValueError('Response calibration is rejected or belongs to different decimation, FIR coefficients/state '
                             'or clock. Select the matching measured response or use the explicitly labelled model.')
        measured_f = np.asarray(calibration['offset_hz'], dtype=float)
        measured_p = np.asarray(calibration['power_gain'], dtype=float)
        if (measured_f.ndim != 1 or measured_f.shape != measured_p.shape or measured_f.size < 3 or
                not np.all(np.isfinite(measured_f+measured_p)) or np.any(np.diff(measured_f) <= 0) or
                np.any(measured_p <= 0)):
            raise ValueError('Measured response must have sorted unique finite offsets and positive power gains.')
        # Calibration moves with LO because a fixed RF tone isolates the digital
        # baseband response. Absolute ADC/RF gain remains a separate calibration.
        gain = np.interp(frequency-float(plan['centre_hz']), measured_f, measured_p, left=np.nan, right=np.nan)
        return gain, 'measured fixed-RF-tone / swept-LO digital response'
    return model['combined'], ('modelled exact CIC; FIR bypassed' if model['fir_bypassed'] else
                               'modelled CIC x actual FIR coefficients; not a physical calibration')


def correct_acquisition_response(spectrum, plan, calibration=None,
                                 maximum_correction_db=MAX_RESPONSE_CORRECTION_DB):
    """Undo measured/modelled gain only where inversion is bounded; never undo aliasing."""
    frequency = np.asarray(spectrum['frequency_hz'], dtype=float)
    raw = np.asarray(spectrum['psd_v2_per_hz'], dtype=float)
    if raw.shape != frequency.shape or not np.isfinite(maximum_correction_db) or maximum_correction_db < 0:
        raise ValueError('Response correction requires matching PSD/frequency arrays and a finite nonnegative correction limit.')
    gain, source = acquisition_response_power(frequency, plan, calibration)
    valid = np.isfinite(gain) & (gain >= 10**(-float(maximum_correction_db)/10))
    corrected = np.full(raw.shape, np.nan)
    corrected[valid] = raw[valid]/gain[valid]
    out = dict(spectrum)
    out.update(raw_psd_v2_per_hz=raw, psd_v2_per_hz=corrected, acquisition_power_gain=gain,
               response_correction_valid=valid, response_source=source,
               maximum_response_correction_db=float(maximum_correction_db))
    return out


def acquisition_plan(f0_hz, q_estimate, expected_shift_hz=0.0, centre_on_shift=False):
    plan = plan_passive_acquisition(
        f0_hz, q_estimate, expected_shift_hz, clock_hz=FS_CLK,
        min_decimation_log2=MIN_ACQUISITION_DECIMATION_LOG2,
        max_decimation_log2=MAX_ACQUISITION_DECIMATION_LOG2,
        bins_per_linewidth=BINS_PER_LINEWIDTH,
        linewidths_each_side=LINEWIDTHS_EACH_SIDE,
        max_samples=MAX_CAPTURE_SAMPLES,
        centre_on_expected_shift=centre_on_shift)
    print('Adaptive plan: span %.1f kHz, bin %.3f Hz, %.1f bins/linewidth, %.2f s/record' %
          (plan['full_span_hz'] / 1e3, plan['bin_hz'],
           plan['bins_per_linewidth'], plan['record_duration_s']))
    return plan

def acquire_passive_records(plan=None, averages=PASSIVE_AVERAGES):
    """Acquire natural resonator noise; never enable a generated tone."""
    plan = INITIAL_ACQUISITION_PLAN if plan is None else plan
    feedback_off()
    records = []
    try:
        for number in range(int(averages)):
            raw = capture(_lo_word(plan['centre_hz']), plan['decimation_log2'],
                          plan['samples_per_record'], settle=SETTLE)
            records.append(raw * float(CIC_VOLTS_PER_COUNT))
            if number in (0, int(averages) - 1):
                print('captured passive record %d/%d' % (number + 1, averages))
    except BaseException:
        feedback_off()
        raise
    return np.asarray(records)

def make_passive_spectrum(records, plan=None):
    plan = INITIAL_ACQUISITION_PLAN if plan is None else plan
    return passive_iq_spectrum(records, plan['sample_rate_hz'], plan['centre_hz'])

def record_health(records, stationarity_cv_limit=0.20):
    """Assess finite values, converter margin, crest factor and record stationarity."""
    values = np.asarray(records)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 16:
        raise CalibrationError(
            'Passive record health requires a 2D array with at least two records '
            'and 16 samples per record; received shape %r.' % (values.shape,))
    if not np.all(np.isfinite(values)):
        bad = int(np.size(values) - np.count_nonzero(np.isfinite(values)))
        raise CalibrationError(
            'Passive records contain %d non-finite samples. Discard the capture; '
            'check DMA completion, scaling and buffer bounds before retrying.' % bad)
    magnitude = np.abs(values)
    peak = float(np.max(magnitude))
    rms_each = np.sqrt(np.mean(magnitude ** 2, axis=1))
    mean_rms = float(np.mean(rms_each))
    if mean_rms <= np.finfo(float).tiny:
        raise CalibrationError(
            'Every passive record is numerically zero. Check IN1 cabling, ADC range, '
            'DMA source and voltage scaling; a zero capture cannot establish safety.')
    cv = float(np.std(rms_each, ddof=1) / mean_rms)
    full_scale_vpk = float(RP_LIMITS['input_lv_full_scale_vpk'])
    operating_guard_vpk = float(ADC_OPERATING_GUARD_VPK)
    clipped = bool(peak >= 0.98 * full_scale_vpk)
    within_guard = bool(peak < operating_guard_vpk)
    stationary = bool(cv <= float(stationarity_cv_limit))
    result = {
        'peak_v': peak,
        'headroom_to_operating_guard_v': operating_guard_vpk - peak,
        'headroom_to_full_scale_v': full_scale_vpk - peak,
        'rms_v': mean_rms,
        'crest_factor': peak / mean_rms,
        'rms_cv': cv,
        'stationarity_cv_limit': float(stationarity_cv_limit),
        'clipped': clipped,
        'within_operating_guard': within_guard,
        'stationary': stationary,
    }
    reasons = []
    if clipped:
        reasons.append('peak %.6g V reached 98%% of the %.6g Vpk LV full scale' %
                       (peak, full_scale_vpk))
    elif not within_guard:
        reasons.append('peak %.6g V exceeded the conservative %.6g Vpk operating guard' %
                       (peak, operating_guard_vpk))
    if not stationary:
        reasons.append('record RMS coefficient of variation %.4f exceeded %.4f' %
                       (cv, stationarity_cv_limit))
    result['rejection_reasons'] = reasons
    result['accepted'] = not reasons
    return result


# Acquisition-filter workflow — code cells 13–18

Follow **predict → check FFT → measure filters → choose correction → measure the resonator**.
The test calls are commented out: rerunning definitions does not start measurements.

The measurement path is IN1 → mixer → four-stage CIC → optional FIR → FFT.
Software divides averaged spectral power by the selected CIC × FIR power response
before fitting Q and frequency. This does not change the separate OUT1 feedback path.
A model is not a measured calibration. Neither can recover filter nulls, aliases,
clipping or lost signal-to-noise.


## 13 — Load filter-test functions

Run this definition after acquisition helpers (cell 12). It creates the model, FFT check and live
calibration functions. It does not touch hardware; no wiring change or plots are
expected. It is safe to rerun. Missing names mean the imports or acquisition-helper
definitions need to run first.


In [ ]:
# Filter-test definitions: no hardware runs until a measurement call is made.
def inspect_acquisition_filters(decimation_log2=8, samples_per_record=8192,
                                centre_hz=40.9e6, half_counts=None, passband_fraction=.20):
    """Plot predicted CIC/FIR/product and alias folding; no hardware access."""
    dec = int(decimation_log2)
    if dec != decimation_log2 or not 6 <= dec <= 12 or not 0 < passband_fraction < .5:
        raise ValueError('Choose integer decimation_log2 6..12 and a passband half-width between 0 and 0.5 of output rate.')
    if int(samples_per_record) != samples_per_record or samples_per_record < 256:
        raise ValueError('FFT comparison requires an integer sample count of at least 256.')
    profile = acquisition_fir_profile(half_counts)
    rate = FS_CLK/(1 << dec)
    plan = dict(centre_hz=centre_hz, sample_rate_hz=rate, decimation_log2=dec,
                fir_half_counts=profile['half_counts'])
    # Include the requested edges exactly: sparse plotting grids must not hide
    # ripple at the steep start of the transition band.
    x = np.unique(np.r_[np.linspace(-.499, .499, 2001), -passband_fraction, passband_fraction])
    response = filter_response_components(centre_hz+x*rate, plan)
    fig, axes = plt.subplots(3, 1, figsize=(11, 10))
    for name in ('cic', 'fir', 'combined'):
        axes[0].plot(x*rate/1e3, 10*np.log10(np.maximum(response[name], 1e-20)), label=name)
    axes[0].set_ylim(-110, 8)
    axes[0].set_title('CIC loss, FIR boost/roll-off, and hardware product (no software correction)')
    axes[0].set_ylabel('Power gain (dB)'); axes[0].legend()
    good = abs(x) <= passband_fraction
    axes[1].plot(x[good]*rate/1e3, 10*np.log10(response['combined'][good]))
    axes[1].set_title('Requested passband detail: not the whole Nyquist band')
    axes[1].set_ylabel('Combined gain (dB)')
    # First adjacent decimation images fold onto the same final frequencies.
    # The post-decimation FIR treats wanted and aliased tones identically.
    alias_plus = cic_power_response(centre_hz+(x+1)*rate, plan)/response['cic']
    alias_minus = cic_power_response(centre_hz+(x-1)*rate, plan)/response['cic']
    axes[2].plot(x*rate/1e3, 10*np.log10(np.maximum(np.maximum(alias_plus, alias_minus), 1e-20)))
    axes[2].set_title('Worst first-image leakage relative to wanted tone: FIR cannot undo folded aliases')
    axes[2].set_ylabel('Relative alias gain (dB)')
    for ax in axes:
        ax.set_xlabel('Offset from LO (kHz)'); ax.grid(True, alpha=.25)
    fig.tight_layout(); plt.show()
    window = np.hanning(int(samples_per_record))
    report = dict(plan, fir_digest=profile['digest'], fir_bypassed=response['fir_bypassed'],
                  bin_hz=rate/samples_per_record,
                  enbw_hz=float(rate*np.sum(window**2)/np.sum(window)**2),
                  record_seconds=samples_per_record/rate,
                  passband_full_span_hz=2*passband_fraction*rate,
                  passband_ripple_db=float(np.ptp(10*np.log10(response['combined'][good]))),
                  group_delay_seconds=4*((1 << dec)-1)/(2*FS_CLK) +
                  (0 if response['fir_bypassed'] else 154/rate),
                  fir_l1_gain_bound=profile['l1_gain_bound'])
    print('R=%d; output %.3f ksample/s; FIR %s; bins %.3f Hz; ENBW %.3f Hz.' %
          (1 << dec, rate/1e3, 'BYPASSED' if response['fir_bypassed'] else 'enabled', report['bin_hz'], report['enbw_hz']))
    print('Requested flat span %.3f kHz; model ripple %.4f dB; nominal filter group delay %.3f ms.' %
          (report['passband_full_span_hz']/1e3, report['passband_ripple_db'], report['group_delay_seconds']*1e3))
    print('This is a model, not evidence that the board/filter or analogue chain has passed calibration.')
    return report


def validate_fft_parseval(records, spectrum=None, maximum_fractional_error=1e-10):
    """Exact Hann-weighted Parseval check; unweighted noise power is NOT identical."""
    data = np.asarray(records, dtype=complex)
    if data.ndim != 2 or min(data.shape) < 2 or not np.all(np.isfinite(data)):
        raise ValueError('FFT check requires finite records shaped [repeat, samples].')
    centred = data-np.mean(data, axis=1, keepdims=True)
    window = np.hanning(data.shape[1]); sumw2 = np.sum(window**2)
    time_power = float(np.mean(np.sum(abs(centred*window)**2, axis=1)/sumw2))
    if spectrum is None:
        # Fout cancels after multiplication by delta_f = Fout/N.
        frequency_power = float(np.mean(np.sum(abs(np.fft.fft(centred*window, axis=1))**2,
                                              axis=1)/(data.shape[1]*sumw2)))
    else:
        # Supply the RAW uncorrected complex-envelope PSD, never an inversely
        # equalized/truncated spectrum; otherwise Parseval compares different signals.
        frequency_power = float(np.sum(spectrum['psd_v2_per_hz'])*spectrum['bin_hz'])
    error = abs(frequency_power/max(time_power, np.finfo(float).tiny)-1)
    return {'accepted': bool(time_power > 0 and error <= maximum_fractional_error),
            'hann_weighted_time_power_v2': time_power, 'integrated_psd_power_v2': frequency_power,
            'fractional_error': error, 'convention': 'complex-envelope mean square, not watts into a load'}


def calibrate_adc_scale(reference_vrms, measured_count_rms):
    """Evaluate an independent raw-ADC volts/count check; never update scaling silently."""
    reference = np.asarray(reference_vrms, dtype=float)
    counts = np.asarray(measured_count_rms, dtype=float)
    if (reference.ndim != 1 or reference.shape != counts.shape or reference.size < 3 or
            not np.all(np.isfinite(reference+counts)) or np.any(reference <= 0) or np.any(counts <= 0)):
        raise ValueError('Supply at least three positive finite pairs: independently measured IN1 RMS volts and raw ADC RMS counts.')
    scale = reference/counts
    scatter = float(np.std(scale, ddof=1)/np.mean(scale))
    return {'accepted': scatter <= .02, 'volts_per_count': float(np.mean(scale)),
            'uncertainty_volts_per_count': float(np.std(scale, ddof=1)), 'relative_scatter': scatter,
            'applied_to_notebook': False}


def analyse_filter_tone_records(records, sample_rate_hz, expected_offset_hz):
    """Measure a known complex tone without confusing FFT bin width with amplitude."""
    from scipy.optimize import minimize_scalar
    data = np.asarray(records, dtype=complex)
    if data.ndim != 2 or data.shape[0] < 2 or data.shape[1] < 256 or not np.all(np.isfinite(data)):
        raise ValueError('Tone analysis needs at least two finite records of at least 256 complex samples.')
    rate = float(sample_rate_hz); n = data.shape[1]
    if not np.isfinite(rate) or rate <= 0 or not np.isfinite(expected_offset_hz):
        raise ValueError('Tone fitting needs a positive finite sample rate and a finite expected frequency offset.')
    if np.mean(abs(data)**2) <= np.finfo(float).tiny:
        raise ValueError('Tone records are all zero. Check the selected source, loopback cable and acquisition; no response can be calibrated.')
    t = np.arange(n)/rate; window = np.hanning(n)
    folded = (expected_offset_hz+rate/2) % rate-rate/2
    amplitudes, offsets, residuals = [], [], []
    for row in data:
        # Search a narrow frequency interval to allow source/ADC clock error.
        # This avoids assuming the generator lands exactly on an FFT bin.
        def objective(offset):
            return -abs(np.sum(row*window*np.exp(-2j*np.pi*(folded+offset)*t))/np.sum(window))**2
        fitted = minimize_scalar(objective, bounds=(-rate/n, rate/n), method='bounded',
                                 options={'xatol': rate/n*1e-6})
        f = folded+fitted.x
        a = np.sum(row*window*np.exp(-2j*np.pi*f*t))/np.sum(window)
        residual = row-a*np.exp(2j*np.pi*f*t)
        amplitudes.append(abs(a)); offsets.append(f)
        residuals.append(float(np.mean(abs(residual)**2)))
    power = np.asarray(amplitudes)**2
    snr = float(10*np.log10(np.mean(power)/max(np.mean(residuals), np.finfo(float).tiny)))
    return {'amplitude_v': float(np.sqrt(np.mean(power))),
            'power_v2': float(np.mean(power)), 'power_relative_scatter': float(np.std(power, ddof=1)/np.mean(power)),
            'observed_offset_hz': float(np.mean(offsets)), 'snr_db': snr,
            'parseval': validate_fft_parseval(data)}


def measure_acquisition_filters(tone_hz=40.9e6, decimation_log2=8,
                                fractions=None, repeats=3, samples_per_record=4096,
                                settle_s=.03, maximum_ripple_db=.15,
                                maximum_model_error_db=.30, seed=7,
                                source='out1', drive_fraction=.02):
    """Fixed RF tone, swept LO, FIR bypass/on pairs; feedback always off.

    DISCONNECT THE TRAP from IN1 and OUT1 first. For source='out1', connect only
    OUT1 -> short coax (optional pad) -> IN1, preferably locally terminated 50 ohm.
    The generator is distinct from feedback, whose coefficients remain zero.
    source='external' retains the independently supplied fixed-tone option.
    Returns compact measurements and optional accepted response dictionaries.
    OUT1 generator A is enabled only for loopback and shut down on every exit.
    No FIR coefficients, feedback coefficients or OUT2 are enabled/rewritten.
    """
    global LAST_FILTER_CALIBRATION
    result = {'accepted': False, 'rows': [], 'responses': {}, 'rejection_reasons': [],
              'timestamp_epoch_s': time.time(), 'shutdown_verified': False,
              'source': source, 'absolute_voltage_calibrated': False,
              'independent_clock_test': False}
    LAST_FILTER_CALIBRATION = result
    try:
        full_shutdown()
        if source not in ('out1', 'external'):
            raise ValueError("source must be 'out1' for the isolated loopback or 'external' for a separate fixed RF source.")
        if source == 'out1' and (not np.isfinite(drive_fraction) or
                not 0 < drive_fraction <= min(.05, DRIVE_MAX) or
                int(round(drive_fraction*8192)) < 2):
            raise ValueError('Loopback drive_fraction must produce at least two generator counts and be <= min(0.05, DRIVE_MAX). '
                             'Start at 0.02; this is a digital fraction, not an independently calibrated voltage.')
        if ADC_RANGE != 'LV':
            raise ValueError('This test uses the notebook LV input scaling. Match the physical range before measuring.')
        dec = int(decimation_log2); n = int(samples_per_record)
        fractions = np.linspace(-.20, .20, 17) if fractions is None else np.asarray(fractions, dtype=float)
        if (dec != decimation_log2 or not 6 <= dec <= 12 or n != samples_per_record or
                n < 1024 or n > 32768 or n & (n-1) or repeats < 2 or repeats != int(repeats) or repeats > 10 or
                fractions.ndim != 1 or not 9 <= fractions.size <= 65 or not np.all(np.isfinite(fractions)) or
                np.any(abs(fractions) > .45) or len(np.unique(fractions)) != len(fractions) or
                not np.any(fractions == 0) or min(fractions) >= 0 or max(fractions) <= 0):
            raise ValueError('Use decimation 6..12, power-of-two samples 1024..32768, repeats 2..10, '
                             'and 9..65 unique offset fractions within +/-0.45 including zero and both signs.')
        if not np.isfinite(tone_hz) or not 0 < tone_hz < 50e6 or not np.isfinite(settle_s) or settle_s < .01:
            raise ValueError('Use a finite RF tone within 0..50 MHz and settle_s >= 0.01.')
        if not (0 < maximum_ripple_db <= 1 and 0 < maximum_model_error_db <= 1):
            raise ValueError('Ripple/model tolerances must be positive and <=1 dB; investigate larger errors.')
        profile = read_acquisition_fir()
        held = ps.readback()
        if held['ci'] or held['cq'] or held['monitor']:
            raise RuntimeError('Feedback or OUT2 is not zero after shutdown. Disconnect OUT1 and resolve register readback before any tone test.')
        # These local checks are shared by startup, the scan and shutdown.
        # A changing generator must not be mistaken for a changing filter.
        expected_generator = {0x04: 0}
        expected_control = [0xC0]

        def check_source():
            if ('0x%08X' % acq.build_id()).lower() != str(BUILD_ID_EXPECTED).lower():
                raise RuntimeError('FPGA changed during calibration; close other Red Pitaya applications before retrying.')
            ps.assert_held(held)
            gen.assert_b_zero()
            if gen.r(0x00) & 0x1FF != expected_control[0]:
                raise RuntimeError('Generator A control changed during the filter test. Stop competing apps; no correction is accepted.')
            for address, expected in expected_generator.items():
                actual = gen.r(address)
                if actual != expected:
                    raise RuntimeError('Generator readback mismatch at 0x%02X: expected 0x%08X, got 0x%08X. '
                                       'Amplitude, DC offset or tuning changed; disconnect loopback and investigate.' %
                                       (address, expected, actual))

        def raw_input():
            check_source()
            codes, adc_rate = sc.acquire(dec=1, n=4096, chan='A')
            codes = np.asarray(codes, dtype=float)
            if (codes.shape != (4096,) or sc.r(0x04) or not np.all(np.isfinite(codes)) or
                    not np.isfinite(adc_rate) or abs(adc_rate/FS_CLK-1) > 1e-9):
                raise RuntimeError('Raw ADC capture/clock invalid or timed out; no response calibration accepted.')
            voltage = codes*ADC_RAW_VOLTS_PER_COUNT
            if np.max(abs(voltage)) >= ADC_OPERATING_GUARD_VPK or abs(np.mean(voltage)) > IN1_DC_OFFSET_MAX_V:
                raise RuntimeError('ADC peak/DC operating guard exceeded. Stop the source, check cabling/offsets and reduce level before retrying.')
            check_source()
            return voltage

        if source == 'out1':
            # Bound even an unterminated output conservatively; never rely on a
            # matched-pad loss to make an excessive command safe.
            count = int(round(drive_fraction*8192))
            bound = 2*(count/8192.0)*RP_LIMITS['output_full_scale_vpk_into_50ohm']
            if bound >= min(ADC_OPERATING_GUARD_VPK, OUT1_OPERATING_GUARD_VPK):
                raise ValueError('Loopback nominal high-impedance output bound exceeds an operating guard; reduce drive_fraction.')
            off = raw_input()
            off_power = float(np.var(off))
            gen.load_sine()
            # The existing ASG has asymmetric low-frequency write/read addresses:
            # write fractional step at 0x4C, read it at 0x14. Clear stale burst
            # state as well; generator apps may have left these non-default.
            for address in (0x4C, 0x18, 0x1C, 0x20, 0x44):
                gen.w(address, 0)
            expected_generator.update({0x14: 0, 0x18: 0, 0x1C: 0, 0x20: 0, 0x44: 0})
            # Verify all waveform words while A is held off, not just control.
            for address in range(16384):
                expected = int(round(8191*np.sin(2*np.pi*address/16384.0))) & 0x3FFF
                if gen.r(0x10000+4*address) & 0x3FFF != expected:
                    raise RuntimeError('Generator sine table readback failed at sample %d. No tone was enabled.' % address)
            tone_hz = gen.freq(float(tone_hz))  # use actual quantized DDS frequency
            expected_generator.update({0x08: (16384 << 16)-1, 0x0C: 0,
                                       0x10: gen.freq_word(tone_hz)})
            check_source()
            result['loopback_startup'] = {'off_power_v2': off_power, 'levels': [],
                                          'nominal_high_z_bound_vpk': bound}
            print('LOOPBACK ONLY: ramping generator A at %.6f MHz; feedback and OUT2 remain zero.' % (tone_hz/1e6))
            for rung in (count//2, count):  # count >= 2: distinct, ascending levels
                gen.amp(rung/8192.0)
                expected_generator[0x04] = rung  # upper DC-offset field must stay zero
                check_source()
                gen.run()
                expected_control[0] = 0x11
                time.sleep(settle_s)
                voltage = raw_input()
                power = float(np.var(voltage))
                if power < (4*ADC_RAW_VOLTS_PER_COUNT)**2 or power < 10*max(off_power, 1e-30):
                    raise RuntimeError('Loopback tone is too weak or does not rise at least 10x above generator-off ADC power. '
                                       'Check OUT1-to-IN1 cabling, termination and excessive pads. No automatic amplitude increase is made.')
                # A fixed frequency permits an independent raw-ADC sine/DC fit.
                # Reject strong other tones or generator distortion before using RMS.
                t = np.arange(voltage.size)/FS_CLK
                design = np.column_stack((np.cos(2*np.pi*tone_hz*t),
                                          np.sin(2*np.pi*tone_hz*t), np.ones(t.size)))
                coefficients = np.linalg.lstsq(design, voltage, rcond=-1)[0]
                residual_power = float(np.mean((voltage-design.dot(coefficients))**2))
                if residual_power/power > .03:
                    raise RuntimeError('Loopback raw input is not a clean sine: >3% residual power. '
                                       'Check distortion, noise, competing signals and range; do not calibrate from this trace.')
                result['loopback_startup']['levels'].append(dict(
                    generator_counts=rung, rms_v=float(np.sqrt(power)), peak_v=float(np.max(abs(voltage))),
                    dc_v=float(np.mean(voltage)), excess_power_v2=power-off_power,
                    residual_fraction=residual_power/power))
                print('  generator count %d: input RMS %.3f mV, peak %.3f mV, DC %.3f mV' %
                      (rung, 1e3*np.sqrt(power), 1e3*np.max(abs(voltage)), 1e3*np.mean(voltage)))
            levels = result['loopback_startup']['levels']
            linearity = ((levels[1]['excess_power_v2']/levels[0]['excess_power_v2']) /
                         (float(levels[1]['generator_counts'])/levels[0]['generator_counts'])**2)
            result['loopback_startup']['power_linearity_ratio'] = linearity
            if not .90 <= linearity <= 1.10:
                raise RuntimeError('Loopback two-level power linearity ratio %.3f is outside 0.90..1.10. '
                                   'Check DAC/ADC compression, source state or noise before accepting a response.' % linearity)
        result['tone_hz'] = float(tone_hz)
        rate = FS_CLK/(1 << dec)
        if min(tone_hz-fractions*rate) <= 0 or max(tone_hz-fractions*rate) >= 50e6:
            raise ValueError('Swept LO leaves the 0..50 MHz supported RF range; reduce offsets.')
        plan = dict(centre_hz=float(tone_hz), sample_rate_hz=rate, decimation_log2=dec,
                    fir_half_counts=profile['half_counts'])
        modes = ('bypass', 'auto') if dec >= 8 else ('bypass',)
        rng = np.random.RandomState(seed)
        # Fixed RF avoids importing analogue gain-versus-frequency ripple into
        # the digital response, even when source and ADC share the board clock.
        for fraction in rng.permutation(fractions):
            lo_word = _lo_word(tone_hz-fraction*rate)
            lo_actual = lo_word*FS_CLK/float(1 << 32)
            offset = tone_hz-lo_actual
            for mode in rng.permutation(modes):
                records, input_powers = [], []
                for repeat in range(int(repeats)):
                    voltage = raw_input()
                    if np.std(voltage) < 4*ADC_RAW_VOLTS_PER_COUNT:
                        raise RuntimeError('Raw input RMS is below four ADC counts: insufficient level for a reliable tone calibration. '
                                           'Check source/cabling and independently verify a small input level before retrying.')
                    input_powers.append(float(np.var(voltage)))
                    raw = capture(lo_word, dec, n, settle=settle_s, fir_mode=str(mode))
                    raw = np.asarray(raw)
                    if raw.shape != (n,) or not np.all(np.isfinite(raw)) or max(np.max(abs(raw.real)), np.max(abs(raw.imag))) >= .98*(1 << 31):
                        raise RuntimeError('DMA shape/nonfinite/32-bit clipping failure; reject this filter measurement.')
                    records.append(raw*CIC_VOLTS_PER_COUNT)
                    check_source()
                stats = analyse_filter_tone_records(records, rate, offset)
                row = dict(stats, fraction=float(fraction), offset_hz=float(offset), fir_mode=str(mode),
                           input_power_v2=float(np.mean(input_powers)),
                           input_power_scatter=float(np.std(input_powers, ddof=1)/max(np.mean(input_powers), 1e-30)))
                # The 2^29 scale makes |I+jQ| the RF sine PEAK amplitude.
                # Therefore half its mean square is the equivalent real RF power.
                row['absolute_gain_check'] = row['power_v2']/max(2*row['input_power_v2'], 1e-30)
                result['rows'].append(row)
                print('offset %+8.2f kHz, FIR %-6s: SNR %.1f dB, repeat scatter %.2f%%' %
                      (offset/1e3, mode, stats['snr_db'], 100*stats['power_relative_scatter']))
        if read_acquisition_fir()['digest'] != profile['digest']:
            raise RuntimeError('FIR coefficients changed during the scan. Discard results; no correction was accepted.')
        full_shutdown()  # Analysis/plotting occurs with the writer and outputs off.
        zero = next(r for r in result['rows'] if r['fraction'] == 0 and r['fir_mode'] == 'bypass')
        reference = zero['absolute_gain_check']
        if not .9 <= reference <= 1.1:
            result['rejection_reasons'].append('CIC-to-raw-ADC sine scaling differs by >10%% (power ratio %.4g).' % reference)
        fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
        for mode in modes:
            rows = sorted([r for r in result['rows'] if r['fir_mode'] == mode], key=lambda r: r['offset_hz'])
            x = np.array([r['offset_hz'] for r in rows])
            measured = np.array([r['absolute_gain_check']/reference for r in rows])
            mode_plan = dict(plan, fir_mode=mode)
            predicted = filter_response_components(tone_hz+x, mode_plan)['combined']
            residual = 10*np.log10(measured/predicted)
            ripple = float(np.ptp(10*np.log10(measured)))
            max_error = float(np.max(abs(residual)))
            reasons = []
            if any(r['snr_db'] < 20 or r['power_relative_scatter'] > .03 or
                   r['input_power_scatter'] > .03 or not r['parseval']['accepted'] for r in rows):
                reasons.append('%s: inadequate tone SNR (<20 dB), >3%% repeat/input scatter, or FFT failure' % mode)
            if max_error > maximum_model_error_db:
                reasons.append('%s: measured/model error %.3f dB exceeds %.3f dB' % (mode, max_error, maximum_model_error_db))
            if mode == 'auto' and ripple > maximum_ripple_db:
                reasons.append('FIR hardware product ripple %.3f dB exceeds %.3f dB: narrow passband, inspect coefficients, or use qualified software correction' %
                               (ripple, maximum_ripple_db))
            response = {'accepted': not reasons and not result['rejection_reasons'],
                        'offset_hz': x, 'power_gain': measured, 'decimation_log2': dec,
                        'fir_bypassed': mode == 'bypass' or dec < 8, 'fir_digest': profile['digest'],
                        'clock_hz': float(FS_CLK), 'timestamp_epoch_s': time.time(),
                        'passband_ripple_db': ripple, 'maximum_model_error_db': max_error,
                        'rejection_reasons': reasons, 'relative_digital_response_only': True,
                        'source': source, 'tone_hz': float(tone_hz),
                        'absolute_voltage_calibrated': False, 'independent_clock_test': False}
            # Conservative propagation; common-reference covariance is not removed.
            # Independent source/ADC calibration errors are not measured by this scan.
            response['power_gain_uncertainty'] = measured*np.sqrt(np.array([
                r['power_relative_scatter']**2+r['input_power_scatter']**2+
                zero['power_relative_scatter']**2+zero['input_power_scatter']**2
                for r in rows])/float(repeats))
            response['uncertainty_description'] = 'repeat-based standard uncertainty; excludes absolute source/ADC calibration'
            result['responses'][mode] = response
            axes[0].plot(x/1e3, 10*np.log10(measured), 'o-', label=mode+' measured')
            axes[0].plot(x/1e3, 10*np.log10(predicted), '--', label=mode+' model')
            axes[1].plot(x/1e3, residual, 'o-', label=mode)
            print('%s: ripple %.4f dB; maximum model residual %.4f dB; accepted=%s' %
                  (mode, ripple, max_error, response['accepted']))
        axes[0].set_ylabel('Gain relative to bypass centre (dB)'); axes[0].legend()
        axes[1].set_ylabel('Measured / predicted (dB)'); axes[1].set_xlabel('Offset from LO (kHz)')
        for ax in axes: ax.grid(True, alpha=.25)
        fig.tight_layout(); plt.show()
        result['accepted'] = all(r['accepted'] for r in result['responses'].values())
        for r in result['responses'].values():
            for reason in r['rejection_reasons']: print('REJECTED: '+reason)
        for reason in result['rejection_reasons']: print('REJECTED: '+reason)
        return result
    except BaseException as exc:
        result['accepted'] = False
        for response in result['responses'].values(): response['accepted'] = False
        result['error'] = '%s: %s' % (type(exc).__name__, exc)
        raise
    finally:
        try:
            full_shutdown()
            if gen.r(0x04) or gen.r(0x00) & 0xFF != 0xC0:
                raise RuntimeError('Generator A amplitude/DC or control is nonzero after shutdown.')
            gen.assert_b_zero()
            result['shutdown_verified'] = True
        except BaseException:
            result['accepted'] = False
            for response in result['responses'].values(): response['accepted'] = False
            raise RuntimeError('Full shutdown could not be verified. Keep OUT1/trap disconnected, turn off the external '
                               'generator manually, and investigate the register connection before retrying.')


## 14 — Predict the usable filter window

Uncomment the model calls below; no board is needed and they are safe to repeat.
Compare exponent 6 (broad baseline) with exponent 8 (narrow feedback).
Plots separate CIC sag, FIR compensation, their product and first-image alias leakage.

At exponent 6 or 7 the FIR is automatically bypassed; at 8–12 it is enabled.
The qualified FIR's designed flat full span is approximately
`0.4 × output sample rate`, about 195 kHz at exponent 8, not the full FFT span.
Choose a window away from the steep cutoff.
Changing `passband_fraction` changes the inspected window, not the FIR taps.
Printed ripple is a prediction, not a hardware pass certificate.


In [ ]:
# Model only: no board access. These two settings match the manual examples.
# broad_filter_model = inspect_acquisition_filters(decimation_log2=6, passband_fraction=.30)
# narrow_filter_model = inspect_acquisition_filters(decimation_log2=8, passband_fraction=.20)


## 15 — Check FFT normalization

The synthetic check below needs no wiring. It compares integrated raw PSD with
the same demeaned, Hann-weighted time-record power.
Pass means their fractional difference is below numerical tolerance.
It is safe to rerun; failure is a software/scaling issue, not a trap issue.

Integrate PSD using FFT bin width. Hann equivalent noise bandwidth (ENBW)
describes the resolution filter; do not multiply every PSD bin by ENBW.
Do not submit a corrected or cropped spectrum to this check.

This does not calibrate ADC volts/count. The notebook uses a complex-envelope
convention: a sinusoid's envelope mean-square is twice its real RF mean-square.
The live test checks that factor separately. dB relative to 1 V²/Hz is not dBm.


In [ ]:
# Synthetic FFT check only: no hardware or live calibration result is changed.
# fft_rng = np.random.RandomState(7)
# fft_test_records = (fft_rng.normal(size=(4, 4096)) +
#                     1j*fft_rng.normal(size=(4, 4096)))
# fft_check = validate_fft_parseval(fft_test_records)
# print(fft_check)


## Before cell 16 — Connect the Red Pitaya loopback

**Hardware change required. Never inject the calibration tone into the trap.**

1. Run `full_shutdown()`. Disconnect the trap/electrode return cable at OUT1
   and isolate it. OUT2 stays disconnected/off. Close other Red Pitaya apps.
2. Disconnect the detection chain from IN1. Follow its power/load requirements;
   never leave an amplifier open if its operating instructions prohibit that.
3. Connect **OUT1 → short coax → IN1**, preferably with a 50 Ω feedthrough
   directly at IN1. A small rated pad is optional; the normal large feedback
   attenuation is not required for this isolated low-level electronics test.
4. No amplifier, trap, resonator, splitter or external source belongs in this loop.
   A native high-impedance IN1 connection is supported for relative testing, but
   matching is less predictable. Treat open-load voltage as potentially twice
   the nominal loaded voltage. Never connect two outputs together.
5. Leave generator control to the test cell. It starts at half of the selected
   low drive and then reaches that drive, with raw-input checks at each level.

There is no new physical feedback loop: phase-shifter coefficients stay zero;
only generator A makes the sine. The call is the command to enable OUT1, without
an extra dialog. Software cannot prove the trap was disconnected.
On a kernel hang, disconnect the loopback and power down the board before rewiring.


## 16 — Measure the broad loopback response

Uncomment the loopback scan you need. It starts and ends with Red Pitaya outputs off.
Rerunning definitions or model checks does not erase earlier scan results.
Generator A makes a fixed approximately 40 MHz tone (default 40.9 MHz, or choose
`tone_hz=40e6`). Only the acquisition LO moves the tone through the band.
Randomized offsets and paired bypass/auto captures separate CIC loss from FIR
compensation. Raw ADC checks monitor overload and input drift.

Before scanning, generator-off noise, two-level power proportionality, sine
purity and input peak/DC are checked. Actual quantized generator frequency is
used. Table, amplitude/DC, tuning and channel-B readback are checked.
`drive_fraction=.02` is 2% of digital full scale, not 20 mV measured at IN1.
The test refuses above 5% or the existing lower drive/operating limit.

The broad scan covers ±0.30 of output rate; cell 17 covers ±0.20.
Expect measured/model curves, residuals, repeat scatter and an acceptance record.
Defaults require tone SNR ≥20 dB, power scatter ≤3%, model error ≤0.30 dB and
enabled-FIR ripple ≤0.15 dB. A bypassed CIC need not be flat.
FFT and RF/envelope scaling, register state and FIR readback are checked too.

If rejected, inspect the reasons: poor SNR, compression, source drift, incorrect
coefficients or a too-wide passband can fail. Never force acceptance.
Hardware exceptions attempt shutdown and preserve partial rows in
`LAST_FILTER_CALIBRATION`. Review failures before repeating with safe test wiring.


In [ ]:
# LIVE LOOPBACK: trap disconnected; OUT1 -> coax -> IN1. No external source.
# FILTER_BASELINE_CALIBRATION = measure_acquisition_filters(
#     source='out1', tone_hz=40.9e6, drive_fraction=.02, decimation_log2=6,
#     fractions=np.linspace(-.30, .30, 13), repeats=3, samples_per_record=4096)


## 17 — Measure the narrow loopback response

Keep the same isolated OUT1-to-IN1 wiring. The function restarts its own tone. Uncomment this
call for exponent 8, where the FIR is active. Expect the same diagnostics, now
including FIR flatness. A rejected record must not be used. It is safe to repeat
after addressing the reported cause; this replaces only the narrow result.
Generator A is shut down automatically afterward; verify the shutdown message.


In [ ]:
# LIVE LOOPBACK: keep OUT1 -> IN1 wiring; feedback coefficients remain zero.
# FILTER_FEEDBACK_CALIBRATION = measure_acquisition_filters(
#     source='out1', tone_hz=40.9e6, drive_fraction=.02, decimation_log2=8,
#     fractions=np.linspace(-.20, .20, 17), repeats=3, samples_per_record=4096)


## 18 — Select baseline and feedback corrections

Run `full_shutdown()`, remove the loopback cable, and restore the commissioned
passive detection and feedback wiring. If using the optional external source,
switch it off yourself. Never start trap measurements with either test source connected.

The selection cell starts with explicitly labelled model correction.
To use measured responses, uncomment its two assignments after those scans pass.
The broad baseline and narrow feedback measurement have separate corrections.

Measured records must match decimation, FIR state, clock and coefficient fingerprint;
there is no extrapolation beyond measured LO-relative offsets.
The manual function checks these before enabling feedback. For a new setting,
measure that setting or deliberately choose the labelled model.

This calibrates relative digital response, not absolute electrode voltage or the
whole analogue chain. Repeat at another RF tone for large centre-frequency changes.
After changing a correction or physical setup, reacquire the baseline once.
Subsequent feedback-window changes do not refit the saved baseline.


In [ ]:
# Explicit correction selection. No hardware access.
# None means theoretical CIC x FIR correction, NOT a passed calibration.
BASELINE_RESPONSE = None
FEEDBACK_RESPONSE = None

# To use measured corrections, uncomment the matching assignments AFTER scans pass.
# Exponent 6 uses bypass even when the manual call asks for fir_mode='auto'.
# BASELINE_RESPONSE = FILTER_BASELINE_CALIBRATION['responses']['bypass']
# FEEDBACK_RESPONSE = FILTER_FEEDBACK_CALIBRATION['responses']['auto']

for response_name, response in (('Baseline', BASELINE_RESPONSE),
                                 ('Feedback', FEEDBACK_RESPONSE)):
    if response is None:
        print('%s: theoretical CIC x FIR model selected; not a measured calibration.' % response_name)
    elif not isinstance(response, dict) or not response.get('accepted', False):
        raise CalibrationError(
            '%s correction is not an accepted measured response. Inspect the scan rejection '
            'reasons and repeat the electronics test, or explicitly select None for the '
            'labelled model. Do not change an acceptance flag to bypass this check.' % response_name)
    else:
        print('%s: measured response selected. Decimation, FIR, clock and frequency '
              'coverage will be checked again by the manual measurement.' % response_name)


## Reference — What can be configured?

| Setting | Effect and limit |
|---|---|
| LO | Moves the RF measurement window, not OUT1 phase. |
| CIC exponent 6–12 | Changes sample rate, null spacing and latency. Four stages and differential delay one stay fixed. |
| FIR auto/bypass | Auto bypasses at exponents 6–7; enabled FIR compensates only its central band. |
| FIR coefficients | Read back and checked, never rewritten by this workflow. |
| FFT samples | More samples give finer bins and longer records, not a wider passband. |
| Averages | Reduce random scatter, not systematic roll-off or drift. |
| Software correction | Measured or model response; cannot recover nulls or aliases. |

Reconfiguration stops/resets acquisition and allows filter history to settle.
None of these settings changes the physical feedback phase/gain.


## Limits — What this self-test does not prove

OUT1, the mixer and ADC share the same clock. A good relative CIC/FIR result
does not independently calibrate clock accuracy, absolute ADC/DAC voltage,
RF bandwidth or the electrode path. This is a fixed-RF, amplitude-response test,
not an independently measured phase/group-delay calibration. Source harmonics
and coherent crosstalk can bias it; raw sine purity and measured/model residuals
help detect problems but cannot rule out every correlated error.

For optional cross-checking, set `source='external'`: disconnect OUT1 and connect
a separate stable, safely attenuated source directly to IN1. Keep its RF frequency
and amplitude fixed. The notebook does not control or turn off an external source.
No external generator is required for the normal loopback workflow.


## Optional — Absolute voltage and alias checks

For ADC scaling, independently measure at least three small input RMS levels
under the actual termination, then pass voltage/count pairs to `calibrate_adc_scale`.
It reports a candidate factor and scatter; it changes neither EEPROM nor notebook
scaling. Repeated increasing/decreasing levels help identify compression.

For alias rejection, keep the disconnected electronics wiring and a fixed LO.
Compare equal independently measured source levels at the wanted frequency and at
that frequency ± one output sample rate. Inspect the folded tone and repeat relevant
image bands. A below-floor result is a bound, not proof of zero aliasing.
The model plots only first images; a post-CIC FIR cannot undo already folded aliases.

Background: [CIC compensation](https://www.mathworks.com/help/dsp/ref/dsp.ciccompensationdecimator-system-object.html)
and [PSD normalization](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.periodogram.html).

Hardware reference: [official STEMlab 125-14 specifications](https://redpitaya.readthedocs.io/en/latest/developerGuide/hardware/ORIG_GEN/125-14/top.html)
specify a ±1 V LV input range, high input impedance and 50 Ω output load.
Absolute damage limits are not operating targets; this workflow keeps the existing
lower notebook guards and makes no EEPROM, RTL or FIR-coefficient changes.


# Manual measurements — code cells 19–21


## What changes

Measure the broad feedback-off resonance **once**, retaining its spectrum and fit
in `manual_baseline`. Then rerun only the feedback call, using a different window,
decimation, FFT length and Q estimate. That call acquires **only the feedback
spectrum** and switches feedback off afterward. It never recaptures or refits the
saved baseline. No JSON, checkpoint, or campaign calibration is needed.

The plots show a broad overview, a zoom of the enhanced measurement window, and
fit residuals. Each spectrum/model keeps its native frequency grid. A narrow
window cannot change the baseline fit or pretend to improve its resolution.
Changing the acquisition LO also leaves the inherited feedback phase reference
unchanged unless you explicitly override `phase_reference_hz`.

An additional figure shows the same averaged spectra and Lorentzian models in
**dB relative to 1 V²/Hz**, with a full overview and (for feedback mode) an
enhanced-window zoom. It uses `10*log10(PSD / (1 V²/Hz))`: for example,
`1e-12 V²/Hz` is `-120 dB`. Negative dB values are normal and do not mean negative
voltage or negative power. The logarithm is applied after averaging linear powers;
fits still use the original data. Zero/non-finite bins appear as gaps.

This is not dBm: native high-impedance IN1 does not measure power delivered into
a known 50-ohm load. There is no new hardware action, recapture or refit for this
additional plot. The baseline and enhanced measurement retain their own grids.


## Preparation and hardware

Run the standard configuration, imports/library, hardware interface, identity and
shutdown cells, then **cell 12's acquisition helpers and cells 13–18's filter workflow**,
including its explicit correction selection, followed by the definition below.
No saved calibration file is needed. Physical commissioning is still necessary:
software cannot establish unknown amplifier/electrode damage limits.

Use the already commissioned detection and return paths. Both generators and
OUT2 stay off. Before changing pads, amplifiers, filters, cables or electrodes,
shut down and follow the hardware's power/connector sequence. **Acquire a new
baseline after a physical setup change**, and start at a conservative gain again.
The function cannot detect such changes automatically.

Defining the function does nothing to hardware. Each call immediately runs the
requested measurement, without another confirmation dialog. Every call attempts
full shutdown on completion, error and Ctrl-C. Keep a physical means to interrupt
the return path: a crashed kernel cannot run shutdown, and the software deadline
is checked between captures, not by an independent watchdog.


## A — Measure the broad baseline once

Run this in its **own cell** after the function definition:

```python
manual_baseline = measure_manual_feedback(
    mode='baseline',                       # No gain is applied.
    frequency_range_hz=(40.4e6, 41.4e6),
    fit_range_hz=(40.5e6, 41.3e6),
    decimation_log2=6, samples_per_record=8192,
    fir_mode='auto', response_calibration=BASELINE_RESPONSE,
    averages=32, repeats=1,
    q_guess=150.0, q_bounds=(20.0, 1e6),
    phase_reference_hz=40.9e6,
    label='current setup — broad off baseline'
)
```

**Expected result:** one averaged off spectrum, an accepted Lorentzian fit with
Q, centre and linewidth, and verified shutdown. Review residuals before accepting
the physical interpretation. Increase averaging for weak random noise; widen the
fit window if the shoulders are missing. A failed baseline cannot be reused.
Rerunning this cell deliberately replaces the baseline.


## B — Adjust and repeat only the enhanced measurement

Run this in a **different cell**. The values below are examples, not a guarantee
that count 32 is safe or that Q=2000 will result. Start at your commissioned low
gain. If the enhanced width/centre is unknown, start with a broad window, then
narrow it once located.

```python
measurement = measure_manual_feedback(
    mode='feedback', baseline_measurement=manual_baseline,
    coefficient_counts=32, phase_deg=0.0,
    frequency_range_hz=(40.81e6, 40.99e6),
    fit_range_hz=(40.82e6, 40.98e6),
    decimation_log2=8, samples_per_record=8192,
    fir_mode='auto', response_calibration=FEEDBACK_RESPONSE,
    averages=32, repeats=1,
    q_guess=2000.0, q_bounds=(20.0, 1e6),
    label='manual Q-enhanced measurement'
)
```

Each call changes only the new measurement. Each repeat averages independent FFT
**powers**, not signed waveforms that would cancel random noise. Raw time records
are discarded as they are processed. The latest/partial result is also retained
in `LAST_MANUAL_FEEDBACK_MEASUREMENT`. No files are loaded or saved; a kernel
restart loses these in-memory results.


## Choosing windows, resolution and gain

| Setting | Purpose |
|---|---|
| `frequency_range_hz` | New acquisition/display window, independent of the baseline. Move it to follow frequency shifts. |
| `fit_range_hz` | Fit only this subset to exclude other resonances. Keep at least one linewidth on each side of the centre. |
| `decimation_log2` | Sample rate is `FS_CLK / 2**decimation_log2`. Larger values narrow the available band and improve resolution. |
| `samples_per_record` | Power-of-two FFT length, 256–65,536. Bin spacing = sample rate / length; at least 20 bins per linewidth are required. |
| `averages`, `repeats` | Spectra per measurement and repetitions of that state. Averaging reduces random noise, not drift. |
| `q_guess`, `q_bounds` | Fit starting estimate and search limits, not a forced Q. |
| `coefficient_counts` **or** `gain` | Choose one. Digital `gain=count/16384` is not physical loop K. Negative gain adds 180 degrees. |
| `phase_deg` | Commanded feedback phase; electrode phase is not independently measured. |
| `lo_hz` | Acquisition LO, separate from the inherited feedback phase reference. |

The printed bin width and Hann-window equivalent noise bandwidth (ENBW) describe
resolution. PSDs are normalised in V²/Hz at the notebook ADC scale and corrected
using its CIC/FIR model, or an optional **in-memory** `response_calibration`.
Narrow peaks can have different apparent heights at different ENBWs; compare
resolved fits, not peak height alone.


Mean removal for the FFT causes a small notch at the LO. The fit excludes that
bin and its neighbours. No DC is subtracted from hardware feedback. A real
electron dip, multiple resonances or self-oscillation cannot be interpreted
reliably by this single Lorentzian plus smooth-background model.


## Checks retained and the baseline-reuse trade-off

Before each feedback repeat, a short raw-ADC off **safety capture** checks current
DC, peak and RMS. It is not a resonance spectrum or a baseline fit. During
acquisition the function checks headroom, readback, generator state, time limits,
large full-band power growth and narrow-carrier evidence.

`max_input_vpk` and `max_output_vpk` may reduce, never raise, notebook operating
guards. `max_power_ratio` compares current raw-ADC power with the fresh short off
capture, not with an old differently decimated spectrum. `max_active_seconds`
bounds each acquisition loop but is not a hardware watchdog.

**A reused baseline does not measure present drift or post-feedback recovery.**
Its age is printed. Zero coefficients prove feedback is off, not that Q/frequency
recovered. Reacquire deliberately after setup changes, suspected heating, or
unexplained drift. There is no automatic age cutoff: the operator must decide
whether the saved baseline remains representative.

`measurement['comparison']` reports Q ratio, shift and effective damping
`1 - linewidth_on / linewidth_baseline` for accepted fits. Fit uncertainties
**exclude unmeasured drift**. This is not an independently calibrated loop K or
permission to approach self-oscillation. `recovery_measured=False` records this
limitation explicitly.


**Pass:** resolved peak, adequate shoulders/resolution, acceptable uncertainty,
and no live-guard failure. `status='complete'` does not certify historical
baseline stability. Failed fits are plotted as unreliable and stop further
repeats; hardware failures raise an error and attempt shutdown.

Optional `mode='bracketed'` retains the old same-window off/on/off measurement
when a recovery check is wanted. It is not needed for this independent-window
workflow.

**Stop:** call `full_shutdown()`. If shutdown cannot be verified, physically
interrupt OUT1 before troubleshooting. Review failures before rerunning; do not
loosen guards to hide instability. The companion Python file is for copying the
definition into an existing notebook, **not** for importing.

Manual calls accept `fir_mode='auto'` or `'bypass'`. cells 13–18 explains electronics-only
calibration and real passband limits. The code reads the installed FIR coefficients
and models their product with CIC instead of assuming a flat FIR. Existing PSD/dB
units follow the complex-envelope convention detailed in cells 13–18. Choosing bypass
changes measurement filtering only; it does not bypass or modify OUT1 feedback.

After upgrading from the earlier assumed-flat-FIR implementation, acquire the
broad baseline once again. Old result dictionaries are rejected to avoid mixing
different correction conventions; the new baseline can then be reused normally.


## 19 — Load manual measurement function

Run this definition once after the acquisition and filter helpers. It does not
operate hardware and is safe to rerun without clearing saved measurements.
Then use the two separate call cells below. No output is expected here.


In [ ]:
def measure_manual_feedback(gain=None, phase_deg=0.0, coefficient_counts=None,
                            frequency_range_hz=(40.4e6, 41.4e6),
                            decimation_log2=6, samples_per_record=8192,
                            averages=32, repeats=1, fit_range_hz=None,
                            q_guess=150.0, q_bounds=(20.0, 1e6),
                            lo_hz=None, phase_reference_hz=None, settle_s=0.10,
                            max_input_vpk=None, max_output_vpk=None,
                            max_power_ratio=10.0, max_active_seconds=30.0,
                            response_calibration=None, label='manual measurement',
                            mode='baseline', baseline_measurement=None, fir_mode='auto',
                            show_plots=True, map_safety_mode=False,
                            minimum_edge_coverage_linewidths=1.0,
                            maximum_fit_uncertainty_linewidths=0.25):
    """Measure a baseline once, then independent feedback spectra on later calls.

    Paste this single definition after the notebook's standard hardware and
    passive-acquisition helpers. No campaign, JSON, or Step 9A is used.
    mode='baseline': acquire feedback-off only; keep the returned dictionary.
    mode='feedback': pass that dictionary as baseline_measurement; acquire only
    the selected feedback state, with its own window, resolution and fit.
    mode='bracketed' retains the optional same-window off/on/off experiment.
    fir_mode='auto' enables the installed FIR at R>=256; 'bypass' measures CIC only.
    Set EITHER gain (signed, dimensionless) OR coefficient_counts (nonnegative
    integer, gain = count/16384). Neither argument means a zero-feedback test.
    decimation_log2=6 means divide 125 MHz by 64, NOT decimation by six.
    averages is the number of independent FFT powers averaged per state;
    repeats repeats the selected state (or bracketed triplet). Records are discarded.

    Returns spectra, Lorentzian fits, per-repeat results, controls and warnings.
    Also retains the latest/partial result in LAST_MANUAL_FEEDBACK_MEASUREMENT.
    A call never leaves feedback running. Ctrl-C attempts full shutdown, but
    cannot protect against a hung/crashed kernel: retain a physical disconnect.
    """
    import time
    import copy
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.optimize import least_squares
    from scipy.ndimage import median_filter

    global LAST_MANUAL_FEEDBACK_MEASUREMENT
    required = ('acq', 'gen', 'ps', 'sc', 'capture', 'full_shutdown',
                'correct_acquisition_response', 'quantized_feedback_vector', 'read_acquisition_fir',
                'FS_CLK', 'RP_LIMITS', 'BUILD_ID_EXPECTED', 'ADC_RANGE',
                'ADC_RAW_VOLTS_PER_COUNT', 'CIC_VOLTS_PER_COUNT',
                'ADC_OPERATING_GUARD_VPK', 'OUT1_OPERATING_GUARD_VPK',
                'IN1_DC_OFFSET_MAX_V', 'MIN_ACQUISITION_DECIMATION_LOG2',
                'MAX_ACQUISITION_DECIMATION_LOG2')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError('Manual measurement needs the standard configuration, '
                           'hardware/identity and cell 12 acquisition helpers. '
                           'Missing: %s. No hardware was accessed.' % ', '.join(missing))
    result = {'label': str(label), 'status': 'initialising', 'spectra': {},
              'fits': {}, 'runs': [], 'warnings': [], 'shutdown_verified': False,
              'created_epoch_s': time.time(),
              'mode': mode, 'recovery_measured': False, 'filter_model_version': 2,
              'electrode_voltage_directly_measured': False,
              'feedback_path_transfer_directly_measured': False}
    LAST_MANUAL_FEEDBACK_MEASUREMENT = result
    try:
        full_shutdown()
        if fir_mode not in ('auto', 'bypass'):
            raise ValueError("fir_mode must be 'auto' or 'bypass'; CIC order is fixed in this FPGA.")
        fir_profile = read_acquisition_fir()
        if ADC_RANGE != 'LV':
            raise ValueError('This helper uses the notebook LV ADC calibration. Restore the '
                             'matching LV jumper/range configuration before using it; do not '
                             'interpret HV input codes with the LV volts-per-count scale.')
        if gain is not None and coefficient_counts is not None:
            raise ValueError('Choose gain OR coefficient_counts, not both. For count 32 '
                             'use coefficient_counts=32 or gain=32/16384.0.')
        signed_gain = 0.0 if gain is None else float(gain)
        if coefficient_counts is not None:
            value = float(coefficient_counts)
            if not np.isfinite(value) or value != int(value) or value < 0:
                raise ValueError('coefficient_counts must be a nonnegative integer; zero is an off-only test.')
            signed_gain = value/16384.0
        if not np.isfinite(signed_gain) or abs(signed_gain) >= 1.99:
            raise ValueError('The existing phase-shifter interface requires |gain| < 1.99. '
                             'This digital range is NOT a safe physical-loop gain range.')
        count = int(round(abs(signed_gain)*16384.0))
        if mode not in ('baseline', 'feedback', 'bracketed'):
            raise ValueError("mode must be 'baseline', 'feedback', or 'bracketed'.")
        if mode == 'baseline' and count:
            raise ValueError("Baseline mode cannot apply gain. First call with mode='baseline' "
                             "and no gain; then use mode='feedback', baseline_measurement=your_result.")
        if mode != 'feedback' and baseline_measurement is not None:
            raise ValueError("baseline_measurement is only used with mode='feedback'.")
        if mode == 'feedback':
            base = baseline_measurement
            if (not isinstance(base, dict) or base.get('mode') != 'baseline' or
                    base.get('status') != 'complete' or not base.get('shutdown_verified') or
                    not base.get('fits', {}).get('before', {}).get('accepted')):
                raise ValueError("A successful mode='baseline' result is required. Measure it once "
                                 "with a broad enough window; do not substitute a failed fit or active result.")
            if base.get('filter_model_version') != 2:
                raise ValueError('This baseline predates the corrected CIC x FIR model. Acquire a new broad baseline '
                                 'once with the updated function, then reuse it for subsequent feedback measurements.')
            expected_context = (str(BUILD_ID_EXPECTED), ADC_RANGE, float(ADC_RAW_VOLTS_PER_COUNT),
                                float(CIC_VOLTS_PER_COUNT), float(FS_CLK))
            if tuple(base.get('input_context', ())) != expected_context:
                raise ValueError('Board identity, range, clock or input scaling differs from the cached baseline. '
                                 'Acquire a new baseline before applying feedback.')
            for key in ('f0_hz', 'q', 'linewidth_hz'):
                if not np.isfinite(base['fits']['before'].get(key, np.nan)) or base['fits']['before'][key] <= 0:
                    raise ValueError('Cached baseline has an invalid %s; acquire a new baseline.' % key)
            # Own copies prevent accumulation/refitting from changing the saved baseline.
            result['spectra']['before'] = copy.deepcopy(base['spectra']['before'])
            result['fits']['before'] = copy.deepcopy(base['fits']['before'])
            result['baseline_age_s'] = max(0.0, time.time()-base['created_epoch_s'])
            result['baseline_label'] = base['label']
            result['baseline_controls'] = copy.deepcopy(base['controls'])
            result['comparison_note'] = ('Historical baseline: drift and spectral recovery are not measured. '
                                         'Reacquire after any physical setup change or suspected drift.')
        result['input_context'] = (str(BUILD_ID_EXPECTED), ADC_RANGE, float(ADC_RAW_VOLTS_PER_COUNT),
                                   float(CIC_VOLTS_PER_COUNT), float(FS_CLK))
        if signed_gain != 0 and count == 0:
            raise ValueError('Requested gain rounds to zero coefficients. Use zero explicitly '
                             'for a control measurement, or at least one coefficient count.')
        if not isinstance(show_plots, (bool, np.bool_)) or not isinstance(map_safety_mode, (bool, np.bool_)):
            raise ValueError('show_plots and map_safety_mode must be Boolean values.')
        if map_safety_mode and mode != 'feedback':
            raise ValueError('map_safety_mode is reserved for feedback points managed by run_feedback_map().')
        if not (0.10 <= float(minimum_edge_coverage_linewidths) <= 2.0):
            raise ValueError('minimum_edge_coverage_linewidths must be 0.10..2.0. Lower values allow a broadened peak near a window edge but do not remove centre, resolution, or uncertainty checks.')
        if not (0.10 <= float(maximum_fit_uncertainty_linewidths) <= 1.0):
            raise ValueError('maximum_fit_uncertainty_linewidths must be 0.10..1.0. It limits both centre and linewidth 1-sigma uncertainty relative to fitted linewidth.')
        if not np.isfinite(phase_deg):
            raise ValueError('phase_deg must be a finite angle in degrees.')
        applied_phase = (float(phase_deg)+(180.0 if signed_gain < 0 else 0.0)) % 360.0
        for name, value, low, high in (
                ('averages', averages, 4, 1024), ('repeats', repeats, 1, 20),
                ('samples_per_record', samples_per_record, 256, 65536),
                ('decimation_log2', decimation_log2,
                 MIN_ACQUISITION_DECIMATION_LOG2, MAX_ACQUISITION_DECIMATION_LOG2)):
            if not np.isfinite(value) or value != int(value) or not low <= value <= high:
                raise ValueError('%s must be an integer in %d..%d; received %r.' % (name, low, high, value))
        n, averages, repeats, decimation_log2 = map(int, (samples_per_record, averages, repeats, decimation_log2))
        if n & (n-1):
            raise ValueError('samples_per_record must be a power of two: 4096, 8192, 16384, etc.')
        low_hz, high_hz = map(float, frequency_range_hz)
        fit_low, fit_high = map(float, frequency_range_hz if fit_range_hz is None else fit_range_hz)
        q_low, q_high = map(float, q_bounds)
        centre = (low_hz+high_hz)/2.0 if lo_hz is None else float(lo_hz)
        reference = (fit_low+fit_high)/2.0 if phase_reference_hz is None else float(phase_reference_hz)
        if mode == 'feedback' and phase_reference_hz is None:
            # Acquisition LO/window changes must not silently rotate the feedback vector.
            reference = float(base['controls']['phase_reference_hz'])
        values = (low_hz, high_hz, fit_low, fit_high, centre, reference,
                  q_low, q_high, q_guess, settle_s, max_active_seconds, max_power_ratio)
        if not np.all(np.isfinite(values)):
            raise ValueError('Frequency ranges, Q values, settling time and guards must all be finite.')
        rf_limit = min(FS_CLK/2, RP_LIMITS['input_bandwidth_hz'], RP_LIMITS['output_bandwidth_hz'])
        if not (0 < low_hz < high_hz < rf_limit and low_hz <= fit_low < fit_high <= high_hz
                and 0 < reference < rf_limit and 0 < q_low < q_guess < q_high):
            raise ValueError('Use an ordered positive frequency range within %.3f MHz, '
                             'a fit range inside it, and q_low < q_guess < q_high.' % (rf_limit/1e6))
        if settle_s < 0.01 or not 1 <= max_active_seconds <= 300 or max_power_ratio < 1:
            raise ValueError('Use settle_s >= 0.01, max_active_seconds in 1..300, '
                             'and max_power_ratio >= 1. These are measurement guards, not interlocks.')
        sample_rate = FS_CLK/(1 << decimation_log2)
        if max(abs(low_hz-centre), abs(high_hz-centre)) > .4*sample_rate:
            raise ValueError('Requested %.3f..%.3f MHz does not fit inside the conservative '
                             '%.3f MHz usable span around LO %.3f MHz at decimation_log2=%d. '
                             'Reduce decimation_log2 or narrow/recentre the requested band.' %
                             (low_hz/1e6, high_hz/1e6, .8*sample_rate/1e6, centre/1e6, decimation_log2))
        if averages*(settle_s+n/sample_rate) >= max_active_seconds:
            raise ValueError('The requested averages already exceed max_active_seconds before '
                             'scope/readout overhead. Reduce averages or choose a deliberate longer limit.')
        input_guard = ADC_OPERATING_GUARD_VPK if max_input_vpk is None else float(max_input_vpk)
        output_guard = OUT1_OPERATING_GUARD_VPK if max_output_vpk is None else float(max_output_vpk)
        if not (np.isfinite(input_guard+output_guard) and
                0 < input_guard <= min(ADC_OPERATING_GUARD_VPK, .98*RP_LIMITS['input_lv_full_scale_vpk']) and
                0 < output_guard <= OUT1_OPERATING_GUARD_VPK):
            raise ValueError('Input/output guards must be positive and no higher than the '
                             'notebook operating guards. They can be reduced for a sensitive setup.')
        plan = {'centre_hz': centre, 'decimation_log2': decimation_log2,
                'sample_rate_hz': sample_rate, 'samples_per_record': n,
                'bin_hz': sample_rate/n, 'full_span_hz': sample_rate,
                'fir_mode': fir_mode, 'fir_half_counts': fir_profile['half_counts'],
                'fir_digest': fir_profile['digest']}
        window = np.hanning(n)
        enbw = sample_rate*np.sum(window**2)/np.sum(window)**2
        frequencies = centre+np.fft.fftshift(np.fft.fftfreq(n, 1.0/sample_rate))
        display = (frequencies >= low_hz) & (frequencies <= high_hz)
        lo_word = int(round(centre/FS_CLK*(1 << 32))) & 0xFFFFFFFF
        command = (quantized_feedback_vector(count, applied_phase, reference) if count else
                   {'ci_word': 0, 'cq_word': 0, 'ci_signed': 0, 'cq_signed': 0, 'actual_gain': 0.0})
        result['controls'] = dict(plan, requested_gain=signed_gain, coefficient_counts=count,
                                  phase_deg=float(phase_deg), applied_phase_deg=applied_phase,
                                  phase_reference_hz=reference, averages=averages, repeats=repeats,
                                  fit_range_hz=(fit_low, fit_high), frequency_range_hz=(low_hz, high_hz),
                                  enbw_hz=float(enbw), actual_coefficients=command,
                                  q_guess=q_guess, q_bounds=(q_low, q_high), settle_s=settle_s,
                                  minimum_edge_coverage_linewidths=float(minimum_edge_coverage_linewidths),
                                  maximum_fit_uncertainty_linewidths=float(maximum_fit_uncertainty_linewidths),
                                  max_input_vpk=input_guard, max_output_vpk=output_guard,
                                  max_power_ratio=max_power_ratio, max_active_seconds=max_active_seconds)

        def _manual_input_check(held, proposed=None):
            """Use raw ADC samples including DC; never mistake mean subtraction for headroom."""
            if ('0x%08X' % acq.build_id()).lower() != str(BUILD_ID_EXPECTED).lower():
                raise RuntimeError('FPGA identity changed. Close other board apps and reload the qualified image.')
            ps.assert_held(held)
            gen.assert_b_zero()
            if gen.r(0x04) != 0 or gen.r(0x00) & 0xFF != 0xC0 or held['monitor']:
                raise RuntimeError('A generator or OUT2 is active; this passive test requires them off.')
            codes, rate = sc.acquire(dec=1, n=4096, chan='A')
            codes = np.asarray(codes, dtype=float)
            if sc.r(0x04) != 0 or codes.shape != (4096,) or not np.all(np.isfinite(codes)):
                raise RuntimeError('Raw ADC safety capture timed out or returned invalid data. Check scope/FPGA state.')
            peak = float(np.max(np.abs(codes)))*ADC_RAW_VOLTS_PER_COUNT
            dc = float(np.mean(codes))*ADC_RAW_VOLTS_PER_COUNT
            rms = float(np.std(codes))*ADC_RAW_VOLTS_PER_COUNT
            state = held if proposed is None else proposed
            words = [int(state[key]) & 0xFFFF for key in ('ci', 'cq')]
            signed = [word-65536 if word & 0x8000 else word for word in words]
            # L1 sum bounds I/Q peaks; factor two allows an unloaded 50-ohm-source
            # output. This bounds the DAC signal, NOT voltage at an unknown electrode.
            bound = (2*peak/RP_LIMITS['input_lv_full_scale_vpk']*sum(abs(v) for v in signed)/16384.0 *
                     RP_LIMITS['output_full_scale_vpk_into_50ohm'])
            if any(signed):
                bound += 4*RP_LIMITS['output_full_scale_vpk_into_50ohm']/8192.0
            if peak >= input_guard or abs(dc) > IN1_DC_OFFSET_MAX_V or bound >= output_guard:
                raise RuntimeError('HEADROOM/DC GUARD: IN1 peak %.6g V (limit %.6g), mean %+.6g V '
                                   '(absolute limit %.6g), unloaded OUT1 bound %.6g Vpk (limit %.6g). '
                                   'Reduce gain/input level or investigate offset; do not subtract it to hide overload.' %
                                   (peak, input_guard, dc, IN1_DC_OFFSET_MAX_V, bound, output_guard))
            return {'adc_peak_v': peak, 'adc_dc_v': dc, 'adc_ac_rms_v': rms, 'out1_bound_vpk': bound}

        def _manual_fit(frequency, psd, expected_q):
            """Lorentzian power peak + smooth positive background, in scaled coordinates."""
            mask = ((frequency >= fit_low) & (frequency <= fit_high) &
                    np.isfinite(psd) & (psd > 0) &
                    (np.abs(frequency-centre) > 2*plan['bin_hz']))
            # Mean removal makes a small artificial notch at the acquisition LO.
            # Exclude it and the adjacent Hann bins; it is not an electron dip.
            f, y_all = frequency[mask], psd[mask]
            if len(f) < 128:
                return {'accepted': False, 'reasons': ['Fewer than 128 usable fit bins; increase samples or widen the fit range.']}
            span, midpoint = float(f[-1]-f[0]), float((f[-1]+f[0])/2)
            x_all = (f-midpoint)/span
            median = median_filter(y_all, size=5, mode='nearest')
            keep = (y_all/median < 5) & (median/y_all < 5)
            x, y = x_all[keep], y_all[keep]
            if len(y) < 128:
                return {'accepted': False, 'reasons': ['Too many narrow spurs/dips; use a clean fit window.']}
            log_y = np.log(y)
            design = np.column_stack((np.ones(len(x)), x))
            bg = np.linalg.lstsq(design, log_y, rcond=-1)[0]  # legacy board NumPy
            minimum_width = max(midpoint/q_high, 4*plan['bin_hz'])/span
            maximum_width = min(midpoint/q_low/span, .95)
            if minimum_width >= maximum_width:
                return {'accepted': False, 'reasons': ['Resolution/window cannot cover the requested Q bounds.']}
            seed_width = np.clip(midpoint/expected_q/span, minimum_width*1.01, maximum_width*.99)
            lower = [bg[0]-8, -20, np.log(np.finfo(float).tiny), -.49, np.log(minimum_width)]
            upper = [bg[0]+5, 20, np.log(max(y)*100), .49, np.log(maximum_width)]
            smooth = median_filter(y, size=max(5, int(seed_width*len(y)/8)) | 1)
            peak_x = float(x[int(np.argmax(smooth))])
            starts = ([bg[0], bg[1], np.log(max(y)), 0.0, np.log(seed_width)],
                      [np.log(np.percentile(y, 10)*.25), 0.0, np.log(max(y)), peak_x, np.log(seed_width)])
            def _manual_model(p, xx):
                return np.exp(p[0]+p[1]*xx)+np.exp(p[2])/(1+4*((xx-p[3])/np.exp(p[4]))**2)
            def _manual_residual(p):
                return np.log(_manual_model(p, x))-log_y
            trials = [least_squares(_manual_residual, np.clip(start, np.asarray(lower)+1e-9,
                       np.asarray(upper)-1e-9), bounds=(lower, upper), loss='soft_l1',
                       f_scale=.15, max_nfev=3000) for start in starts]
            solution = min(trials, key=lambda trial: trial.cost)
            width = float(np.exp(solution.x[4])*span)
            f0 = float(midpoint+solution.x[3]*span)
            rss = max(float(np.sum(_manual_residual(solution.x)**2)), np.finfo(float).tiny)
            rss0 = max(float(np.sum((design.dot(bg)-log_y)**2)), np.finfo(float).tiny)
            delta_bic = len(y)*np.log(rss0/rss)-3*np.log(len(y))
            covariance = np.linalg.pinv(solution.jac.T.dot(solution.jac))*2*rss/max(1, len(y)-5)
            sf = float(span*np.sqrt(max(0, covariance[3, 3])))
            sw = float(width*np.sqrt(max(0, covariance[4, 4])))
            q = f0/width
            # Include centre/width covariance when propagating Q = f0/width.
            gradient = np.array([span/width, -q])
            cov_fw = covariance[np.ix_([3, 4], [3, 4])]
            sq = float(np.sqrt(max(0, gradient.dot(cov_fw).dot(gradient))))
            model = _manual_model(solution.x, x_all)
            reasons = []
            if not solution.success or delta_bic < 10:
                reasons.append('Peak is not convincingly distinguished from background (delta BIC %.2f).' % delta_bic)
            if width <= minimum_width*span*1.02 or width >= maximum_width*span*.98:
                reasons.append('Linewidth reached a fit bound; revise Q bounds/window/resolution.')
            if width/plan['bin_hz'] < 20:
                reasons.append('Only %.1f bins per linewidth; increase samples or decimation.' % (width/plan['bin_hz']))
            edge_coverage=min(f0-f[0],f[-1]-f0)/width
            if edge_coverage < float(minimum_edge_coverage_linewidths):
                reasons.append('Fit window reaches only %.2f linewidths from the centre to its nearer edge; this measurement requires %.2f. Widen/recentre the configured range, or deliberately lower minimum_edge_coverage_linewidths for a coarse Q-reducing diagnostic.' %
                               (edge_coverage,float(minimum_edge_coverage_linewidths)))
            uncertainty_limit=float(maximum_fit_uncertainty_linewidths)*width
            if sf > uncertainty_limit or sw > uncertainty_limit or not np.isfinite(sf+sw+sq):
                reasons.append('Fit uncertainty is too large: centre uncertainty is %.3f and linewidth uncertainty %.3f linewidths; the configured limit is %.3f. Increase averaging or widen/isolate the correct peak.' %
                               (sf/width,sw/width,float(maximum_fit_uncertainty_linewidths)))
            if np.max(y_all/model) > 8:
                reasons.append('Strong narrow residual/spur; inspect for oscillation or another mode.')
            return {'accepted': not reasons, 'reasons': reasons, 'f0_hz': f0, 'q': q,
                    'linewidth_hz': width, 'f0_uncertainty_hz': sf, 'q_uncertainty': sq,
                    'linewidth_uncertainty_hz': sw, 'frequency_hz': f, 'model': model,
                    'delta_bic': float(delta_bic), 'uncertainty_kind': 'approximate correlated-bin fit covariance'}

        # Check the requested frequency window against the actual response mask
        # BEFORE energising anything. Measured correction is optional, never loaded.
        trial = correct_acquisition_response({'frequency_hz': frequencies,
                  'psd_v2_per_hz': np.ones(n)}, plan, calibration=response_calibration)
        if not np.all(np.isfinite(trial['psd_v2_per_hz'][display])):
            raise ValueError('Part of the requested band needs excessive/unknown filter correction. '
                             'The installed FIR is flat only in its central passband (about +/-0.20 of output rate), '
                             'not throughout Nyquist. Narrow/recentre the band, reduce decimation, or deliberately '
                             "select fir_mode='bypass' and assess CIC alias rejection. No feedback has been applied.")
        print('%s: gain %+.7g -> count %d; phase %.2f deg; LO %.6f MHz; '
              'span %.1f kHz; bins %.2f Hz; ENBW %.2f Hz; %d averages x %d repeats.' %
              (label, signed_gain, count, applied_phase, centre/1e6, (high_hz-low_hz)/1e3,
               plan['bin_hz'], enbw, averages, repeats))
        states = ('before',) if mode == 'baseline' else (('on',) if mode == 'feedback' else ('before', 'on', 'after'))
        print('Acquiring %s only. Both generators and OUT2 stay off.' % ' -> '.join(states))
        if mode == 'feedback':
            print('Reusing baseline "%s" (age %.1f minutes), unchanged and without recapture/refit. '
                  'Phase reference stays at %.6f MHz.' %
                  (result['baseline_label'], result['baseline_age_s']/60, reference/1e6))
            print(result['comparison_note'])
        print('The fit excludes +/- %.2f Hz around the LO, where record-mean removal '
              'creates an artificial notch; no DC is subtracted from the FPGA feedback path.' % (2*plan['bin_hz']))
        result['status'] = 'measuring'
        for repetition in range(repeats):
            run = {'repeat': repetition+1, 'fits': {}, 'health': {}}
            result['runs'].append(run)
            for state_name in states:
                result['active_stage'] = (repetition+1, state_name)
                full_shutdown()
                held = ps.readback()
                if mode == 'feedback':
                    # Fresh, short full-band safety sample, NOT a resonance/baseline spectrum.
                    run['fresh_off_health'] = _manual_input_check(held)
                if state_name == 'on' and count:
                    ps.config_delay(reference)
                    held = ps.readback()
                    proposed = dict(held, ci=command['ci_word'], cq=command['cq_word'])
                    _manual_input_check(held, proposed)
                    ps.set(count/16384.0, applied_phase)
                    ps.assert_held(proposed)
                    held = proposed
                total = np.zeros(n)
                rms_values, raw_rms = [], []
                deadline = time.monotonic()+max_active_seconds
                try:
                    for record in range(averages):
                        if time.monotonic() > deadline:
                            raise RuntimeError('Measurement time limit reached during %s. Reduce averages '
                                               'or inspect slow capture; this software limit is not a watchdog.' % state_name)
                        health = _manual_input_check(held)
                        raw_rms.append(health['adc_ac_rms_v'])
                        if state_name == 'on':
                            off_rms = (run['fresh_off_health']['adc_ac_rms_v'] if mode == 'feedback'
                                       else run['health']['before']['mean_adc_ac_rms_v'])
                            reference_power = max(off_rms**2,
                                                  ADC_RAW_VOLTS_PER_COUNT**2)
                            if not map_safety_mode and raw_rms[-1]**2/reference_power > max_power_ratio:
                                raise RuntimeError('Full-band ADC power exceeded %.2fx the feedback-off '
                                                   'reference. Reduce gain; inspect amplifier/pickup/oscillation.' % max_power_ratio)
                        raw = capture(lo_word, decimation_log2, n, settle=settle_s, fir_mode=fir_mode)
                        values = np.asarray(raw)*CIC_VOLTS_PER_COUNT
                        if values.shape != (n,) or not np.all(np.isfinite(values)):
                            raise RuntimeError('DMA sample count/non-finite data failure. Check capture bounds and FPGA identity.')
                        values = values-np.mean(values)  # FFT only; raw DC still counts for headroom above.
                        power = np.abs(np.fft.fftshift(np.fft.fft(values*window)))**2/(sample_rate*np.sum(window**2))
                        if not np.all(np.isfinite(power)) or np.sum(power) <= 0:
                            raise RuntimeError('FFT power is zero/non-finite. Check acquisition and input scaling.')
                        total += power
                        rms_values.append(float(np.sqrt(np.mean(np.abs(values)**2))))
                        ps.assert_held(held)
                        if state_name == 'on' and not map_safety_mode and (record+1) % 8 == 0:
                            running_psd = total/(record+1)
                            running_floor = median_filter(running_psd, size=7, mode='nearest')
                            if np.max(running_psd[display]/np.maximum(running_floor[display], np.finfo(float).tiny)) > 20:
                                raise RuntimeError('A narrow carrier appeared during feedback. Stop and '
                                                   'inspect it before calling it a high-Q resonance.')
                finally:
                    full_shutdown()  # Fit/plot only after feedback has been removed.
                if read_acquisition_fir()['digest'] != fir_profile['digest']:
                    raise RuntimeError('FIR coefficients changed during measurement. Feedback is off; discard this result.')
                spectrum = correct_acquisition_response({'frequency_hz': frequencies,
                               'psd_v2_per_hz': total/averages}, plan, calibration=response_calibration)
                observed = spectrum['psd_v2_per_hz'][display]
                fit = _manual_fit(frequencies[display], observed, q_guess)
                run['fits'][state_name] = {key: value for key, value in fit.items()
                                           if key not in ('model', 'frequency_hz')}
                cv = float(np.std(rms_values, ddof=1)/np.mean(rms_values))
                run['health'][state_name] = dict(health, rms_cv=cv,
                                               mean_adc_ac_rms_v=float(np.mean(raw_rms)))
                if cv > .30:
                    fit['accepted'] = False
                    fit['reasons'].append('Nonstationary record RMS (CV %.3f > 0.30).' % cv)
                    run['fits'][state_name]['accepted'] = False
                result['fits'][state_name] = fit
                if state_name not in result['spectra']:
                    result['spectra'][state_name] = {'frequency_hz': frequencies[display].copy(),
                        'psd_v2_per_hz': observed.copy(), 'completed_repeats': 1,
                        'response_source': spectrum.get('response_source', 'unspecified')}
                else:
                    accumulated = result['spectra'][state_name]
                    completed = accumulated['completed_repeats']
                    accumulated['psd_v2_per_hz'] += (observed-accumulated['psd_v2_per_hz'])/(completed+1)
                    accumulated['completed_repeats'] += 1
                if state_name == 'before' and not fit['accepted']:
                    result['warnings'].append('Feedback was not enabled: the fresh off baseline is unreliable. '
                                               + ' '.join(fit['reasons']))
                    break
            if mode != 'bracketed':
                if not all(row['accepted'] for row in run['fits'].values()):
                    result['warnings'].append('Stopped further repeats because the measured fit is unreliable.')
                    break
                continue
            if len(run['fits']) < 3:
                break
            before, on, after = [run['fits'][key] for key in ('before', 'on', 'after')]
            result['recovery_measured'] = True
            if not all(row['accepted'] for row in (before, on, after)):
                result['warnings'].append('Stopped further repeats because at least one fit is unreliable.')
                break
            off_width = (before['linewidth_hz']+after['linewidth_hz'])/2
            run['k_damping'] = 1-on['linewidth_hz']/off_width
            run['shift_hz'] = on['f0_hz']-(before['f0_hz']+after['f0_hz'])/2
            print('Repeat %d: effective damping K %.4f; centre shift %+.2f Hz.' %
                  (repetition+1, run['k_damping'], run['shift_hz']))
            if (abs(after['f0_hz']-before['f0_hz']) > .25*off_width or
                    abs(after['linewidth_hz']-before['linewidth_hz']) > .25*off_width):
                result['warnings'].append('Off baseline did not recover within 0.25 linewidth '
                                           'in centre/width. Inspect drift/heating before another call.')
                break
        # Overlay the final, averaged spectra. All fitted/plot arrays share the
        # identical frequency mask, avoiding the earlier x/y-length mismatch.
        fig, axes = plt.subplots(3 if mode == 'feedback' else 2, 1,
                                 figsize=(11, 10 if mode == 'feedback' else 7))
        residual_axis = axes[-1]
        for state_name, color in (('before', '#1f77b4'), ('on', '#ff7f0e'), ('after', '#2ca02c')):
            if state_name not in result['spectra']:
                continue
            spectrum = result['spectra'][state_name]
            f, p = spectrum['frequency_hz'], spectrum['psd_v2_per_hz']
            # A single repeat was already fitted after shutdown; reuse that fit.
            # Multiple repeats need a fresh fit to their averaged spectrum.
            cached = mode == 'feedback' and state_name == 'before'
            fit = (result['fits'][state_name] if cached or spectrum['completed_repeats'] == 1
                   else _manual_fit(f, p, q_guess))
            prior = [run['fits'][state_name] for run in result['runs'] if state_name in run['fits']]
            if any(not row['accepted'] for row in prior):
                fit['accepted'] = False
                fit['reasons'].append('At least one individual repeat failed; averaging does not remove that failure.')
            if len(prior) > 1 and all('f0_hz' in row for row in prior):
                spread = float(np.std([row['f0_hz'] for row in prior], ddof=1))
                width_spread = float(np.std([row['linewidth_hz'] for row in prior], ddof=1))
                if max(spread, width_spread) > .1*fit.get('linewidth_hz', 0):
                    fit['accepted'] = False
                    fit['reasons'].append('Centre/linewidth drift distorts the average; inspect per-repeat fits.')
            result['fits'][state_name] = fit
            state_averages = result['baseline_controls']['averages'] if cached else averages
            axes[0].semilogy(f/1e6, p, color=color, alpha=.5, linewidth=.8,
                             label='%s PSD (%d x %d spectra)' %
                             (state_name, spectrum['completed_repeats'], state_averages))
            if mode == 'feedback':
                axes[1].semilogy(f/1e6, p, color=color, alpha=.5, linewidth=.8)
            if 'model' in fit:
                caption = '%s: f0 %.6f MHz, Q %.1f%s' % (state_name, fit['f0_hz']/1e6,
                          fit['q'], '' if fit['accepted'] else ' [UNRELIABLE]')
                axes[0].semilogy(fit['frequency_hz']/1e6, fit['model'], color=color,
                                 linewidth=2, linestyle='-' if fit['accepted'] else '--', label=caption)
                if mode == 'feedback':
                    axes[1].semilogy(fit['frequency_hz']/1e6, fit['model'], color=color,
                                     linewidth=2, linestyle='-' if fit['accepted'] else '--')
                residual_axis.plot(fit['frequency_hz']/1e6,
                             np.interp(fit['frequency_hz'], f, p)/fit['model']-1, color=color, alpha=.7)
                print('%s: f0 %.6f +/- %.6f MHz; Q %.2f +/- %.2f; linewidth %.2f Hz; accepted=%s' %
                      (state_name, fit['f0_hz']/1e6, fit['f0_uncertainty_hz']/1e6,
                       fit['q'], fit['q_uncertainty'], fit['linewidth_hz'], fit['accepted']))
            for reason in fit['reasons']:
                print('  %s: %s' % (state_name, reason))
        axes[0].set_ylabel('Corrected PSD (V²/Hz)')
        axes[0].set_title(str(label)+' — one manually selected feedback setting')
        axes[0].legend(fontsize=8)
        residual_axis.axhline(0, color='black', linewidth=.8)
        residual_axis.set_ylabel('PSD / fitted model − 1')
        residual_axis.set_xlabel('Frequency (MHz)')
        residual_axis.set_xlim(low_hz/1e6, high_hz/1e6)
        if mode == 'feedback':
            axes[0].set_title(str(label)+' — full baseline and feedback windows (native grids)')
            axes[1].set_title('Feedback-window zoom; baseline fit is unchanged')
            axes[1].set_ylabel('Corrected PSD (V²/Hz)')
            axes[1].set_xlim(low_hz/1e6, high_hz/1e6)
            before, on = result['fits']['before'], result['fits'].get('on', {})
            if on.get('accepted'):
                ratio = on['linewidth_hz']/before['linewidth_hz']
                result['comparison'] = {
                    'q_ratio': on['q']/before['q'], 'k_damping': 1-ratio,
                    'shift_hz': on['f0_hz']-before['f0_hz'],
                    'shift_fit_uncertainty_hz': float(np.hypot(on['f0_uncertainty_hz'], before['f0_uncertainty_hz'])),
                    'k_fit_uncertainty': float(ratio*np.hypot(on['linewidth_uncertainty_hz']/on['linewidth_hz'],
                                                            before['linewidth_uncertainty_hz']/before['linewidth_hz'])),
                    'unmeasured_drift_included': False}
                print('Relative to saved baseline: Q/Q0 %.4f, effective damping K %.4f, shift %+.2f Hz. '
                      'These are comparisons, not independently calibrated loop gain; unmeasured drift is excluded.' %
                      (result['comparison']['q_ratio'], result['comparison']['k_damping'], result['comparison']['shift_hz']))
        else:
            axes[0].set_xlim(low_hz/1e6, high_hz/1e6)
        fig.tight_layout()
        if show_plots:
            plt.show()
        else:
            plt.close(fig)
        # Additional dB view: logarithm of the averaged LINEAR PSD, not an
        # average of logarithms. Native high-impedance IN1 has no defined 50-ohm
        # delivered power, so do not label this as dBm or dBm/Hz.
        # BEGIN COPYABLE DB PLOT
        plot_result = result
        db_zoom = plot_result.get('mode') == 'feedback'
        db_fig, db_axes = plt.subplots(2 if db_zoom else 1, 1,
                                      figsize=(11, 8 if db_zoom else 4.5), squeeze=False)
        for state_name, color in (('before', '#1f77b4'), ('on', '#ff7f0e'), ('after', '#2ca02c')):
            spectrum = plot_result.get('spectra', {}).get(state_name)
            if spectrum is None:
                continue
            fit = plot_result.get('fits', {}).get(state_name, {})
            state_label = {'before': 'Feedback off (baseline)', 'on': 'Feedback on',
                           'after': 'Feedback off (after)'}[state_name]
            traces = [(spectrum['frequency_hz'], spectrum['psd_v2_per_hz'],
                       state_label + ' PSD', False)]
            if 'model' in fit:
                caption = '%s fit: Q %.1f, f0 %.6f MHz%s' % (
                    state_label, fit['q'], fit['f0_hz']/1e6,
                    '' if fit.get('accepted') else ' [UNRELIABLE]')
                traces.append((fit['frequency_hz'], fit['model'], caption, True))
            for frequency, power_density, caption, is_fit in traces:
                frequency = np.asarray(frequency, dtype=float)
                power_density = np.asarray(power_density, dtype=float)
                if frequency.ndim != 1 or frequency.shape != power_density.shape:
                    raise ValueError('dB plot: %s frequency/PSD arrays must have matching 1-D shapes.' % caption)
                # Leave invalid/zero bins as gaps; never manufacture a noise floor.
                valid = np.isfinite(frequency) & np.isfinite(power_density) & (power_density > 0)
                density_db = np.full(power_density.shape, np.nan)
                density_db[valid] = 10.0*np.log10(power_density[valid] / 1.0)
                for db_axis in db_axes[:, 0]:
                    db_axis.plot(frequency/1e6, density_db, color=color,
                                 alpha=1.0 if is_fit else .5,
                                 linewidth=2 if is_fit else .8,
                                 linestyle='--' if is_fit and not fit.get('accepted') else '-',
                                 label=caption)
        for db_axis in db_axes[:, 0]:
            db_axis.set_xlabel('Frequency (MHz)')
            db_axis.set_ylabel('PSD (dB re 1 V²/Hz)')
            db_axis.grid(True, alpha=.25)
        db_axes[0, 0].set_title(str(plot_result.get('label', 'Measurement')) + ' — dB spectrum')
        db_axes[0, 0].legend(fontsize=8)
        if db_zoom:
            limits = plot_result['controls']['frequency_range_hz']
            db_axes[1, 0].set_xlim(limits[0]/1e6, limits[1]/1e6)
            db_axes[1, 0].set_title('Feedback-window zoom; original baseline fit retained')
        db_fig.tight_layout()
        if locals().get('show_plots', True):
            plt.show()
        else:
            plt.close(db_fig)
        # END COPYABLE DB PLOT
        for warning in result['warnings']:
            print('WARNING: '+warning)
        expected_fits = 1 if mode == 'baseline' else (2 if mode == 'feedback' else 3)
        result['status'] = ('complete' if len(result['fits']) == expected_fits and not result['warnings'] and
                            all(row['accepted'] for row in result['fits'].values()) else 'review_required')
        return result
    except BaseException as exc:
        result['status'] = 'aborted'
        result['error'] = '%s: %s' % (type(exc).__name__, exc)
        raise
    finally:
        try:
            full_shutdown()
            result['shutdown_verified'] = True
        except BaseException:
            result['status'] = 'shutdown_failed'
            raise RuntimeError('Shutdown could not be verified. Physically disconnect OUT1 '
                               'before troubleshooting; do not start another measurement.')


# Define the function above first; then explicitly run a call in a separate cell.
# First acquire the broad OFF baseline once (retain this variable in memory):
# manual_baseline = measure_manual_feedback(mode='baseline', averages=32,
#     frequency_range_hz=(40.4e6, 41.4e6), q_guess=150, label='current setup baseline')
# Then rerun ONLY this call, choosing its own window/decimation/Q estimate:
# measurement = measure_manual_feedback(mode='feedback', baseline_measurement=manual_baseline,
#     coefficient_counts=32, phase_deg=0,
#     frequency_range_hz=(40.4e6, 41.4e6),
#     fit_range_hz=(40.5e6, 41.3e6),
#     decimation_log2=6, samples_per_record=8192, averages=32, repeats=1,
#     label='current amplifier / attenuator arrangement')
# Emergency software stop: full_shutdown()


## 20 — Measure the broad baseline once

Restore the passive detection/return wiring and remove the loopback cable and turn any external test source off.
Uncomment this call to acquire only feedback-off data with the selected broad
correction. Expect a spectrum, dB plot, Lorentzian fit, Q, frequency and shutdown.
A rejected fit must be investigated before reuse. Rerunning replaces the baseline;
changing only the later feedback window does not require rerunning this cell.


In [ ]:
# manual_baseline = measure_manual_feedback(
#     mode='baseline', frequency_range_hz=(40.4e6, 41.4e6),
#     fit_range_hz=(40.5e6, 41.3e6), decimation_log2=6,
#     samples_per_record=8192, averages=32, q_guess=150.0,
#     fir_mode='auto', response_calibration=BASELINE_RESPONSE,
#     phase_reference_hz=40.9e6, label='broad off baseline')


## 21 — Measure only the feedback spectrum

Use the same commissioned passive wiring and the accepted baseline above.
Choose a commissioned low coefficient and phase; count 32 is an example, not
proof of safe gain for a different circuit. The 180 kHz window fits inside the
195 kHz measured band at exponent 8. Change the correction if changing decimation.

Uncomment to acquire the active spectrum and overlay its fit with the saved baseline.
Expect Q/shift, linear and dB plots, then full shutdown. The baseline is not reacquired.
Review fit/guard failures before rerunning; every call can energise OUT1.
Emergency stop: `full_shutdown()`.


In [ ]:
# measurement = measure_manual_feedback(
#     mode='feedback', baseline_measurement=manual_baseline,
#     coefficient_counts=32, phase_deg=0.0,
#     frequency_range_hz=(40.81e6, 40.99e6),
#     fit_range_hz=(40.82e6, 40.98e6), decimation_log2=8,
#     samples_per_record=8192, averages=32, q_guess=2000.0,
#     fir_mode='auto', response_calibration=FEEDBACK_RESPONSE,
#     label='manual enhanced measurement')


# Disabled gain–phase campaign — archived cells 22–26

This map is disabled for the present workflow. All hardware-running calls remain
commented out. The definitions are retained only so existing compact campaign files
can still be plotted or inspected. Do not uncomment cells 24 or 25.

The archived campaign maps how feedback changes Q and resonant frequency. It does not
assume that 225° is regenerative: the coarse sweep samples it deliberately and
fits the physical axes from all accepted points.

The map reuses the one-point measurement’s fitted spectra, CIC/FIR correction,
raw-input checks, exact coefficient readback and shutdown. OUT2 remains off.
Only averaged spectra and fit records are retained; raw time records are discarded.


## 22 — Load the resumable map functions

Run the definition cell once. It defines the campaign, plotting and target-selection
interfaces without accessing hardware or creating files. It is safe to rerun.

The private helpers load saved summaries and calculate phase axes, repeatability
and leave-one-out interpolation checks. They are kept in this cell so the notebook
remains standalone.


In [ ]:
"""Resumable passive gain/phase mapping; Python 3.5 and old SciPy compatible."""


def run_feedback_map(stage, config, coarse_map=None, resume=False,
                     measurement_function=None):
    """Run one explicit coarse/final map stage and checkpoint every spectrum.

    This function can energise OUT1. It never applies more than the configured
    coefficient ceiling and always attempts full_shutdown() on every exit.
    """
    import os, io, csv, json, time, math, hashlib, shutil, gc
    import numpy as np
    from scipy.ndimage import median_filter

    measure = measure_manual_feedback if measurement_function is None else measurement_function
    def call_measurement(**kwargs):
        return measure(**kwargs)
    cfg = {
        'campaign_name': 'current_feedback_map',
        'minimum_coefficient_counts': 128,
        'maximum_coefficient_counts': 1024, 'maximum_k': .85,
        'phase_seed_deg': 225., 'profile': 'fast',
        'coarse_averages': 6, 'final_averages': 16,
        'time_budget_minutes': 55., 'campaign_root': 'feedback_maps',
        'baseline_frequency_range_hz': (40.15e6, 41.65e6),
        'baseline_fit_range_hz': (40.20e6, 41.60e6),
        'locator_frequency_range_hz': (40.15e6, 41.65e6),
        'baseline_decimation_log2': 6, 'baseline_samples_per_record': 8192,
        'baseline_averages': 16, 'feedback_decimation_log2': 8,
        'feedback_minimum_span_hz': 180e3, 'feedback_maximum_span_hz': 1.50e6,
        'feedback_linewidths_full_span': 4.0,
        'feedback_samples_per_record': 8192,
        'map_minimum_edge_coverage_linewidths': .35,
        'coarse_maximum_fit_uncertainty_linewidths': .60,
        'final_maximum_fit_uncertainty_linewidths': .35,
        'feedback_response_calibration': None, 'baseline_response_calibration': None,
        'maximum_background_ratio': 4., 'maximum_carrier_fraction': .35,
        'repeat_fraction': .10, 'seed': 225, 'q_bounds': (20., 1e6)}
    cfg.update(dict(config))
    if stage not in ('coarse', 'final'):
        raise ValueError("stage must be 'coarse' or 'final'. No hardware was accessed.")
    integers = ('minimum_coefficient_counts','maximum_coefficient_counts', 'coarse_averages', 'final_averages',
                'baseline_decimation_log2', 'baseline_samples_per_record',
                'feedback_decimation_log2', 'feedback_samples_per_record')
    for name in integers:
        value = cfg[name]
        if not np.isfinite(value) or value != int(value):
            raise ValueError('%s must be a finite integer; received %r.' % (name, value))
        cfg[name] = int(value)
    if not (1 <= cfg['maximum_coefficient_counts'] <= 1024):
        raise ValueError('maximum_coefficient_counts must be 1..1024. The 1024 ceiling is hard-coded in both planning and application.')
    if not (1 <= cfg['minimum_coefficient_counts'] <= cfg['maximum_coefficient_counts']):
        raise ValueError('minimum_coefficient_counts must be at least 1 and no greater than maximum_coefficient_counts.')
    if not (.05 <= float(cfg['maximum_k']) <= .85):
        raise ValueError('maximum_k must be 0.05..0.85; values closer to one are outside this campaign design.')
    if cfg['profile'] != 'fast' or not 1 <= float(cfg['time_budget_minutes']) <= 55:
        raise ValueError("This implementation supports profile='fast' with time_budget_minutes in 1..55.")
    if not (4 <= cfg['coarse_averages'] <= 64 and 4 <= cfg['final_averages'] <= 128 and
            0 <= float(cfg['repeat_fraction']) <= .25):
        raise ValueError('Averages must be 4..64 coarse and 4..128 final; repeat_fraction must be 0..0.25.')
    for name in ('baseline_frequency_range_hz','baseline_fit_range_hz','locator_frequency_range_hz'):
        if len(cfg[name]) != 2 or not np.all(np.isfinite(cfg[name])) or not 0 < cfg[name][0] < cfg[name][1] < FS_CLK/2:
            raise ValueError('%s must contain two ordered frequencies between zero and Nyquist.' % name)
    if not (cfg['baseline_frequency_range_hz'][0] <= cfg['baseline_fit_range_hz'][0] <
            cfg['baseline_fit_range_hz'][1] <= cfg['baseline_frequency_range_hz'][1]):
        raise ValueError('baseline_fit_range_hz must lie inside baseline_frequency_range_hz.')
    if not (10e3 <= float(cfg['feedback_minimum_span_hz']) <= float(cfg['feedback_maximum_span_hz']) and
            float(cfg['feedback_maximum_span_hz']) <= .8*FS_CLK/(1 << MIN_ACQUISITION_DECIMATION_LOG2)):
        raise ValueError('Feedback spans must be ordered, at least 10 kHz, and no wider than the acquisition path can safely display.')
    if not (1.0 <= float(cfg['feedback_linewidths_full_span']) <= 12.0 and
            .10 <= float(cfg['map_minimum_edge_coverage_linewidths']) <= 1.0 and
            .10 <= float(cfg['coarse_maximum_fit_uncertainty_linewidths']) <= 1.0 and
            .10 <= float(cfg['final_maximum_fit_uncertainty_linewidths']) <= 1.0):
        raise ValueError('Use feedback_linewidths_full_span=1..12, map edge coverage=0.10..1.0, and coarse/final fit-uncertainty limits=0.10..1.0.')

    def scalar(value):
        if isinstance(value, np.generic): return value.item()
        if isinstance(value, np.ndarray): return [scalar(v) for v in value.tolist()]
        if isinstance(value, (list, tuple)): return [scalar(v) for v in value]
        if isinstance(value, dict): return {str(k): scalar(v) for k, v in value.items()}
        return value

    def digest(value):
        raw = json.dumps(scalar(value), sort_keys=True, separators=(',', ':'), allow_nan=False)
        return hashlib.sha256(raw.encode('utf-8')).hexdigest()

    def compact_fit(fit):
        """Keep scalar fit results only; frequency grids and model curves are discarded."""
        names=('accepted','reasons','f0_hz','f0_uncertainty_hz','q','q_uncertainty',
               'linewidth_hz','linewidth_uncertainty_hz','delta_bic','uncertainty_kind')
        return dict((name,scalar(fit[name])) for name in names if name in fit)

    def atomic_json(path, value):
        temporary = path+'.tmp'
        with io.open(temporary, 'w', encoding='utf-8') as handle:
            handle.write(json.dumps(scalar(value), sort_keys=True, indent=2, allow_nan=False))
            handle.flush(); os.fsync(handle.fileno())
        for attempt in range(6):
            try:
                os.replace(temporary, path); break
            except PermissionError:
                if attempt == 5: raise
                time.sleep(.05*(attempt+1))

    def accepted_fit(result, state):
        fit = result.get('fits', {}).get(state, {})
        if result.get('status') not in ('complete', 'review_required') or not fit.get('accepted'):
            raise RuntimeError('%s fit was rejected: %s' %
                               (state, '; '.join(fit.get('reasons', result.get('warnings', ['unknown fit failure'])))))
        return fit

    def baseline_call(label):
        return call_measurement(mode='baseline', averages=cfg['baseline_averages'], repeats=1,
            frequency_range_hz=tuple(cfg['baseline_frequency_range_hz']),
            fit_range_hz=tuple(cfg['baseline_fit_range_hz']),
            decimation_log2=cfg['baseline_decimation_log2'],
            samples_per_record=cfg['baseline_samples_per_record'],
            q_guess=float(cfg.get('baseline_q_guess', 150.)), q_bounds=tuple(cfg['q_bounds']),
            phase_reference_hz=float(cfg.get('phase_reference_hz', CENTRE_HZ)),
            response_calibration=cfg['baseline_response_calibration'],
            fir_mode='auto', show_plots=False, label=label)

    def point_call(count, phase, baseline, prediction, averages, label, force_broad=False):
        if not 1 <= int(count) <= min(1024, cfg['maximum_coefficient_counts']):
            raise RuntimeError('Hardware application refused coefficient count %r outside 1..%d.' %
                               (count, min(1024, cfg['maximum_coefficient_counts'])))
        centre = float(prediction.get('f0_hz', accepted_fit(baseline, 'before')['f0_hz']))
        q = max(21., float(prediction.get('q', accepted_fit(baseline, 'before')['q'])))
        width = centre/q
        if force_broad:
            low,high=map(float,cfg['locator_frequency_range_hz'])
            centre=(low+high)/2.;span=high-low
        else:
            span=min(float(cfg['feedback_maximum_span_hz']),
                     max(float(cfg['feedback_minimum_span_hz']),
                         float(cfg['feedback_linewidths_full_span'])*width))
            low,high=centre-span/2.,centre+span/2.
        dec = cfg['feedback_decimation_log2']
        while dec>0 and .80*FS_CLK/(1 << dec)<span: dec-=1
        rate = FS_CLK/(1 << dec)
        # Increase N, rather than silently narrowing the requested mapped band.
        wanted = int(math.ceil(30.*rate/max(width, 1.)))
        n = cfg['feedback_samples_per_record']
        while n < wanted and n < 65536: n *= 2
        n = min(n, 65536)
        return call_measurement(mode='feedback', baseline_measurement=baseline,
            coefficient_counts=int(count), phase_deg=float(phase), averages=int(averages), repeats=1,
            frequency_range_hz=(low, high), fit_range_hz=(low+span*.05, high-span*.05),
            lo_hz=centre, phase_reference_hz=float(cfg.get('phase_reference_hz', CENTRE_HZ)),
            decimation_log2=dec, samples_per_record=n, q_guess=q, q_bounds=tuple(cfg['q_bounds']),
            response_calibration=cfg['feedback_response_calibration'], fir_mode='auto',
            minimum_edge_coverage_linewidths=float(cfg['map_minimum_edge_coverage_linewidths']),
            maximum_fit_uncertainty_linewidths=float(cfg[stage+'_maximum_fit_uncertainty_linewidths']),
            max_active_seconds=300., show_plots=False, map_safety_mode=True, label=label)

    def vector(count, phase):
        command = quantized_feedback_vector(int(count), float(phase),
                                            float(cfg.get('phase_reference_hz', CENTRE_HZ)))
        return int(command['ci_signed']), int(command['cq_signed']), float(command['actual_phase_deg'])

    def coarse_schedule():
        circles = [v for v in (128, 512, 1024)
                   if cfg['minimum_coefficient_counts'] <= v <= cfg['maximum_coefficient_counts']]
        line = [v for v in (128, 256, 384, 512, 640, 768, 896, 1024)
                if cfg['minimum_coefficient_counts'] <= v <= cfg['maximum_coefficient_counts']]
        if cfg['minimum_coefficient_counts'] not in circles+line:
            line.append(cfg['minimum_coefficient_counts'])
        seeded=[220.,float(cfg['phase_seed_deg'])%360.,230.]
        phases = sorted(dict.fromkeys([float(v) for v in range(0, 360, 30)] + seeded +
                                      [(v+180.)%360. for v in seeded]))
        rows = []
        for count in sorted(dict.fromkeys(circles+line)):
            use = phases if count in circles else [float(cfg['phase_seed_deg']) % 360.,
                                                    (float(cfg['phase_seed_deg'])+180.) % 360.]
            # Deterministic opposite pairs limit drift bias while preserving resume order.
            remaining = list(use); pairs = []
            while remaining:
                phase = remaining.pop(0); pair=[phase]
                opposite = (phase+180.) % 360.
                if opposite in remaining: remaining.remove(opposite); pair.append(opposite)
                pairs.append(pair)
            np.random.RandomState(int(cfg['seed'])+int(count)).shuffle(pairs)
            ordered=[phase for pair in pairs for phase in pair]
            rows.extend([(count, phase, False) for phase in ordered])
        return rows

    def final_counts(coarse):
        rows = [p for p in coarse.get('points', []) if p.get('accepted') and p.get('k') is not None]
        by_count = {}
        for row in rows: by_count[row['count']] = max(row['k'], by_count.get(row['count'], -1e9))
        positive = sorted((k, v) for k, v in by_count.items() if v >= 0)
        if len(positive) < 2:
            raise ValueError('Final mapping needs at least two accepted coarse gains with nonnegative K.')
        counts = np.array([v[0] for v in positive], float)
        kvals = np.maximum.accumulate(np.array([v[1] for v in positive], float))
        keep = np.r_[True, np.diff(kvals) > 1e-5]
        counts, kvals = counts[keep], kvals[keep]
        target = np.linspace(0, min(.80, float(np.max(kvals))), 9)
        result = sorted(dict.fromkeys(int(round(v)) for v in np.interp(target, kvals, counts)))
        return [v for v in result if cfg['minimum_coefficient_counts'] <= v <= cfg['maximum_coefficient_counts']]

    if stage == 'coarse':
        schedule = coarse_schedule(); averages = cfg['coarse_averages']
    else:
        if isinstance(coarse_map, str): coarse_map = _feedback_map_load(coarse_map)
        if not isinstance(coarse_map, dict) or coarse_map.get('stage') != 'coarse':
            raise ValueError('Final mapping requires the accepted/partial coarse map dictionary or its directory.')
        if coarse_map.get('status') != 'complete':
            raise ValueError('Final mapping requires a completed coarse campaign. Review and explicitly resume the stopped coarse campaign first.')
        counts = final_counts(coarse_map)
        base_phases = [float(v) for v in range(0, 360, 15)]
        seed_phase = float(cfg['phase_seed_deg']) % 360.
        schedule=[]
        for count in counts:
            phases=list(base_phases)+[seed_phase]
            if coarse_map.get('phase_axes'):
                nearest=min(coarse_map['phase_axes'],key=lambda row:abs(row['count']-count))
                phases.extend([float(nearest[name])%360. for name in
                               ('regenerative_deg','degenerative_deg','positive_shift_deg','negative_shift_deg')])
            phases=sorted(dict.fromkeys(round(value,6) for value in phases))
            schedule.extend((count,phase,False) for phase in phases)
        rng = np.random.RandomState(int(cfg['seed']))
        repeats = int(math.ceil(len(schedule)*float(cfg['repeat_fraction'])))
        for index in sorted(rng.choice(len(schedule), size=repeats, replace=False), reverse=True):
            count, phase, _ = schedule[index]
            schedule.insert(index+1, (count, phase, True))
        averages = cfg['final_averages']

    estimated_seconds = sum(averages*(SETTLE+cfg['feedback_samples_per_record'] /
                            (FS_CLK/(1 << cfg['feedback_decimation_log2'])))+.5 for _ in schedule)
    estimated_seconds += 2*len(dict.fromkeys(row[0] for row in schedule))*cfg['baseline_averages']*(SETTLE+.01)
    estimated_bytes = max(16384,len(schedule)*1024)
    print('%s map: %d points; estimated %.1f minutes and %.2f MB (fit table only; no spectra or raw records).' %
          (stage.upper(), len(schedule), estimated_seconds/60., estimated_bytes/2**20))
    if estimated_seconds > 60*float(cfg['time_budget_minutes']):
        raise ValueError('Estimated %.1f minutes exceeds the %.1f-minute budget. Reduce averages/grid deliberately; no hardware was accessed.' %
                         (estimated_seconds/60., cfg['time_budget_minutes']))

    campaign_id = str(cfg['campaign_name'])+'_'+stage
    root = os.path.abspath(str(cfg['campaign_root'])); directory = os.path.join(root, campaign_id)
    if not os.path.isdir(directory): os.makedirs(directory)
    if shutil.disk_usage(directory).free < max(100*2**20, estimated_bytes*2):
        raise IOError('Insufficient disk space for atomic compact checkpoints; no hardware was accessed.')
    manifest_path = os.path.join(directory, 'manifest.json')
    config_digest = digest(cfg)
    stored_config=dict(cfg)
    for calibration_name in ('baseline_response_calibration','feedback_response_calibration'):
        calibration=stored_config.get(calibration_name)
        stored_config[calibration_name]=(None if calibration is None else
            {'digest':str(calibration.get('digest',digest(calibration))) if isinstance(calibration,dict) else digest(calibration)})
    hardware_digest=(globals().get('TOPOLOGY_RECORD',{}).get('digest') or
                     digest(globals().get('TOPOLOGY',{'unspecified':True})))
    manifest = {'schema_version': 2, 'stage': stage, 'config': scalar(stored_config),
                'config_digest': config_digest, 'campaign_directory': directory,
                'created_epoch_s': time.time(), 'status': 'planned', 'completed_ids': [],
                'stop_reason': None, 'build_id': str(BUILD_ID_EXPECTED),
                'fir_digest': read_acquisition_fir()['digest'],
                'hardware_digest':hardware_digest,'out2_used': False}
    if os.path.exists(manifest_path):
        with io.open(manifest_path, 'r', encoding='utf-8') as handle: old = json.load(handle)
        if not resume:
            raise IOError('Campaign directory already exists. Choose a new campaign_name, or explicitly pass resume=True.')
        for name in ('schema_version', 'stage', 'config_digest', 'build_id', 'fir_digest','hardware_digest'):
            if old.get(name) != manifest.get(name):
                raise ValueError('Resume refused: %s changed. Start a new campaign rather than mixing incompatible points.' % name)
        terminal_reason=str(old.get('stop_reason') or '').lower()
        if 'k stop' in terminal_reason or 'time-interpolated k' in terminal_reason:
            raise RuntimeError('Resume refused: this campaign reached its K safety boundary. Review the partial map; a resume must never advance to a higher gain after a regenerative stop.')
        manifest = old
    elif resume:
        raise IOError('resume=True was requested but no manifest exists at %s.' % manifest_path)
    atomic_json(manifest_path, manifest)

    completed = dict((int(v), True) for v in manifest.get('completed_ids', []))
    points = []
    summary_path = os.path.join(directory, 'summary.csv')

    def save_summary():
        fields = ('point_id','stage','count','phase_deg','actual_phase_deg','ci','cq','repeat',
                  'timestamp_epoch_s','accepted','q','q_uncertainty','q_ratio','f0_hz',
                  'f0_uncertainty_hz','linewidth_hz','linewidth_uncertainty_hz','shift_hz',
                  'shift_uncertainty_hz','k','k_uncertainty','delta_bic','background_ratio',
                  'carrier_fraction','reason')
        temporary = summary_path+'.tmp'
        with io.open(temporary, 'w', newline='', encoding='utf-8') as handle:
            writer = csv.DictWriter(handle, fieldnames=fields); writer.writeheader()
            for row in sorted(points, key=lambda value: value['point_id']):
                writer.writerow({name: scalar(row.get(name, '')) for name in fields})
            handle.flush(); os.fsync(handle.fileno())
        for attempt in range(6):
            try:
                os.replace(temporary, summary_path); break
            except PermissionError:
                if attempt == 5: raise
                time.sleep(.05*(attempt+1))

    if os.path.exists(summary_path):
        with io.open(summary_path, 'r', newline='', encoding='utf-8') as handle:
            for row in csv.DictReader(handle):
                for name in ('point_id','count','ci','cq'): row[name] = int(row[name])
                for name in ('phase_deg','actual_phase_deg','timestamp_epoch_s','q','q_uncertainty',
                             'q_ratio','f0_hz','f0_uncertainty_hz','linewidth_hz','linewidth_uncertainty_hz',
                             'shift_hz','shift_uncertainty_hz','k','k_uncertainty','delta_bic','background_ratio','carrier_fraction'):
                    row[name] = float(row[name])
                row['accepted'] = row['accepted'].lower() == 'true'; row['repeat'] = row['repeat'].lower() == 'true'
                points.append(row); completed[row['point_id']]=True

    result = {'stage': stage, 'config': stored_config, 'campaign_directory': directory,
              'points': points, 'baselines': [], 'status': 'running',
              'estimated_minutes': estimated_seconds/60., 'out2_used': False}
    baseline_identity_checked=False
    try:
        full_shutdown()
        for count in sorted(dict.fromkeys(row[0] for row in schedule)):
            relevant = [(index, row) for index, row in enumerate(schedule) if row[0] == count]
            pending = [(index, row) for index, row in relevant if index not in completed]
            if not pending: continue
            before = baseline_call('%s count %d before' % (stage, count)); before_fit = accepted_fit(before, 'before')
            before_time=time.time()
            signature={'f0_hz':float(before_fit['f0_hz']),'q':float(before_fit['q']),
                       'linewidth_hz':float(before_fit['linewidth_hz'])}
            if manifest.get('initial_baseline') is None:
                manifest['initial_baseline']=signature; atomic_json(manifest_path,manifest)
            elif resume and not baseline_identity_checked:
                old=manifest['initial_baseline']
                frequency_change=abs(signature['f0_hz']-float(old['f0_hz']))
                q_fraction=abs(signature['q']/max(float(old['q']),1e-30)-1.)
                if frequency_change > max(signature['linewidth_hz'],float(old['linewidth_hz'])) or q_fraction > .30:
                    raise RuntimeError('Resume refused after a fresh feedback-off measurement: the resonance changed by %.3f linewidths and %.1f%% in Q. Start a new campaign so incompatible hardware or resonator states are not mixed.' %
                                       (frequency_change/max(signature['linewidth_hz'],float(old['linewidth_hz'])),100*q_fraction))
                baseline_identity_checked=True
            result['baselines'].append({'position':'before','count':count,'fit':compact_fit(before_fit),'timestamp_epoch_s':before_time})
            rung_points = []
            for point_id, (_, phase, is_repeat) in pending:
                # Use nearest accepted coarse point only to centre/resolve; never as map interpolation.
                prediction = before_fit
                source_points = coarse_map.get('points', []) if isinstance(coarse_map, dict) else points
                candidates = [p for p in source_points if p.get('accepted') and p.get('q')]
                if candidates:
                    near = min(candidates, key=lambda p: abs(p['count']-count)/max(1., count)+
                               abs((p['phase_deg']-phase+180.)%360.-180.)/180.)
                    prediction = {'f0_hz': near['f0_hz'], 'q': near['q']}
                measurement = point_call(count, phase, before, prediction, averages,
                                         '%s point %04d' % (stage, point_id))
                try:
                    on = accepted_fit(measurement, 'on')
                except RuntimeError as first_error:
                    print('Fine window did not produce an accepted fit at count %d phase %.1f deg. Running the one permitted broad bounded locator before retrying the fine measurement.' % (count,phase))
                    locator=point_call(count,phase,before,before_fit,max(4,cfg['coarse_averages']),
                                       '%s point %04d broad retry' % (stage,point_id),force_broad=True)
                    located=accepted_fit(locator,'on')
                    measurement=point_call(count,phase,before,located,averages,
                                           '%s point %04d fine retry' % (stage,point_id))
                    try: on=accepted_fit(measurement,'on')
                    except RuntimeError:
                        raise RuntimeError('Resonance location failed after one broad retry. Initial failure was: %s' % str(first_error))
                ci, cq, actual_phase = vector(count, phase)
                spectrum = measurement['spectra']['on']
                psd = np.asarray(spectrum['psd_v2_per_hz'], float)
                carrier_fraction = float(np.max(psd)/np.sum(psd))
                base_spectrum = before['spectra']['before']
                base_frequency = np.asarray(base_spectrum['frequency_hz'], float)
                base_psd = np.asarray(base_spectrum['psd_v2_per_hz'], float)
                baseline_here = np.interp(np.asarray(spectrum['frequency_hz'], float), base_frequency, base_psd)
                ratio_bins = psd/np.maximum(baseline_here, np.finfo(float).tiny)
                # Median is deliberately insensitive to the central resonance;
                # a broad increase across the acquisition window is not.
                background_ratio = float(np.median(ratio_bins))
                width_ratio = on['linewidth_hz']/before_fit['linewidth_hz']
                k = 1.-width_ratio
                k_unc = float(width_ratio*np.hypot(on['linewidth_uncertainty_hz']/on['linewidth_hz'],
                                                   before_fit['linewidth_uncertainty_hz']/before_fit['linewidth_hz']))
                shift = on['f0_hz']-before_fit['f0_hz']
                shift_unc = float(np.hypot(on['f0_uncertainty_hz'], before_fit['f0_uncertainty_hz']))
                reasons = []
                if carrier_fraction > float(cfg['maximum_carrier_fraction']): reasons.append('narrow carrier fraction exceeded')
                if background_ratio > float(cfg['maximum_background_ratio']): reasons.append('broadband background increased')
                if k >= cfg['maximum_k'] or k+2*k_unc >= cfg['maximum_k']: reasons.append('K stop/uncertainty margin reached')
                if is_repeat:
                    originals=[r for r in points if r.get('accepted') and not r.get('repeat') and
                               r['count']==count and r['phase_deg']==phase]
                    if originals:
                        original=originals[-1]
                        q_limit=3*np.hypot(on['q_uncertainty'],original['q_uncertainty'])
                        f_limit=3*np.hypot(on['f0_uncertainty_hz'],original['f0_uncertainty_hz'])
                        if abs(on['q']-original['q'])>q_limit or abs(on['f0_hz']-original['f0_hz'])>f_limit:
                            reasons.append('repeat disagreed beyond combined 3-sigma fit uncertainty')
                row = {'point_id':point_id,'stage':stage,'count':count,'phase_deg':phase,
                       'actual_phase_deg':actual_phase,'ci':ci,'cq':cq,'repeat':is_repeat,
                       'timestamp_epoch_s':time.time(),'accepted':not reasons,'q':on['q'],
                       'q_uncertainty':on['q_uncertainty'],'q_ratio':on['q']/before_fit['q'],
                       'f0_hz':on['f0_hz'],'f0_uncertainty_hz':on['f0_uncertainty_hz'],
                       'linewidth_hz':on['linewidth_hz'],'linewidth_uncertainty_hz':on['linewidth_uncertainty_hz'],
                       'shift_hz':shift,'shift_uncertainty_hz':shift_unc,'k':k,'k_uncertainty':k_unc,
                       'delta_bic':float(on.get('delta_bic',np.nan)),
                       'background_ratio':background_ratio,'carrier_fraction':carrier_fraction,
                       'reason':'; '.join(reasons)}
                points.append(row); rung_points.append(row); completed[point_id] = True
                manifest['completed_ids'] = sorted(completed); manifest['status'] = 'running'; save_summary(); atomic_json(manifest_path, manifest)
                print('saved %04d: count %d phase %.1f deg, Q %.1f, shift %+.1f Hz, K %.4f [%s]' %
                      (point_id, count, phase, on['q'], shift, k, 'accepted' if row['accepted'] else row['reason']))
                if reasons:
                    manifest['status']='stopped'; manifest['stop_reason']=row['reason']; atomic_json(manifest_path, manifest)
                    raise RuntimeError('MAP STOPPED SAFELY at count %d phase %.1f deg: %s. Partial data were saved; resume is never automatic.' %
                                       (count, phase, row['reason']))
                del measurement,spectrum,psd,on,baseline_here,ratio_bins
                gc.collect()
            after = baseline_call('%s count %d after' % (stage, count)); after_fit = accepted_fit(after, 'before')
            after_time=time.time()
            result['baselines'].append({'position':'after','count':count,'fit':compact_fit(after_fit),'timestamp_epoch_s':after_time})
            # Replace the provisional before-reference comparison with the
            # time-interpolated off baseline. The summary is authoritative;
            # spectra remain unchanged in their per-point NPZ files.
            for row in rung_points:
                weight=np.clip((row['timestamp_epoch_s']-before_time)/max(after_time-before_time,1e-9),0,1)
                base={}
                for name in ('q','f0_hz','linewidth_hz'):
                    base[name]=(1-weight)*before_fit[name]+weight*after_fit[name]
                base_q_unc=np.hypot((1-weight)*before_fit['q_uncertainty'],weight*after_fit['q_uncertainty'])
                base_f_unc=np.hypot((1-weight)*before_fit['f0_uncertainty_hz'],weight*after_fit['f0_uncertainty_hz'])
                base_w_unc=np.hypot((1-weight)*before_fit['linewidth_uncertainty_hz'],weight*after_fit['linewidth_uncertainty_hz'])
                row['q_ratio']=row['q']/base['q']; row['shift_hz']=row['f0_hz']-base['f0_hz']; row['linewidth_hz']=float(row['linewidth_hz'])
                ratio=row['linewidth_hz']/base['linewidth_hz']; row['k']=1-ratio
                row['shift_uncertainty_hz']=float(np.hypot(row['f0_uncertainty_hz'],base_f_unc))
                row['k_uncertainty']=float(ratio*np.hypot(row['linewidth_uncertainty_hz']/row['linewidth_hz'],base_w_unc/base['linewidth_hz']))
                row['baseline_q_uncertainty']=base_q_unc; row['baseline_interpolation_weight']=float(weight)
                if row['k']>=cfg['maximum_k'] or row['k']+2*row['k_uncertainty']>=cfg['maximum_k']:
                    row['accepted']=False; row['reason']=(row['reason']+'; ' if row['reason'] else '')+'time-interpolated K stop reached'
            save_summary()
            if any('time-interpolated K stop reached' in row['reason'] for row in rung_points):
                manifest['status']='stopped';manifest['stop_reason']='time-interpolated K stop reached';atomic_json(manifest_path,manifest)
                raise RuntimeError('MAP STOPPED SAFELY after count %d: time-interpolated K reached its uncertainty-aware limit. Partial data were saved.' % count)
            shift_recovery = abs(after_fit['f0_hz']-before_fit['f0_hz'])
            width_recovery = abs(after_fit['linewidth_hz']-before_fit['linewidth_hz'])
            shift_limit = max(3*np.hypot(after_fit['f0_uncertainty_hz'],before_fit['f0_uncertainty_hz']), before_fit['linewidth_hz'])
            width_limit = max(3*np.hypot(after_fit['linewidth_uncertainty_hz'],before_fit['linewidth_uncertainty_hz']), .3*before_fit['linewidth_hz'])
            if shift_recovery > shift_limit or width_recovery > width_limit:
                for row in rung_points:
                    row['accepted']=False; row['reason']=(row['reason']+'; ' if row['reason'] else '')+'feedback-off recovery failed'
                save_summary()
                manifest['status']='stopped'; manifest['stop_reason']='feedback-off baseline failed recovery'; atomic_json(manifest_path,manifest)
                raise RuntimeError('MAP STOPPED SAFELY: feedback-off baseline failed recovery after count %d. Partial data were saved.' % count)
            del before,after,before_fit,after_fit,rung_points
            gc.collect()
        result['status']='complete'; manifest['status']='complete'; manifest['stop_reason']=None
        result.update(_feedback_map_analyse(result));
        inconsistent=[axis for axis in result['phase_axes'] if not axis['first_harmonic_consistent']]
        if inconsistent:
            manifest['status']='stopped';manifest['stop_reason']='phase response failed first-harmonic consistency'
            atomic_json(manifest_path,manifest);save_summary()
            raise RuntimeError('MAP STOPPED: phase response is not repeatably sinusoidal at count(s) %s. Partial data are saved.' %
                               ', '.join(str(v['count']) for v in inconsistent))
        atomic_json(manifest_path,manifest); save_summary()
        return result
    except BaseException as exc:
        result['status']='stopped'
        manifest['status']='stopped'
        if not manifest.get('stop_reason'):
            manifest['stop_reason']='%s: %s' % (type(exc).__name__,str(exc))
        try: atomic_json(manifest_path,manifest)
        except Exception: pass
        raise
    finally:
        full_shutdown()


def _feedback_map_load(directory):
    """Internal compact loader used by plotting, selection and explicit resume."""
    import os, io, csv, json
    manifest_path = os.path.join(os.path.abspath(str(directory)), 'manifest.json')
    with io.open(manifest_path, 'r', encoding='utf-8') as handle: manifest=json.load(handle)
    points=[]
    with io.open(os.path.join(os.path.dirname(manifest_path),'summary.csv'),'r',newline='',encoding='utf-8') as handle:
        for row in csv.DictReader(handle):
            for n in ('point_id','count','ci','cq'): row[n]=int(row[n])
            for n in ('phase_deg','actual_phase_deg','timestamp_epoch_s','q','q_uncertainty','q_ratio','f0_hz',
                      'f0_uncertainty_hz','linewidth_hz','linewidth_uncertainty_hz','shift_hz','shift_uncertainty_hz',
                      'k','k_uncertainty','delta_bic','background_ratio','carrier_fraction'): row[n]=float(row[n])
            row['accepted']=row['accepted'].lower()=='true'; row['repeat']=row['repeat'].lower()=='true'; points.append(row)
    result={'stage':manifest['stage'],'config':manifest['config'],'campaign_directory':os.path.dirname(manifest_path),
            'points':points,'status':manifest['status'],'out2_used':False}
    result.update(_feedback_map_analyse(result)); return result


def _feedback_map_analyse(mapping):
    """Fit phase axes and quantify repeat/leave-one-out consistency without extrapolation."""
    import numpy as np
    rows=[r for r in mapping.get('points',[]) if r.get('accepted') and not r.get('repeat')]
    axes=[]
    for count in sorted(dict.fromkeys(r['count'] for r in rows)):
        group=[r for r in rows if r['count']==count]
        if len(group)<6: continue
        phase=np.deg2rad([r['actual_phase_deg'] for r in group]); design=np.column_stack((np.ones(len(group)),np.cos(phase),np.sin(phase)))
        wk=1/np.maximum([r['k_uncertainty'] for r in group],1e-9); ws=1/np.maximum([r['shift_uncertainty_hz'] for r in group],1e-9)
        ck=np.linalg.lstsq(design*wk[:,None],np.array([r['k'] for r in group])*wk,rcond=-1)[0]
        cs=np.linalg.lstsq(design*ws[:,None],np.array([r['shift_hz'] for r in group])*ws,rcond=-1)[0]
        kr=np.sqrt(np.mean((design.dot(ck)-np.array([r['k'] for r in group]))**2))
        sr=np.sqrt(np.mean((design.dot(cs)-np.array([r['shift_hz'] for r in group]))**2))
        ktolerance=max(4*np.median([r['k_uncertainty'] for r in group]),.25*np.ptp([r['k'] for r in group]))
        stolerance=max(4*np.median([r['shift_uncertainty_hz'] for r in group]),.25*np.ptp([r['shift_hz'] for r in group]))
        regenerative=float(np.rad2deg(np.arctan2(ck[2],ck[1]))%360.)
        positive=float(np.rad2deg(np.arctan2(cs[2],cs[1]))%360.)
        axes.append({'count':count,'regenerative_deg':regenerative,'degenerative_deg':(regenerative+180)%360,
                     'positive_shift_deg':positive,'negative_shift_deg':(positive+180)%360,
                     'axis_nonorthogonality_deg':abs(abs((positive-regenerative+180)%360-180)-90),
                     'k_first_harmonic_rms':float(kr),'shift_first_harmonic_rms_hz':float(sr),
                     'first_harmonic_consistent':bool(kr<=ktolerance and sr<=stolerance)})
    repeats=[]
    for row in [r for r in mapping.get('points',[]) if r.get('accepted') and r.get('repeat')]:
        candidates=[r for r in rows if r['count']==row['count'] and r['phase_deg']==row['phase_deg']]
        if candidates: repeats.append({'count':row['count'],'phase_deg':row['phase_deg'],
            'q_fractional_difference':abs(row['q']/candidates[0]['q']-1),
            'shift_difference_hz':abs(row['shift_hz']-candidates[0]['shift_hz'])})
    # Leave-one-out local interpolation is an honest check: boundary points for
    # which removal requires extrapolation are reported as untestable.
    validation=[]
    if len(rows)>=10:
        from scipy.interpolate import LinearNDInterpolator
        grouped={}
        for row in rows: grouped.setdefault((row['ci'],row['cq']),[]).append(row)
        groups=list(grouped.values())
        xy=np.array([[g[0]['ci'],g[0]['cq']] for g in groups],float)
        qvalues=np.array([np.mean([r['q_ratio'] for r in g]) for g in groups])
        svalues=np.array([np.mean([r['shift_hz'] for r in g]) for g in groups])
        qrange=max(float(np.ptp(qvalues)),.02); srange=max(float(np.ptp(svalues)),1.)
        for index,group in enumerate(groups):
            row=group[0]; keep=np.arange(len(groups))!=index
            try:
                qp=float(np.asarray(LinearNDInterpolator(xy[keep],qvalues[keep])(xy[index])).reshape(-1)[0])
                sp=float(np.asarray(LinearNDInterpolator(xy[keep],svalues[keep])(xy[index])).reshape(-1)[0])
            except Exception:
                qp=sp=np.nan
            qu=np.mean([r['q_uncertainty']/max(r['q']/r['q_ratio'],1e-30) for r in group])
            su=np.mean([r['shift_uncertainty_hz'] for r in group])
            testable=bool(np.isfinite(qp+sp))
            passed=bool(testable and abs(qp-qvalues[index])<=max(3*qu,.15*qrange) and
                        abs(sp-svalues[index])<=max(3*su,.15*srange))
            validation.append({'point_id':row['point_id'],'testable':bool(np.isfinite(qp+sp)),
                'accepted':passed,
                'q_ratio_error':None if not testable else float(qp-qvalues[index]),
                'shift_error_hz':None if not testable else float(sp-svalues[index])})
    tested=[v for v in validation if v['testable']]
    return {'phase_axes':axes,'repeatability':repeats,'interpolation_validation':validation,
            'interpolation_validation_accepted':bool(tested and all(v['accepted'] for v in tested)),
            'electrode_voltage_directly_measured':False,'interpolation_extrapolation_allowed':False}


def plot_feedback_map(mapping, output_prefix=None):
    """Render the requested heatmaps/phase cuts; rejected and extrapolated areas stay black."""
    import os, numpy as np, matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from scipy.interpolate import LinearNDInterpolator
    if isinstance(mapping,str): mapping=_feedback_map_load(mapping)
    rows=[r for r in mapping.get('points',[]) if r.get('accepted') and not r.get('repeat')]
    if len(rows)<6: raise ValueError('At least six accepted non-repeat map points are required.')
    grouped={}
    for row in rows: grouped.setdefault((row['ci'],row['cq']),[]).append(row)
    groups=list(grouped.values())
    xy=np.array([[group[0]['ci'],group[0]['cq']] for group in groups],float)
    phase=np.linspace(0,360,181); gains=np.linspace(0,max(r['count'] for r in rows),121)
    pp,gg=np.meshgrid(phase,gains)
    commands=[({'ci_signed':0,'cq_signed':0} if int(round(g))==0 else
              quantized_feedback_vector(int(round(g)),float(p),
              float(mapping['config'].get('phase_reference_hz',CENTRE_HZ))))
              for g,p in zip(gg.ravel(),pp.ravel())]
    query=np.array([[v['ci_signed'],v['cq_signed']] for v in commands],float)
    def grid(name): return np.asarray(LinearNDInterpolator(xy,np.array([np.mean([r[name] for r in group]) for group in groups]))(query)).reshape(gg.shape)
    qgrid,kgrid,sgrid=grid('q_ratio'),grid('k'),grid('shift_hz')
    # Deliberately cut holes around rejected commands. A local interpolator must
    # never bridge across a point at which the hardware or fit guard fired.
    rejected=[r for r in mapping.get('points',[]) if not r.get('accepted')]
    if rejected:
        for row in rejected:
            distance=np.sqrt((query[:,0]-row['ci'])**2+(query[:,1]-row['cq'])**2)
            unsafe=(distance<=max(2.,.02*max(gains))).reshape(gg.shape)
            qgrid[unsafe]=np.nan;kgrid[unsafe]=np.nan;sgrid[unsafe]=np.nan
    try: cmap=CMAP_SEQ
    except NameError:
        try: cmap=plt.get_cmap('viridis')
        except ValueError: cmap=plt.get_cmap('jet')
    cmap.set_bad('black')
    fig=plt.figure(figsize=(12,8)); outer=gridspec.GridSpec(2,2,height_ratios=[2.2,1.0],hspace=.12,wspace=.25)
    axes=[fig.add_subplot(outer[0,0]),fig.add_subplot(outer[0,1]),fig.add_subplot(outer[1,0]),fig.add_subplot(outer[1,1])]
    images=[axes[0].imshow(np.ma.masked_invalid(qgrid),cmap=cmap,origin='lower',aspect='auto',extent=(0,360,0,max(gains))),
            axes[1].imshow(np.ma.masked_invalid(sgrid),cmap=cmap,origin='lower',aspect='auto',extent=(0,360,0,max(gains)))]
    for ax,image,label in ((axes[0],images[0],'Q / Q0'),(axes[1],images[1],'Shift (Hz)')):
        box=ax.get_position(); cax=fig.add_axes([box.x0,box.y1+.012,box.width,.018])
        bar=fig.colorbar(image,cax=cax,orientation='horizontal');bar.set_label(label);cax.xaxis.set_ticks_position('top');cax.xaxis.set_label_position('top')
        ax.axvline(225,color='#ff7f0e',linestyle='--',linewidth=1);ax.set_ylabel('Coefficient count')
    for level,color in ((.5,'white'),(.7,'#ffbb33'),(.85,'#d62728')):
        if np.nanmin(kgrid)<=level<=np.nanmax(kgrid): axes[0].contour(phase,gains,kgrid,levels=[level],colors=[color],linestyles='--',linewidths=1)
    if np.nanmin(sgrid)<=0<=np.nanmax(sgrid): axes[1].contour(phase,gains,sgrid,levels=[0],colors=['white'],linestyles='--',linewidths=1)
    selected=np.unique(np.rint(np.percentile([r['count'] for r in rows],[20,55,100])).astype(int))
    colors=('#4c9be8','#ef6c55','#f5b642')
    for count,color in zip(selected,colors):
        near=[r for r in rows if r['count']==min(dict.fromkeys(v['count'] for v in rows),key=lambda v:abs(v-count))]
        near=sorted(near,key=lambda r:r['actual_phase_deg'])
        x=np.array([r['actual_phase_deg'] for r in near]); q=np.array([r['q_ratio'] for r in near]); shift=np.array([r['shift_hz'] for r in near])
        qu=np.array([r['q_uncertainty']/max(r['q']/r['q_ratio'],1e-30) for r in near]); su=np.array([r['shift_uncertainty_hz'] for r in near])
        axes[2].plot(x,q,'-o',color=color,label='count %d'%near[0]['count']);axes[2].fill_between(x,q-qu,q+qu,color=color,alpha=.15)
        axes[3].plot(x,shift,'-o',color=color,label='count %d'%near[0]['count']);axes[3].fill_between(x,shift-su,shift+su,color=color,alpha=.15)
    for ax in axes[2:]: ax.set_xlabel('Phase (degrees)');ax.set_xlim(0,360);ax.legend(fontsize=8);ax.grid(True,alpha=.25)
    axes[2].set_ylabel('Q / Q0');axes[3].set_ylabel('Shift (Hz)');fig.suptitle('Passive feedback gain-phase map')
    if output_prefix is None: output_prefix=os.path.join(mapping['campaign_directory'],'feedback_map')
    fig.savefig(output_prefix+'.png',dpi=180,bbox_inches='tight');fig.savefig(output_prefix+'.pdf',bbox_inches='tight')
    axis_figure=None;axis_png=None;axis_pdf=None
    phase_axes=mapping.get('phase_axes',[])
    if phase_axes:
        axis_figure,axis_plot=plt.subplots(2,1,figsize=(8,6),sharex=True)
        counts=np.array([r['count'] for r in phase_axes],float)
        for field,label,color in (('regenerative_deg','regenerative','#f5b642'),
                                  ('degenerative_deg','degenerative','#4c9be8'),
                                  ('positive_shift_deg','positive shift','#ef6c55'),
                                  ('negative_shift_deg','negative shift','#54a96b')):
            axis_plot[0].plot(counts,[r[field] for r in phase_axes],'-o',label=label,color=color)
        axis_plot[0].axhline(225,color='#ff7f0e',linestyle='--',label='225 degree seed')
        axis_plot[0].set_ylabel('Empirical phase axis (degrees)');axis_plot[0].set_ylim(0,360)
        axis_plot[0].legend(fontsize=8,ncol=2);axis_plot[0].grid(True,alpha=.25)
        axis_plot[1].plot(counts,[r['axis_nonorthogonality_deg'] for r in phase_axes],'-o',color='#7b5ea7')
        axis_plot[1].axhline(0,color='black',linewidth=.8);axis_plot[1].set_xlabel('Coefficient count')
        axis_plot[1].set_ylabel('Axis non-orthogonality (degrees)');axis_plot[1].grid(True,alpha=.25)
        axis_figure.suptitle('Measured feedback axes versus gain')
        axis_png=output_prefix+'_axes.png';axis_pdf=output_prefix+'_axes.pdf'
        axis_figure.savefig(axis_png,dpi=180,bbox_inches='tight');axis_figure.savefig(axis_pdf,bbox_inches='tight')
    plt.show()
    return {'figure':fig,'png':output_prefix+'.png','pdf':output_prefix+'.pdf',
            'axis_figure':axis_figure,'axis_png':axis_png,'axis_pdf':axis_pdf}


def select_feedback_operating_point(mapping, target_q=None, q_ratio=None,
                                    target_frequency_hz=None, shift_hz=None,
                                    maximum_candidates=5):
    """Return measured-envelope candidates; never apply feedback or extrapolate."""
    import math, numpy as np
    from scipy.interpolate import LinearNDInterpolator
    if isinstance(mapping,str): mapping=_feedback_map_load(mapping)
    if mapping.get('interpolation_validation') and not mapping.get('interpolation_validation_accepted'):
        raise ValueError('Operating-point selection is blocked because withheld measured points were not reproduced within the combined uncertainty. Inspect the map or acquire more local points.')
    if sum(v is not None for v in (target_q,q_ratio))>1 or sum(v is not None for v in (target_frequency_hz,shift_hz))>1:
        raise ValueError('Choose at most one Q target and at most one frequency target.')
    if all(v is None for v in (target_q,q_ratio,target_frequency_hz,shift_hz)):
        raise ValueError('Provide a Q and/or frequency target; this function never chooses the largest gain automatically.')
    rows=[r for r in mapping.get('points',[]) if r.get('accepted') and not r.get('repeat')]
    if len(rows)<6: raise ValueError('Too few accepted points for bounded interpolation.')
    q0=np.median([r['q']/r['q_ratio'] for r in rows]); f0=np.median([r['f0_hz']-r['shift_hz'] for r in rows])
    tq=float(target_q)/q0 if target_q is not None else (float(q_ratio) if q_ratio is not None else 1.)
    ts=float(target_frequency_hz)-f0 if target_frequency_hz is not None else (float(shift_hz) if shift_hz is not None else 0.)
    if (target_q is not None or q_ratio is not None) and not min(r['q_ratio'] for r in rows)<=tq<=max(r['q_ratio'] for r in rows):
        raise ValueError('Requested Q lies outside the measured Q/Q0 range; extrapolation is prohibited.')
    if (target_frequency_hz is not None or shift_hz is not None) and not min(r['shift_hz'] for r in rows)<=ts<=max(r['shift_hz'] for r in rows):
        raise ValueError('Requested frequency shift lies outside the measured shift range; extrapolation is prohibited.')
    grouped={}
    for row in rows: grouped.setdefault((row['ci'],row['cq']),[]).append(row)
    groups=list(grouped.values())
    xy=np.array([[g[0]['ci'],g[0]['cq']] for g in groups],float)
    qv=np.array([np.mean([r['q_ratio'] for r in g]) for g in groups]); sv=np.array([np.mean([r['shift_hz'] for r in g]) for g in groups])
    quv=np.array([np.mean([r['q_uncertainty']/max(r['q']/r['q_ratio'],1e-30) for r in g]) for g in groups])
    suv=np.array([np.mean([r['shift_uncertainty_hz'] for r in g]) for g in groups])
    kv=np.array([np.mean([r['k'] for r in g]) for g in groups]); kuv=np.array([np.mean([r['k_uncertainty'] for r in g]) for g in groups])
    iq,isf=LinearNDInterpolator(xy,qv),LinearNDInterpolator(xy,sv)
    iqu,isu=LinearNDInterpolator(xy,quv),LinearNDInterpolator(xy,suv)
    ik,iku=LinearNDInterpolator(xy,kv),LinearNDInterpolator(xy,kuv)
    limit=max(r['count'] for r in rows); phase=np.linspace(0,360,361)
    count_step=max(1,int(math.ceil(limit/256.)))
    gain=range(0,int(limit)+1,count_step)
    # Evaluate in strips to avoid a large temporary mesh on the Red Pitaya.
    candidates=[]; qscale=max(.02,np.std(qv)); sscale=max(1.,np.std(sv))
    for count in gain:
        commands=[({'ci_signed':0,'cq_signed':0} if int(round(count))==0 else
                  quantized_feedback_vector(int(round(count)),float(p),
                  float(mapping['config'].get('phase_reference_hz',CENTRE_HZ)))) for p in phase]
        query=np.array([[v['ci_signed'],v['cq_signed']] for v in commands],float)
        qp,sp=np.asarray(iq(query)).reshape(-1),np.asarray(isf(query)).reshape(-1)
        qup,sup=np.asarray(iqu(query)).reshape(-1),np.asarray(isu(query)).reshape(-1)
        kp,kup=np.asarray(ik(query)).reshape(-1),np.asarray(iku(query)).reshape(-1)
        valid=np.isfinite(qp)&np.isfinite(sp)&np.isfinite(qup)&np.isfinite(sup)&np.isfinite(kp)&np.isfinite(kup)
        valid &= (kp+2*kup<float(mapping['config']['maximum_k']))
        local=[]
        for index in np.where(valid)[0]:
            distance=float(np.min(np.sqrt(np.sum((xy-query[index])**2,axis=1))))
            margin=max(float(mapping['config']['maximum_k'])-kp[index]-2*kup[index],1e-9)
            score=(((qp[index]-tq)/qscale)**2+((sp[index]-ts)/sscale)**2+
                   .05*(qup[index]/qscale)**2+.05*(sup[index]/sscale)**2+
                   .02*(count/limit)**2+.01*(distance/max(1.,limit))**2+
                   .01*(1.-margin/float(mapping['config']['maximum_k']))**2)
            local.append((score,count,phase[index],qp[index],sp[index],qup[index],sup[index],kp[index],kup[index],distance))
        candidates.extend(sorted(local)[:3])
    candidates.sort(); selected=[]
    # Refine the best coarse neighbourhoods to integer count and quarter-degree
    # resolution without allocating/evaluating a million-point array.
    refined=[]
    for _,coarse_count,coarse_phase,_,_,_,_,_,_,_ in candidates[:20]:
        for count in range(max(0,int(coarse_count)-count_step),min(int(limit),int(coarse_count)+count_step)+1):
            phases=np.arange(coarse_phase-1.,coarse_phase+1.001,.25)%360.
            commands=[({'ci_signed':0,'cq_signed':0} if count==0 else
                      quantized_feedback_vector(count,float(p),float(mapping['config'].get('phase_reference_hz',CENTRE_HZ)))) for p in phases]
            query=np.array([[v['ci_signed'],v['cq_signed']] for v in commands],float)
            qp,sp=np.asarray(iq(query)).reshape(-1),np.asarray(isf(query)).reshape(-1)
            qup,sup=np.asarray(iqu(query)).reshape(-1),np.asarray(isu(query)).reshape(-1)
            kp,kup=np.asarray(ik(query)).reshape(-1),np.asarray(iku(query)).reshape(-1)
            valid=np.isfinite(qp)&np.isfinite(sp)&np.isfinite(qup)&np.isfinite(sup)&np.isfinite(kp)&np.isfinite(kup)
            valid &= (kp+2*kup<float(mapping['config']['maximum_k']))
            for index in np.where(valid)[0]:
                distance=float(np.min(np.sqrt(np.sum((xy-query[index])**2,axis=1))))
                margin=max(float(mapping['config']['maximum_k'])-kp[index]-2*kup[index],1e-9)
                score=(((qp[index]-tq)/qscale)**2+((sp[index]-ts)/sscale)**2+
                       .05*(qup[index]/qscale)**2+.05*(sup[index]/sscale)**2+
                       .02*(count/limit)**2+.01*(distance/max(1.,limit))**2+
                       .01*(1.-margin/float(mapping['config']['maximum_k']))**2)
                refined.append((score,count,float(phases[index]),qp[index],sp[index],qup[index],sup[index],kp[index],kup[index],distance))
    candidates=sorted(candidates+refined)
    for score,count,phase_value,qp,sp,qup,sup,kp,kup,distance in candidates:
        if any(abs(count-r['coefficient_counts'])<2 and abs((phase_value-r['phase_deg']+180)%360-180)<2 for r in selected): continue
        selected.append({'coefficient_counts':int(round(count)),'gain':float(count/16384.),'phase_deg':float(phase_value),
                         'predicted_q':float(qp*q0),'predicted_q_ratio':float(qp),
                         'predicted_q_uncertainty':float(qup*q0),
                         'predicted_frequency_hz':float(f0+sp),'predicted_shift_hz':float(sp),
                         'predicted_shift_uncertainty_hz':float(sup),'predicted_k':float(kp),
                         'predicted_k_uncertainty':float(kup),
                         'k_stability_margin':float(mapping['config']['maximum_k']-kp-2*kup),
                         'distance_to_nearest_measurement_counts':distance,'interpolated_only':True,
                         'extrapolated':False,'automatic_application_allowed':False,
                         'validation_required':'Run one manual feedback measurement and verify shutdown/recovery.'})
        if len(selected)>=int(maximum_candidates): break
    if not selected: raise ValueError('Requested target has no solution inside the accepted measured interpolation envelope.')
    return selected


## 23 — Configure the campaign

Run the configuration cell and edit the campaign name before a new physical setup.
The hard maximum is 1024 counts and the uncertainty-aware stop is K=0.85.
The configuration describes software measurement choices, not unknown electrode
voltage or analogue gain. The three frequency controls below are deliberately
editable: the baseline and locator ranges are absolute bounds, while the feedback
minimum/maximum spans bound the Q-adapted window around each predicted centre.

For coarse Q-reducing points, 0.35 linewidth of clean data at the nearer fit edge is
accepted instead of requiring a full linewidth. This is not extrapolation: the fitted
centre must remain inside the window and resolution, uncertainty, spur and safety
checks remain active. Increase the locator range or averages if uncertainty dominates.

The printed estimate is repeated before hardware starts. Existing campaign
directories are never overwritten. Use a new name after changing attenuation,
amplification, cabling, filter correction or trap conditions.


In [ ]:
MAP_CONFIG = {
    'campaign_name': 'current_feedback_map_compact_v3',
    'minimum_coefficient_counts': 128,
    'maximum_coefficient_counts': 1024,
    'maximum_k': 0.85,
    'phase_seed_deg': 225.0,
    'profile': 'fast',
    'coarse_averages': 6,
    'final_averages': 16,
    'time_budget_minutes': 55,
    'campaign_root': 'feedback_maps',
    'baseline_frequency_range_hz': (40.15e6, 41.65e6),
    'baseline_fit_range_hz': (40.20e6, 41.60e6),
    'locator_frequency_range_hz': (40.15e6, 41.65e6),
    'feedback_minimum_span_hz': 180e3,
    'feedback_maximum_span_hz': 1.50e6,
    'feedback_linewidths_full_span': 4.0,
    'map_minimum_edge_coverage_linewidths': 0.35,
    'coarse_maximum_fit_uncertainty_linewidths': 0.60,
    'final_maximum_fit_uncertainty_linewidths': 0.35,
    'baseline_response_calibration': globals().get('BASELINE_RESPONSE'),
    'feedback_response_calibration': globals().get('FEEDBACK_RESPONSE'),
}
print('Map configured: count range %d..%d; K stop %.2f; expected phase seed %.1f degrees.' %
      (MAP_CONFIG['minimum_coefficient_counts'], MAP_CONFIG['maximum_coefficient_counts'],
       MAP_CONFIG['maximum_k'], MAP_CONFIG['phase_seed_deg']))
print('No hardware was accessed. Change campaign_name after any physical setup change.')


## 24 — Run or resume the coarse diagnostic map

Before uncommenting the call, restore the passive trap wiring, remove the OUT1–IN1
filter-calibration loopback, and verify the external physical stop for OUT1.
Run the broad manual baseline successfully first if you want a visual check.

The coarse sweep starts at 128 counts by default, advances in ascending rungs, measures
opposite phases closely in time and brackets every rung with feedback-off baselines.
It uses full phase circles at selected gains and extra points around 225°.

Every point returns to full shutdown before fitting and is written atomically.
Any K limit, carrier, broadband, fit, repeatability or recovery failure stops the
whole campaign with partial data preserved. A stopped campaign never resumes by
itself. Use `resume=True` only after investigating the printed reason.


In [ ]:
# LIVE HARDWARE — explicitly uncomment one call only after reading the instructions.
# COARSE_MAP = run_feedback_map('coarse', MAP_CONFIG)

# After investigating a stopped run, resume without reacquiring saved point IDs:
# COARSE_MAP = run_feedback_map('coarse', MAP_CONFIG, resume=True)

# Emergency software stop: full_shutdown()


## 25 — Run the final map and produce the figure

Only run this after reviewing the coarse axes and accepted points. The final grid
chooses about nine gain rungs distributed through measured K, uses 15° phase steps,
and repeats ten percent of points. Each active spectrum is centred and resolved
from the nearest accepted coarse result.

The plotting call creates PNG and PDF output beside the campaign data. Its upper
panels are Q/Q0 and frequency-shift heatmaps; lower panels show measured phase
cuts at representative gains. Rejected and extrapolated areas stay black.
Contours show K=0.5, 0.7 and 0.85, and the orange vertical line marks the original
225° expectation. Interpolation uses actual quantized CI/CQ coordinates.


In [ ]:
# LIVE HARDWARE — run only after reviewing a completed coarse result.
# FINAL_MAP = run_feedback_map('final', MAP_CONFIG, coarse_map=COARSE_MAP)

# Resume is always explicit:
# FINAL_MAP = run_feedback_map('final', MAP_CONFIG, coarse_map=COARSE_MAP, resume=True)

# Plot after completion or from any sufficiently populated partial result:
# MAP_FIGURE = plot_feedback_map(FINAL_MAP)
# MAP_FIGURE = plot_feedback_map('feedback_maps/current_feedback_map_compact_v3_final')


## 26 — Select a future operating point

Call `select_feedback_operating_point` with a Q target, frequency target, or both.
It searches only inside the accepted triangulated map and returns several candidates.
It never applies feedback. If Q alone is supplied it favours low residual shift;
if shift alone is supplied it favours little Q change.

Each candidate reports the predicted Q/frequency, quantized count and phase,
distance to measured data, and the requirement for a fresh manual validation.
Do not treat interpolation as permission to approach K=1. If no accepted local
solution exists, take additional measurements rather than enabling extrapolation.


In [ ]:
# PURE ANALYSIS — examples; this cell never applies feedback.
# candidates = select_feedback_operating_point(FINAL_MAP, target_q=1000.0)
# candidates = select_feedback_operating_point(FINAL_MAP, shift_hz=1000.0)
# candidates = select_feedback_operating_point(
#     'feedback_maps/current_feedback_map_compact_v3_final', target_q=1000.0, shift_hz=500.0)
# for candidate in candidates:
#     print(candidate)


## 27 — Define a continuously held feedback setting

**Purpose.** Defines an explicit enable/disable pair for an external measurement. The definition cell does not touch hardware.

nable_held_feedback() performs a fresh raw-IN1 headroom check, verifies the FPGA identity, enforces both the requested ceiling and an unraiseable 1024-count ceiling, writes the phase-corrected coefficients, and verifies their readback. OUT2 and both generators remain off.

The FPGA holds the coefficients after the function returns. There is no timeout or watchdog: a kernel crash cannot call the disable function. Keep a physical OUT1 disconnect accessible and do not leave the setup unattended. **Safe to rerun:** only while feedback is disabled.


In [ ]:
"""Explicit enable/disable pair for externally observed continuous feedback."""

HELD_FEEDBACK_HARD_CEILING_COUNTS = 1024
if 'HELD_FEEDBACK_STATE' not in globals():
    HELD_FEEDBACK_STATE = {'enabled': False, 'last_action': 'definition loaded'}


def enable_held_feedback(coefficient_counts, phase_deg,
                         reference_frequency_hz=40.9e6,
                         maximum_coefficient_counts=1024):
    """Latch one bounded gain/phase setting until explicitly disabled.

    The function returns after writing the registers. The FPGA continues using
    those coefficients until disable_held_feedback(), another program, a new
    bitstream, or a reboot changes them. There is deliberately no software loop
    or automatic timeout, so an external physical disconnect remains essential.
    """
    import time
    import numpy as np
    global HELD_FEEDBACK_STATE
    try:
        full_shutdown()
        count_value=float(coefficient_counts)
        ceiling_value=float(maximum_coefficient_counts)
        if (not np.isfinite(count_value) or count_value != int(count_value) or
                not np.isfinite(ceiling_value) or ceiling_value != int(ceiling_value)):
            raise ValueError('coefficient_counts and maximum_coefficient_counts must be finite integers. No feedback was enabled.')
        count=int(count_value); ceiling=int(ceiling_value)
        if not 1 <= ceiling <= HELD_FEEDBACK_HARD_CEILING_COUNTS:
            raise ValueError('maximum_coefficient_counts must be 1..%d. The hard ceiling cannot be raised by this function.' % HELD_FEEDBACK_HARD_CEILING_COUNTS)
        if not 1 <= count <= ceiling:
            raise ValueError('Requested coefficient_counts=%d is outside the enabled range 1..%d. Use disable_held_feedback() for zero.' % (count,ceiling))
        phase=float(phase_deg); reference=float(reference_frequency_hz)
        rf_limit=min(FS_CLK/2.,RP_LIMITS['input_bandwidth_hz'],RP_LIMITS['output_bandwidth_hz'])
        if not np.isfinite(phase) or not np.isfinite(reference) or not 0 < reference < rf_limit:
            raise ValueError('phase_deg must be finite and reference_frequency_hz must lie between zero and %.3f MHz.' % (rf_limit/1e6))
        actual_build='0x%08X' % acq.build_id()
        if actual_build.lower() != str(BUILD_ID_EXPECTED).lower():
            raise RuntimeError('Feedback was not enabled: FPGA identity %s does not match qualified build %s.' % (actual_build,BUILD_ID_EXPECTED))

        # This fresh raw-ADC capture includes DC and out-of-band pickup. It is a
        # headroom guard, not a measurement of the resonant tone amplitude.
        codes,_=sc.acquire(dec=1,n=4096,chan='A')
        voltage=np.asarray(codes,dtype=float)*float(ADC_RAW_VOLTS_PER_COUNT)
        if voltage.shape != (4096,) or not np.all(np.isfinite(voltage)):
            raise RuntimeError('Feedback was not enabled: the fresh IN1 safety capture was incomplete or non-finite.')
        input_peak_v=float(np.max(np.abs(voltage)))
        input_dc_v=float(np.mean(voltage))
        command=quantized_feedback_vector(count,phase,reference)
        output_bound_vpk=(2.*input_peak_v/RP_LIMITS['input_lv_full_scale_vpk']*
                          (abs(command['ci_signed'])+abs(command['cq_signed']))/16384.*
                          RP_LIMITS['output_full_scale_vpk_into_50ohm'])
        if input_peak_v >= ADC_OPERATING_GUARD_VPK:
            raise RuntimeError('Feedback was not enabled: fresh IN1 peak %.4f V reached the %.4f V operating guard.' % (input_peak_v,ADC_OPERATING_GUARD_VPK))
        if abs(input_dc_v) > IN1_DC_OFFSET_MAX_V:
            raise RuntimeError('Feedback was not enabled: fresh IN1 mean %+.4f V exceeded the +/-%.4f V DC guard.' % (input_dc_v,IN1_DC_OFFSET_MAX_V))
        if output_bound_vpk >= OUT1_OPERATING_GUARD_VPK:
            raise RuntimeError('Feedback was not enabled: conservative OUT1 bound %.4f Vpk reached the %.4f Vpk operating guard.' % (output_bound_vpk,OUT1_OPERATING_GUARD_VPK))

        gen.assert_b_zero()
        ps.config_delay(reference)
        ps.set(count/16384.0,phase)
        readback=ps.readback()
        if (readback['ci'] != command['ci_word'] or readback['cq'] != command['cq_word'] or
                readback['bypass'] != 0 or readback['monitor'] != 0):
            raise RuntimeError('Feedback coefficient verification failed: expected CI/CQ 0x%04X/0x%04X with bypass and OUT2 off, read %r.' %
                               (command['ci_word'],command['cq_word'],readback))
        gen.assert_b_zero()
        HELD_FEEDBACK_STATE={'enabled':True,'coefficient_counts':count,
            'maximum_coefficient_counts':ceiling,'requested_phase_deg':phase%360.,
            'actual_phase_deg':command['actual_phase_deg'],'reference_frequency_hz':reference,
            'ci_signed':command['ci_signed'],'cq_signed':command['cq_signed'],
            'actual_gain':command['actual_gain'],'input_peak_v':input_peak_v,
            'input_dc_v':input_dc_v,'conservative_out1_bound_vpk':output_bound_vpk,
            'enabled_epoch_s':time.time(),'out2_enabled':False,
            'automatic_timeout':False}
        print('HELD FEEDBACK ENABLED: count %d/%d, requested phase %.2f deg, actual phase %.2f deg at %.6f MHz.' %
              (count,ceiling,phase%360.,command['actual_phase_deg'],reference/1e6))
        print('CI=%d, CQ=%d; IN1 peak %.4f V, mean %+.4f V; conservative OUT1 bound %.4f Vpk.' %
              (command['ci_signed'],command['cq_signed'],input_peak_v,input_dc_v,output_bound_vpk))
        print('The coefficients remain active after this cell returns. Run disable_held_feedback() immediately after the external measurement.')
        return dict(HELD_FEEDBACK_STATE)
    except BaseException as exc:
        HELD_FEEDBACK_STATE={'enabled':False,'last_error':'%s: %s' % (type(exc).__name__,str(exc))}
        try: full_shutdown()
        except BaseException as shutdown_error:
            raise RuntimeError('Feedback setup failed and shutdown could not be verified. Disconnect OUT1 physically. Setup error: %s; shutdown error: %s' % (exc,shutdown_error))
        raise


def disable_held_feedback():
    """Clear held feedback, both generators, OUT2 and acquisition, then verify."""
    import time
    global HELD_FEEDBACK_STATE
    full_shutdown()
    readback=ps.readback()
    if readback['ci'] or readback['cq'] or readback['monitor']:
        raise RuntimeError('DISABLE FAILED: feedback/OUT2 readback remains nonzero after full shutdown. Disconnect OUT1 physically: %r' % readback)
    HELD_FEEDBACK_STATE={'enabled':False,'disabled_epoch_s':time.time(),
                         'last_action':'full shutdown verified'}
    print('HELD FEEDBACK DISABLED: CI=CQ=0; generators, OUT2 and acquisition are off.')
    return dict(HELD_FEEDBACK_STATE)


## 28 — Enable for the external measurement, then disable

Edit the count, phase, reference frequency and local ceiling below. Run the enable call once, make the external measurement promptly, and then run the disable call. The enable call starts from full shutdown and prints the exact quantized phase and coefficients that are held.

The local maximum may be reduced but cannot exceed 1024. To change the setting, disable first and then call enable again. If the notebook or connection becomes unresponsive, use the physical OUT1 disconnect; do not rely on Python cleanup.


In [ ]:
# LIVE HARDWARE - uncomment only the enable call you intend to use.
# HELD_SETTING = enable_held_feedback(
#     coefficient_counts=128,
#     phase_deg=225.0,
#     reference_frequency_hz=40.9e6,
#     maximum_coefficient_counts=512)

# Run immediately after the external measurement:
# disable_held_feedback()

# Emergency alternatives already defined earlier:
# full_shutdown()
# disable_everything()
